In [ ]:
# Block 1: Notebook Description
#Notebook description

#This notebook is being used to evaluate the techinical market conditions of a single asset and assess
#the appropriate strategy to take in order to maximize returns.

In [ ]:
# Block 2: Imports and Project Bootstrap
import logging
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()
SINGLE_ASSET_DIRECTORY = PROJECT_ROOT / "Research" / "Single Asset"
if str(SINGLE_ASSET_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SINGLE_ASSET_DIRECTORY))
from _params import get_single_asset_params


from Quantapp.data import yf as qa_yf
from Quantapp.visualization import (
    BarChartPlotter,
    Plotter,
    build_time_range_buttons,
)
from Quantapp.visualization.views.single_asset_profile.pricing.options_pricing import (
    build_current_atm_iv_by_expiration,
    build_historical_atm_iv_history,
    build_historical_iv_premium_history,
    build_atm_implied_move_term_structure,
    plot_atm_iv_realized_view,
    plot_atm_implied_move_term_structure_view,
    plot_gbm_paths_view,
    plot_current_atm_iv_spread_view,
    plot_historical_atm_iv_summary_view,
    plot_historical_implied_move_candlestick_view,
    plot_historical_iv_premium_view,
    plot_historical_iv_rank_percentile_view,
    plot_implied_volatility_by_strike_view,
    plot_iv_minus_realized_by_strike_view,
    plot_median_iv_minus_realized_view,
    plot_open_interest_overview_view,
    plot_open_interest_implied_move_ranges_view,
    plot_option_chain_table_view,
    plot_svi_surface_view,
)
from Quantapp.visualization.views.single_asset_profile.pricing.distribution import (
    plot_distribution_shape_zscores_view,
    plot_fixed_payout_strategy_backtest_view,
    plot_trade_range_history_profile,
    plot_trade_range_probability_cone,
    plot_trade_range_breach_average_view,
    plot_trade_range_breach_excess_view,
    plot_trade_range_stack_view,
    plot_volatility_model_comparison_view,
)
from scipy.stats import kurtosis, skew

from Quantapp.analytics import Helper, SeriesTransforms
from Quantapp.analytics import compute
from Quantapp.analytics.series_utils import (
    calculate_historical_var_metrics,
    calculate_textbook_rolling_max_drawdown,
    calculate_zscore,
    coerce_close_series,
    gini_coefficient,
)
from Quantapp.data import (
    MacroDataClient,
    align_series_to_common_index,
    get_current_options_chain,
    get_historical_options_eod_panel,
    get_market_history,
    load_benchmark_data,
    normalize_benchmark_tickers,
)
from Quantapp.secrets import load_project_env, require_secret

load_project_env()

warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")

# Use one dark visual system for every Plotly figure in this notebook.
NOTEBOOK_PLOT_TEMPLATE = 'plotly_dark'
NOTEBOOK_PLOT_BACKGROUND = '#111827'
NOTEBOOK_PLOT_GRID = '#374151'
pio.templates.default = NOTEBOOK_PLOT_TEMPLATE

def apply_notebook_plot_theme(fig):
    fig.update_layout(
        template=NOTEBOOK_PLOT_TEMPLATE,
        paper_bgcolor=NOTEBOOK_PLOT_BACKGROUND,
        plot_bgcolor=NOTEBOOK_PLOT_BACKGROUND,
        font=dict(color='#E5E7EB'),
        legend=dict(bgcolor='rgba(17, 24, 39, 0.75)'),
    )
    fig.update_xaxes(gridcolor=NOTEBOOK_PLOT_GRID, zerolinecolor='#6B7280')
    fig.update_yaxes(gridcolor=NOTEBOOK_PLOT_GRID, zerolinecolor='#6B7280')

    # Plotly table cells use explicit fills, so the base template cannot recolor them.
    table_traces = [trace for trace in fig.data if trace.type == 'table']
    table_header_colors = ('#0F766E', '#9F1239')
    for table_number, trace in enumerate(table_traces):
        dark_cell_colors = [
            [
                'rgba(34, 197, 94, 0.45)' if '144, 238, 144' in str(color) else '#111827'
                for color in column_colors
            ]
            for column_colors in trace.cells.fill.color
        ]
        trace.update(
            header=dict(
                fill_color=table_header_colors[table_number % len(table_header_colors)],
                font=dict(color='#F9FAFB', size=11),
            ),
            cells=dict(
                fill_color=dark_cell_colors,
                font=dict(color='#E5E7EB'),
            ),
        )

    return fig

INSANE_SPIKE_SMOOTHING_CONFIG = {
    'price_columns': ('Open', 'High', 'Low', 'Close', 'Adj Close'),
    'rolling_window': 63,
    'absolute_log_return_threshold': np.log1p(0.35),
    'endpoint_absolute_log_return_threshold': np.log1p(0.75),
    'robust_z_threshold': 12.0,
    'endpoint_robust_z_threshold': 20.0,
    'interpolation_limit': 3,
}

def _robust_centered_mad_zscore(series, window):
    values = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan)
    min_periods = max(10, int(window) // 4)
    rolling_median = values.rolling(window, center=True, min_periods=min_periods).median()
    rolling_mad = values.sub(rolling_median).abs().rolling(
        window,
        center=True,
        min_periods=min_periods,
    ).median()
    scale = (1.4826 * rolling_mad).replace(0, np.nan)
    return values.sub(rolling_median).div(scale).replace([np.inf, -np.inf], np.nan)

def _isolated_price_spike_mask(
    series,
    *,
    rolling_window=None,
    absolute_log_return_threshold=None,
    endpoint_absolute_log_return_threshold=None,
    robust_z_threshold=None,
    endpoint_robust_z_threshold=None,
):
    config = INSANE_SPIKE_SMOOTHING_CONFIG
    rolling_window = int(rolling_window or config['rolling_window'])
    absolute_log_return_threshold = float(
        absolute_log_return_threshold or config['absolute_log_return_threshold']
    )
    endpoint_absolute_log_return_threshold = float(
        endpoint_absolute_log_return_threshold
        or config['endpoint_absolute_log_return_threshold']
    )
    robust_z_threshold = float(robust_z_threshold or config['robust_z_threshold'])
    endpoint_robust_z_threshold = float(
        endpoint_robust_z_threshold or config['endpoint_robust_z_threshold']
    )

    clean = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan)
    log_values = np.log(clean.where(clean > 0))
    inbound_return = log_values.diff()
    outbound_return = log_values.shift(-1).sub(log_values)
    inbound_z = _robust_centered_mad_zscore(inbound_return, rolling_window)
    outbound_z = _robust_centered_mad_zscore(outbound_return, rolling_window)

    inbound_extreme = (
        inbound_return.abs().gt(absolute_log_return_threshold)
        | inbound_z.abs().gt(robust_z_threshold)
    )
    outbound_extreme = (
        outbound_return.abs().gt(absolute_log_return_threshold)
        | outbound_z.abs().gt(robust_z_threshold)
    )
    isolated_reversal = (
        inbound_extreme
        & outbound_extreme
        & inbound_return.notna()
        & outbound_return.notna()
        & np.sign(inbound_return).ne(np.sign(outbound_return))
    )

    endpoint_spike = (
        outbound_return.isna()
        & inbound_return.notna()
        & (
            inbound_return.abs().gt(endpoint_absolute_log_return_threshold)
            | inbound_z.abs().gt(endpoint_robust_z_threshold)
        )
    )
    return (isolated_reversal | endpoint_spike).fillna(False)

def smooth_insane_price_spikes(
    price_frame,
    *,
    label=None,
    price_columns=None,
    rolling_window=None,
    absolute_log_return_threshold=None,
    endpoint_absolute_log_return_threshold=None,
    robust_z_threshold=None,
    endpoint_robust_z_threshold=None,
    interpolation_limit=None,
):
    if price_frame is None or price_frame.empty:
        return price_frame, pd.DataFrame(columns=['column', 'spike_count'])

    config = INSANE_SPIKE_SMOOTHING_CONFIG
    price_columns = tuple(price_columns or config['price_columns'])
    interpolation_limit = int(interpolation_limit or config['interpolation_limit'])
    smoothed = price_frame.copy()
    report_rows = []
    interpolation_method = 'time' if isinstance(smoothed.index, pd.DatetimeIndex) else 'linear'

    for column in price_columns:
        if column not in smoothed.columns:
            continue
        original = pd.to_numeric(smoothed[column], errors='coerce')
        spike_mask = _isolated_price_spike_mask(
            original,
            rolling_window=rolling_window,
            absolute_log_return_threshold=absolute_log_return_threshold,
            endpoint_absolute_log_return_threshold=endpoint_absolute_log_return_threshold,
            robust_z_threshold=robust_z_threshold,
            endpoint_robust_z_threshold=endpoint_robust_z_threshold,
        )
        if not bool(spike_mask.any()):
            continue

        cleaned = original.mask(spike_mask)
        cleaned = cleaned.interpolate(
            method=interpolation_method,
            limit=interpolation_limit,
            limit_direction='both',
        ).ffill().bfill()
        smoothed[column] = cleaned.where(original.notna(), original)
        report_rows.append({'column': column, 'spike_count': int(spike_mask.sum())})

    if {'Open', 'High', 'Low', 'Close'}.issubset(smoothed.columns):
        smoothed['High'] = pd.concat(
            [smoothed['High'], smoothed['Open'], smoothed['Close']],
            axis=1,
        ).max(axis=1)
        smoothed['Low'] = pd.concat(
            [smoothed['Low'], smoothed['Open'], smoothed['Close']],
            axis=1,
        ).min(axis=1)

    spike_report = pd.DataFrame(report_rows)
    if label and not spike_report.empty:
        total_spikes = int(spike_report['spike_count'].sum())
        adjusted_columns = ', '.join(spike_report['column'].astype(str))
        print(
            f'Smoothed {total_spikes} isolated price spike(s) in {label} '
            f'across: {adjusted_columns}'
        )
    return smoothed, spike_report

def smooth_insane_series_spikes(
    series,
    *,
    label=None,
    rolling_window=63,
    robust_z_threshold=8.0,
    interpolation_limit=5,
    floor=None,
    ceiling=None,
):
    values = pd.to_numeric(pd.Series(series).copy(), errors='coerce').replace([np.inf, -np.inf], np.nan)
    values = values.mask(values.lt(floor)) if floor is not None else values
    values = values.mask(values.gt(ceiling)) if ceiling is not None else values
    robust_z = _robust_centered_mad_zscore(values, int(rolling_window))
    spike_mask = robust_z.abs().gt(float(robust_z_threshold)).fillna(False)
    if ceiling is not None:
        spike_mask = spike_mask | pd.to_numeric(pd.Series(series), errors='coerce').gt(float(ceiling)).fillna(False)

    if not bool(spike_mask.any()):
        report = pd.DataFrame(columns=['series', 'spike_count'])
        values.name = getattr(series, 'name', None)
        return values, report

    interpolation_method = 'time' if isinstance(values.index, pd.DatetimeIndex) else 'linear'
    smoothed = values.mask(spike_mask).interpolate(
        method=interpolation_method,
        limit=int(interpolation_limit),
        limit_direction='both',
    ).ffill().bfill()
    if floor is not None:
        smoothed = smoothed.clip(lower=float(floor))
    if ceiling is not None:
        smoothed = smoothed.clip(upper=float(ceiling))
    smoothed.name = getattr(series, 'name', None)
    report = pd.DataFrame([
        {'series': label or getattr(series, 'name', 'series'), 'spike_count': int(spike_mask.sum())}
    ])
    if label:
        print(f'Smoothed {int(spike_mask.sum())} insane spike(s) in {label}.')
    return smoothed, report

In [ ]:
# Block 3: Shared and Options-Specific Parameters

pricing_params = get_single_asset_params()
TIMEFRAME_PROFILES = {
    "swing": {"short": 3, "mid": 9, "long": 21},
    "position": {"short": 21, "mid": 50, "long": 200},
    "structural": {"short": 200, "mid": 500, "long": 1000},
}

def resolve_time_frame_map(strategy):
    normalized_strategy = str(strategy).strip().lower()
    if normalized_strategy not in TIMEFRAME_PROFILES:
        raise ValueError(f"Invalid trading_strategy: {strategy}")
    return dict(TIMEFRAME_PROFILES[normalized_strategy])

trading_strategy = "position"
time_frame_map = resolve_time_frame_map(trading_strategy)
options_params = {
    **pricing_params,
    "risk_free_ticker": "^IRX",
    "benchmark_tickers": ["SPY"],
    "trading_strategy": trading_strategy,
    "length_of_plots": 20,
    "var_position_value": None,
    "risk_free_rate": 0.02 / 252,
    "time_frame_week": 7,
    "time_frame_short": time_frame_map["short"],
    "time_frame_mid": time_frame_map["mid"],
    "time_frame_long": time_frame_map["long"],
}

ticker_str = options_params["ticker_str"]
interval = options_params["interval"]
period = options_params["period"]
risk_free_ticker = options_params["risk_free_ticker"]
risk_free_rate = options_params["risk_free_rate"]
time_frame_week = options_params["time_frame_week"]
time_frame_short = options_params["time_frame_short"]
time_frame_mid = options_params["time_frame_mid"]
time_frame_long = options_params["time_frame_long"]
benchmark_tickers = list(options_params["benchmark_tickers"])
trading_strategy = options_params["trading_strategy"]
length_of_plots = options_params["length_of_plots"]
var_position_value = options_params["var_position_value"]

options_params


In [ ]:
# Block 4: TODO Notes
#take all compuation functions and put them in a separate file

#simplify the date x axis on the percent drawdown chart

#default the zoom range to a comfortable range, and create a dropdown to select the time range for Volatility section

#properly label and annoate the garch models

#Remove the VIX charting, its redudnant now that we have the volatility models

In [ ]:
# Block 5: Shared Clients

qp = Plotter()
qe = MacroDataClient()
helper = Helper()
barChartPlotter = BarChartPlotter()
series_transforms = SeriesTransforms()


In [ ]:
# Block 6: Underlying Data Load
#Load data: underlying data
print(f"Loading data for {ticker_str} with period {period} and interval {interval}")
ticker_handle = qa_yf.Ticker(ticker_str)  # Used for options metadata/fallbacks through Quantapp.data.

market_history = get_market_history(
    symbols=[ticker_str],
    period=period,
    interval=interval,
    provider="yfinance",
    align=False,
)
ticker = market_history.get(str(ticker_str).strip().upper(), pd.DataFrame())

if ticker.empty:
    raise ValueError(f"No underlying price history returned for {ticker_str}.")

ticker_raw = ticker.copy()
ticker, ticker_spike_smoothing_report = smooth_insane_price_spikes(
    ticker_raw,
    label=ticker_str,
)
market_history[str(ticker_str).strip().upper()] = ticker

price_series = ticker['Close'].dropna()
log_returns = np.log(price_series / price_series.shift(1)).dropna()
spot_price = float(price_series.iloc[-1])

rolling_vol_window = min(len(log_returns), 252)
if rolling_vol_window < 2:
    raise ValueError(f"Not enough history to compute volatility for {ticker_str}.")

annualized_vol = log_returns.iloc[-rolling_vol_window:].std() * np.sqrt(252)

In [ ]:
# Block 7: Expiration Dates
#Load data: Expiration Dates
print(f"Loading options expiration dates for {ticker_str}")
options_expiration_dates = pd.DataFrame(ticker_handle.options, columns=['Expiration Date'])

if options_expiration_dates.empty:
    raise ValueError(f"No listed option expirations returned for {ticker_str}.")

options_expiration_dates = options_expiration_dates.sort_values('Expiration Date').reset_index(drop=True)
options_expiration_dates['Date Till Expiration'] = (
    pd.to_datetime(options_expiration_dates['Expiration Date']) - pd.Timestamp.today().normalize()
).dt.days

In [ ]:
# Block 8: Current Options Chain Snapshot
# Retrieve the current options chain snapshot.

print(f"Loading options chain for {ticker_str}")

call_contract_chain = {}
put_contract_chain = {}
drop_columns = ['lastTradeDate', 'contractSize', 'currency', 'percentChange', 'change']
today = pd.Timestamp.today().normalize()


try:
    massive_chain = get_current_options_chain(ticker_str, fallback_underlying_price=spot_price)
    massive_chain_df = massive_chain.chain
    underlying_price = massive_chain.underlying_price
    call_contract_chain = massive_chain.calls_by_expiration
    put_contract_chain = massive_chain.puts_by_expiration
    expirations = massive_chain.expirations
    options_expiration_dates = pd.DataFrame({'Expiration Date': expirations})
    options_expiration_dates['Date Till Expiration'] = (
        pd.to_datetime(options_expiration_dates['Expiration Date']) - today
    ).dt.days

    first_expiration_date = expirations[0]
    call_contracts = call_contract_chain[first_expiration_date].copy()
    put_contracts = put_contract_chain[first_expiration_date].copy()
    underlying_data = {'regularMarketPrice': underlying_price}
    first_expiration_chain = {'underlying': underlying_data, 'calls': call_contracts, 'puts': put_contracts}

    call_contract_chain_concat = massive_chain.calls
    put_contract_chain_concat = massive_chain.puts
    all_contracts_concat = pd.concat([call_contract_chain_concat, put_contract_chain_concat], ignore_index=True)

    print(f"Loaded options chain from Massive for {ticker_str}: {len(all_contracts_concat)} contracts")

except Exception as massive_error:
    print(f"Massive load failed ({massive_error}); falling back to Quantapp.data yfinance compatibility")

    first_expiration_date = options_expiration_dates['Expiration Date'].iloc[0]
    first_expiration_chain = ticker_handle.option_chain(first_expiration_date)
    underlying_data = first_expiration_chain.underlying
    call_contracts = first_expiration_chain.calls.copy()
    put_contracts = first_expiration_chain.puts.copy()

    for expiration_date in options_expiration_dates['Expiration Date']:
        option_chain = ticker_handle.option_chain(expiration_date)
        days_till_expiration = (pd.to_datetime(expiration_date) - today).days

        call_df = option_chain.calls.drop(columns=drop_columns, errors='ignore').copy()
        put_df = option_chain.puts.drop(columns=drop_columns, errors='ignore').copy()

        for df, option_type in ((call_df, 'Call'), (put_df, 'Put')):
            df['Days Till Expiration'] = days_till_expiration
            df['Expiration Date'] = expiration_date
            df['bid-ask spread'] = df['ask'] - df['bid']
            df['Expiration day'] = pd.to_datetime(expiration_date).day
            df['Expiration day name'] = pd.to_datetime(expiration_date).strftime('%A')
            df['Type'] = option_type
            df['mid'] = (df['bid'] + df['ask']) / 2

        call_contract_chain[expiration_date] = call_df
        put_contract_chain[expiration_date] = put_df

    expirations = sorted(call_contract_chain.keys())
    call_contract_chain_concat = pd.concat(call_contract_chain.values(), ignore_index=True)
    put_contract_chain_concat = pd.concat(put_contract_chain.values(), ignore_index=True)
    all_contracts_concat = pd.concat([call_contract_chain_concat, put_contract_chain_concat], ignore_index=True)

In [ ]:
# Block 9: Open Interest Overview

# Associate each upcoming earnings event with the listed expiration closest to it.
# The matched DTE is highlighted consistently on expiration-based plots.
def _upcoming_earnings_dates(ticker):
    today_normalized = pd.Timestamp.today().normalize()
    dates = pd.DatetimeIndex([])
    try:
        earnings = ticker.get_earnings_dates(limit=12)
        if earnings is not None and not earnings.empty:
            dates = pd.DatetimeIndex(pd.to_datetime(earnings.index, errors='coerce', utc=True)).tz_convert(None)
    except Exception:
        pass

    dates = pd.DatetimeIndex(dates.dropna()).normalize().unique().sort_values()
    future_dates = dates[dates >= today_normalized]
    if len(future_dates) == 0:
        try:
            calendar = ticker.get_calendar() or {}
            calendar_dates = calendar.get('Earnings Date', [])
            if not isinstance(calendar_dates, (list, tuple, pd.Series, pd.Index, np.ndarray)):
                calendar_dates = [calendar_dates]
            calendar_dates = pd.DatetimeIndex(
                pd.to_datetime(calendar_dates, errors='coerce', utc=True)
            ).tz_convert(None)
            calendar_dates = calendar_dates.dropna().normalize().unique().sort_values()
            future_dates = calendar_dates[calendar_dates >= today_normalized]
        except Exception as earnings_error:
            print(f'Upcoming earnings dates unavailable for {ticker_str}: {earnings_error}')
    return future_dates

upcoming_earnings_dates = _upcoming_earnings_dates(ticker_handle)
listed_expiration_dates = pd.DatetimeIndex(pd.to_datetime(expirations, errors='coerce')).dropna().normalize().sort_values()
earnings_expiration_map = {}
earnings_expiration_details = []
for earnings_date in upcoming_earnings_dates:
    if len(listed_expiration_dates):
        distance_days = np.abs((listed_expiration_dates - earnings_date).days)
        closest_position = int(np.argmin(distance_days))
        associated_expiration = listed_expiration_dates[closest_position]
        matched_dte = int((associated_expiration - pd.Timestamp.today().normalize()).days)
        match_distance = int(distance_days[closest_position])
        earnings_expiration_map.setdefault(associated_expiration, []).append(earnings_date)
        earnings_expiration_details.append({
            'Earnings Date': earnings_date,
            'Nearest Expiration': associated_expiration,
            'Matched DTE': matched_dte,
            'Calendar-Day Distance': match_distance,
        })
earnings_expiration_details = pd.DataFrame(earnings_expiration_details)

def _earnings_expiration_label(expiration):
    expiration_date = pd.Timestamp(expiration).normalize()
    return '<br><span style="color:#FDA4AF">◆ Earnings-nearest</span>' if expiration_date in earnings_expiration_map else ''

def mark_upcoming_earnings_contracts(figure, expiration_to_x=None):
    """Add an earnings summary and optional matched-DTE guide lines."""
    if not earnings_expiration_map:
        figure.add_annotation(
            x=1, y=1.04, xref='paper', yref='paper',
            xanchor='right', yanchor='bottom', showarrow=False, align='left',
            text=(
                f'<b>◇ Earnings schedule unavailable</b><br>'
                f'yfinance returned no future corporate earnings date for {ticker_str}. '
                'ETFs such as XLC do not have their own earnings announcement.'
            ),
            bgcolor='rgba(18, 20, 30, 0.96)', bordercolor='#64748B',
            borderwidth=1, borderpad=7, font=dict(color='#CBD5E1', size=10),
        )
        current_top_margin = figure.layout.margin.t or 80
        figure.update_layout(margin=dict(t=max(int(current_top_margin), 150)))
        return figure
    event_lines = []
    for expiration_date, earnings_dates in earnings_expiration_map.items():
        event_text = ', '.join(date.strftime('%b %d, %Y') for date in earnings_dates)
        matched_dte = int((expiration_date - pd.Timestamp.today().normalize()).days)
        distances = [abs(int((expiration_date - date).days)) for date in earnings_dates]
        distance_text = ', '.join(f'{distance}d away' for distance in distances)
        event_lines.append(
            f'<b>◆ Earnings {event_text}</b><br>'
            f'Nearest expiry {expiration_date:%b %d} · {matched_dte} DTE · {distance_text}'
        )
        if expiration_to_x is not None and expiration_date in expiration_to_x:
            marker_location = expiration_to_x[expiration_date]
            if isinstance(marker_location, dict):
                x0 = marker_location['band_start']
                x1 = marker_location['band_end']
                marker_x = marker_location['marker']
                category_shift = marker_location.get('category_shift', {})
            elif isinstance(marker_location, tuple):
                x0, x1 = marker_location
                marker_x = (x0 + x1) / 2
                category_shift = {}
            else:
                x0 = x1 = marker_x = marker_location
                category_shift = {'x0shift': -0.48, 'x1shift': 0.48}
            figure.add_shape(
                type='rect', x0=x0, x1=x1, y0=0, y1=1,
                xref='x', yref='paper', layer='below',
                fillcolor='rgba(244, 63, 94, 0.08)',
                line=dict(color='rgba(251, 113, 133, 0.38)', width=1, dash='dot'),
                **category_shift,
            )
            figure.add_shape(
                type='line', x0=marker_x, x1=marker_x,
                y0=0, y1=1, xref='x', yref='paper', layer='above',
                line=dict(color='rgba(251, 113, 133, 0.68)', width=2, dash='solid'),
            )
            figure.add_annotation(
                x=marker_x, y=0.98, xref='x', yref='paper',
                text=f'<b>Earnings warning</b><br>Pre-earnings window ≤ {matched_dte} DTE',
                showarrow=False, xanchor='right', yanchor='top', align='right',
                bgcolor='rgba(18, 20, 30, 0.86)',
                bordercolor='rgba(251, 113, 133, 0.50)', borderwidth=1, borderpad=4,
                font=dict(color='#FDA4AF', size=10),
            )
    figure.add_annotation(
        x=1, y=1.04, xref='paper', yref='paper', xanchor='right', yanchor='bottom',
        text='<br><br>'.join(event_lines), showarrow=False, align='left',
        bgcolor='rgba(18, 20, 30, 0.94)', bordercolor='rgba(251, 113, 133, 0.62)', borderwidth=1, borderpad=7,
        font=dict(color='#FFE4E6', size=10),
    )
    current_top_margin = figure.layout.margin.t or 80
    figure.update_layout(margin=dict(t=max(int(current_top_margin), 170)))
    return figure

# Compute totals
total_oi_calls = [call_contract_chain[exp]['openInterest'].sum() for exp in expirations]
total_oi_puts = [put_contract_chain[exp]['openInterest'].sum() for exp in expirations]
dte_list = [call_contract_chain[exp]['Days Till Expiration'].iloc[0] for exp in expirations]
x_tick_labels = [f'{exp}<br>{dte} DTE{_earnings_expiration_label(exp)}' for exp, dte in zip(expirations, dte_list)]
expiration_dates = pd.DatetimeIndex(pd.to_datetime(expirations)).normalize()
expiration_years = expiration_dates.year.tolist()

# Aggregate total OI (calls + puts)
total_oi_all = [c + p for c, p in zip(total_oi_calls, total_oi_puts)]
put_call_ratio = [
    (put_oi / call_oi) if call_oi else np.nan
    for call_oi, put_oi in zip(total_oi_calls, total_oi_puts)
]

fig = plot_open_interest_overview_view(
    expirations=expirations,
    total_oi_calls=total_oi_calls,
    total_oi_puts=total_oi_puts,
    total_oi_all=total_oi_all,
    dte_list=dte_list,
    put_call_ratio=put_call_ratio,
)
fig.update_xaxes(
    tickmode='array',
    tickvals=[str(exp) for exp in expirations],
    ticktext=x_tick_labels,
)
fig.update_xaxes(title_text='Option Expiration (DTE)', row=3, col=1)

# Shade each expiration year and label it relative to the current year.
current_year = pd.Timestamp.today().year
year_band_colors = [
    ('rgba(59, 130, 246, 0.14)', '#60A5FA'),
    ('rgba(52, 211, 153, 0.14)', '#34D399'),
    ('rgba(251, 191, 36, 0.14)', '#FBBF24'),
    ('rgba(192, 132, 252, 0.14)', '#C084FC'),
]
expiration_count = len(expirations)
open_interest_panel_refs = [
    ('x domain', 'y domain'),
    ('x2 domain', 'y2 domain'),
    ('x3 domain', 'y3 domain'),
]

for band_number, year in enumerate(dict.fromkeys(expiration_years)):
    year_positions = [i for i, expiration_year in enumerate(expiration_years) if expiration_year == year]
    first_position, last_position = min(year_positions), max(year_positions)
    fill_color, accent_color = year_band_colors[band_number % len(year_band_colors)]
    band_start = first_position / expiration_count
    band_end = (last_position + 1) / expiration_count

    for xref, yref in open_interest_panel_refs:
        fig.add_shape(
            type='rect',
            xref=xref,
            yref=yref,
            x0=band_start,
            x1=band_end,
            y0=0,
            y1=1,
            fillcolor=fill_color,
            line_width=0,
            layer='below',
        )

    if first_position > 0:
        boundary_x = first_position / expiration_count
        for xref, yref in open_interest_panel_refs:
            fig.add_shape(
                type='line',
                xref=xref,
                yref=yref,
                x0=boundary_x,
                x1=boundary_x,
                y0=0,
                y1=1,
                line=dict(color=accent_color, width=2, dash='dot'),
            )

    if year == current_year:
        year_context = 'Current year'
    elif year == current_year + 1:
        year_context = 'Next year'
    else:
        year_context = f'{year - current_year:+d} years'

    fig.add_annotation(
        x=(band_start + band_end) / 2,
        y=1,
        xref='x3 domain',
        yref='y3 domain',
        yshift=-6,
        text=f'<b>{year}</b><br><span style="font-size:10px">{year_context}</span>',
        showarrow=False,
        bordercolor=accent_color,
        borderwidth=1,
        bgcolor='rgba(17, 24, 39, 0.90)',
        font=dict(color=accent_color),
    )

# Mark standard quarterly expirations (third Friday of Mar/Jun/Sep/Dec).
expiration_positions = {date: position for position, date in enumerate(expiration_dates)}
quarterly_expirations = []
quarterly_year_months = sorted(
    {(date.year, date.month) for date in expiration_dates if date.month in (3, 6, 9, 12)}
)

for year, month in quarterly_year_months:
    month_start = pd.Timestamp(year=year, month=month, day=1)
    first_friday = month_start + pd.Timedelta(days=(4 - month_start.weekday()) % 7)
    third_friday = first_friday + pd.Timedelta(weeks=2)
    quarterly_date = third_friday

    # If Friday is a market holiday, use the preceding Thursday when available.
    if quarterly_date not in expiration_positions:
        quarterly_date = third_friday - pd.Timedelta(days=1)

    if quarterly_date in expiration_positions:
        quarterly_expirations.append((expiration_positions[quarterly_date], quarterly_date))

for position, quarterly_date in quarterly_expirations:
    quarterly_x = (position + 0.5) / expiration_count
    for xref, yref in open_interest_panel_refs:
        fig.add_shape(
            type='line',
            xref=xref,
            yref=yref,
            x0=quarterly_x,
            x1=quarterly_x,
            y0=0,
            y1=1,
            line=dict(color='#F87171', width=3, dash='dash'),
        )
    fig.add_annotation(
        x=quarterly_x,
        y=1,
        xref='x domain',
        yref='y domain',
        xshift=7,
        yshift=-5,
        text=f'<b>Q{quarterly_date.quarter} {quarterly_date.year}</b>',
        textangle=-90,
        showarrow=False,
        xanchor='left',
        yanchor='top',
        font=dict(color='#FCA5A5', size=10),
    )
open_interest_expiration_to_x = {
    expiration: {
        'band_start': str(expirations[0]),
        'band_end': str(expirations[position]),
        'marker': str(expirations[position]),
        'category_shift': {},
    }
    for position, expiration in enumerate(expiration_dates)
}
mark_upcoming_earnings_contracts(fig, open_interest_expiration_to_x)
apply_notebook_plot_theme(fig)
fig.show()

In [ ]:
# Block 10: Open Interest and Put-Call Skew by Individual DTE

if not expirations:
    raise ValueError('No option expirations are available to plot.')

implied_move_spot = pd.to_numeric(underlying_price, errors='coerce')
if not np.isfinite(implied_move_spot) or implied_move_spot <= 0:
    implied_move_spot = float(spot_price)
else:
    implied_move_spot = float(implied_move_spot)

# The DTE selector shows one expiration at a time. Calls and puts share the
# upper panel; put IV minus call IV is plotted directly below.
open_interest_implied_move_fig = plot_open_interest_implied_move_ranges_view(
    call_contract_chain=call_contract_chain,
    put_contract_chain=put_contract_chain,
    expirations=expirations,
    spot_price=implied_move_spot,
)
mark_upcoming_earnings_contracts(open_interest_implied_move_fig)
apply_notebook_plot_theme(open_interest_implied_move_fig)
open_interest_implied_move_fig.show()

# Separate term structure: one ATM implied-move observation per expiration.
implied_move_by_expiration = build_atm_implied_move_term_structure(
    call_contract_chain,
    put_contract_chain,
    expirations,
    spot_price=implied_move_spot,
)
implied_move_term_structure_fig = plot_atm_implied_move_term_structure_view(
    implied_move_by_expiration,
    spot_price=implied_move_spot,
    ticker_label=ticker_str,
)
implied_move_expiration_to_x = {
    pd.Timestamp(expiration).normalize(): {
        'band_start': 0,
        'band_end': position,
        'marker': position,
    }
    for position, expiration in enumerate(
        pd.to_datetime(implied_move_by_expiration['Expiration Date'])
    )
}
mark_upcoming_earnings_contracts(
    implied_move_term_structure_fig,
    implied_move_expiration_to_x,
)
apply_notebook_plot_theme(implied_move_term_structure_fig)
implied_move_term_structure_fig.show()

In [ ]:
# Block 11: ATM IV and Realized Volatility
#plot the ATM IV and Realized Volatility

# Define ATM IV fetcher
def get_atm_iv_for_expiration(expiration_date, contract_chain):
    contracts = contract_chain.get(expiration_date)
    if contracts is None or contracts.empty:
        return np.nan

    required_columns = {'strike', 'impliedVolatility'}
    if not required_columns.issubset(contracts.columns):
        return np.nan

    valid_contracts = contracts[['strike', 'impliedVolatility']].copy()
    valid_contracts['strike'] = pd.to_numeric(valid_contracts['strike'], errors='coerce')
    valid_contracts['impliedVolatility'] = pd.to_numeric(valid_contracts['impliedVolatility'], errors='coerce')
    valid_contracts = valid_contracts.dropna(subset=['strike', 'impliedVolatility'])
    if valid_contracts.empty:
        return np.nan

    idx = (valid_contracts['strike'] - spot_price).abs().idxmin()
    return float(valid_contracts.loc[idx, 'impliedVolatility'])

# Calculate realized vol for each expiration
def get_realized_vol_for_expiration(expiration_date):
    days = (pd.to_datetime(expiration_date) - pd.Timestamp.today().normalize()).days
    if 1 < days < len(log_returns):
        window_returns = log_returns.iloc[-days:]
        realized_vol = window_returns.std() * np.sqrt(252)
        return realized_vol
    return np.nan

atm_df = pd.DataFrame({'Expiration Date': expirations})
atm_df['ATM IV Call'] = atm_df['Expiration Date'].apply(
    lambda exp: get_atm_iv_for_expiration(exp, call_contract_chain)
 )
atm_df['ATM IV Put'] = atm_df['Expiration Date'].apply(
    lambda exp: get_atm_iv_for_expiration(exp, put_contract_chain)
 )
atm_df['Days Till Expiration'] = atm_df['Expiration Date'].apply(
    lambda d: (pd.to_datetime(d) - pd.Timestamp.today().normalize()).days
 )
atm_df['Realized Vol'] = atm_df['Expiration Date'].apply(get_realized_vol_for_expiration)
atm_df = atm_df.sort_values('Days Till Expiration').reset_index(drop=True)

# Compute spreads
atm_df['IV-RV Call'] = atm_df['ATM IV Call'] - atm_df['Realized Vol']
atm_df['IV-RV Put'] = atm_df['ATM IV Put'] - atm_df['Realized Vol']

# Calculate Put-Call IV Skew
atm_df['IV Skew'] = atm_df['ATM IV Put'] - atm_df['ATM IV Call']

fig = plot_atm_iv_realized_view(atm_df, ticker_label=ticker_str)
atm_dte_values = atm_df['Days Till Expiration'].tolist()
atm_x_positions = list(range(len(atm_df)))
atm_expiration_dates = pd.DatetimeIndex(pd.to_datetime(atm_df['Expiration Date'])).normalize()
atm_expiration_years = atm_expiration_dates.year.tolist()
atm_x_tick_labels = [
    f'{pd.to_datetime(expiration):%Y-%m-%d}<br>{int(dte)} DTE{_earnings_expiration_label(expiration)}'
    for expiration, dte in zip(atm_df['Expiration Date'], atm_dte_values)
]
atm_hover_data = list(zip(atm_df['Expiration Date'].astype(str), atm_dte_values))
for trace in fig.data:
    trace.update(
        x=atm_x_positions,
        customdata=atm_hover_data,
        hovertemplate=(
            'Expiration: %{customdata[0]}<br>'
            'DTE: %{customdata[1]} days<br>'
            'Value: %{y:.2%}<extra>%{fullData.name}</extra>'
        ),
    )
fig.update_xaxes(
    tickmode='array',
    tickvals=atm_x_positions,
    ticktext=atm_x_tick_labels,
    range=[-0.5, len(atm_x_positions) - 0.5],
    tickangle=-45,
    automargin=True,
)
fig.update_xaxes(title_text='Expiration Date (DTE)', row=3, col=1)

# Add the same year context used in Block 11 across all three panels.
atm_current_year = pd.Timestamp.today().year
atm_year_band_colors = [
    ('rgba(59, 130, 246, 0.14)', '#60A5FA'),
    ('rgba(52, 211, 153, 0.14)', '#34D399'),
    ('rgba(251, 191, 36, 0.14)', '#FBBF24'),
    ('rgba(192, 132, 252, 0.14)', '#C084FC'),
]

for band_number, year in enumerate(dict.fromkeys(atm_expiration_years)):
    year_positions = [
        i for i, expiration_year in enumerate(atm_expiration_years) if expiration_year == year
    ]
    first_position, last_position = min(year_positions), max(year_positions)
    fill_color, accent_color = atm_year_band_colors[band_number % len(atm_year_band_colors)]

    band_start = first_position - 0.5
    band_end = last_position + 0.5

    for row in (1, 2, 3):
        fig.add_vrect(
            x0=band_start,
            x1=band_end,
            fillcolor=fill_color,
            line_width=0,
            layer='below',
            row=row,
            col=1,
        )

    if first_position > 0:
        fig.add_vline(
            x=band_start,
            line_color=accent_color,
            line_width=2,
            line_dash='dot',
            row='all',
            col=1,
        )

    if year == atm_current_year:
        year_context = 'Current year'
    elif year == atm_current_year + 1:
        year_context = 'Next year'
    else:
        year_context = f'{year - atm_current_year:+d} years'

    fig.add_annotation(
        x=(first_position + last_position) / 2,
        y=1,
        xref='x3',
        yref='y3 domain',
        yshift=-6,
        text=f'<b>{year}</b><br><span style="font-size:10px">{year_context}</span>',
        showarrow=False,
        bordercolor=accent_color,
        borderwidth=1,
        bgcolor='rgba(17, 24, 39, 0.90)',
        font=dict(color=accent_color),
    )

# Mark standard quarterly expirations as in Block 11.
atm_expiration_x = dict(zip(atm_expiration_dates, atm_x_positions))
atm_quarterly_expirations = []
atm_quarterly_year_months = sorted(
    {(date.year, date.month) for date in atm_expiration_dates if date.month in (3, 6, 9, 12)}
)

for year, month in atm_quarterly_year_months:
    month_start = pd.Timestamp(year=year, month=month, day=1)
    first_friday = month_start + pd.Timedelta(days=(4 - month_start.weekday()) % 7)
    third_friday = first_friday + pd.Timedelta(weeks=2)
    quarterly_date = third_friday

    if quarterly_date not in atm_expiration_x:
        quarterly_date = third_friday - pd.Timedelta(days=1)

    if quarterly_date in atm_expiration_x:
        atm_quarterly_expirations.append((atm_expiration_x[quarterly_date], quarterly_date))

for position, quarterly_date in atm_quarterly_expirations:
    fig.add_vline(
        x=position,
        line_color='#F87171',
        line_width=3,
        line_dash='dash',
        row='all',
        col=1,
    )
    fig.add_annotation(
        x=position,
        y=1,
        xref='x',
        yref='y domain',
        xshift=7,
        yshift=-5,
        text=f'<b>Q{quarterly_date.quarter} {quarterly_date.year}</b>',
        textangle=-90,
        showarrow=False,
        xanchor='left',
        yanchor='top',
        font=dict(color='#FCA5A5', size=10),
    )
atm_earnings_band_lookup = {
    expiration: {
        'band_start': 0,
        'band_end': position,
        'marker': position,
    }
    for expiration, position in atm_expiration_x.items()
}
mark_upcoming_earnings_contracts(fig, atm_earnings_band_lookup)
apply_notebook_plot_theme(fig)
fig.show()

In [ ]:
# Block 12: IV Minus Realized by Strike
#Plot IV - Realized Volatility by Strike for Calls and Puts

from importlib import reload
from Quantapp.visualization.views.single_asset_profile.pricing.options_pricing import iv_realized_by_strike as block19_iv_realized_views

block19_iv_realized_views = reload(block19_iv_realized_views)
plot_iv_minus_realized_by_strike_view = block19_iv_realized_views.plot_iv_minus_realized_by_strike_view

# Historical price data (must be a Series indexed by date, most recent last)
price_series = pd.to_numeric(ticker['Close'], errors='coerce').copy()
price_series.index = pd.to_datetime(price_series.index, errors='coerce', utc=True).tz_convert(None).normalize()
price_series = price_series.replace([np.inf, -np.inf], np.nan).dropna().sort_index()
price_series = price_series[price_series.gt(0)].groupby(level=0).last()
log_returns = np.log(price_series / price_series.shift(1)).dropna()

# Spot price for reference line
spot_price = float(price_series.iloc[-1])
today = pd.Timestamp.today().normalize()

IV_RV_MONEYNESS_CLUSTER_WIDTH = 0.01
IV_RV_DTE_CLUSTER_BINS = [
    (2, 7, '2-7 DTE'),
    (8, 14, '8-14 DTE'),
    (15, 30, '15-30 DTE'),
    (31, 60, '31-60 DTE'),
    (61, 90, '61-90 DTE'),
    (91, 180, '91-180 DTE'),
    (181, 365, '181-365 DTE'),
    (366, 730, '366-730 DTE'),
    (731, np.inf, '731+ DTE'),
]

def _block19_dte_cluster(days_till_expiration):
    days = int(days_till_expiration)
    for cluster_order, (lower_bound, upper_bound, label) in enumerate(IV_RV_DTE_CLUSTER_BINS):
        if lower_bound <= days <= upper_bound:
            return cluster_order, label
    return len(IV_RV_DTE_CLUSTER_BINS), f'{days} DTE'

def build_iv_minus_realized_by_strike(contract_chain):
    raw_cluster_frames = []
    for exp in sorted(contract_chain.keys()):
        if contract_chain[exp] is None or contract_chain[exp].empty:
            continue
        df_sorted = contract_chain[exp].copy()
        if not {'strike', 'impliedVolatility'}.issubset(df_sorted.columns):
            continue
        df_sorted['strike'] = pd.to_numeric(df_sorted['strike'], errors='coerce')
        df_sorted['impliedVolatility'] = pd.to_numeric(df_sorted['impliedVolatility'], errors='coerce')
        df_sorted = df_sorted.dropna(subset=['strike', 'impliedVolatility'])
        df_sorted = df_sorted[df_sorted['strike'].gt(0) & df_sorted['impliedVolatility'].gt(0)].copy()
        if df_sorted.empty:
            continue
        exp_date = pd.to_datetime(exp)
        days_till_exp = int((exp_date.normalize() - today).days)

        if days_till_exp < 2 or days_till_exp > len(log_returns):
            continue

        realized_vol_n = log_returns.rolling(window=days_till_exp).std().iloc[-1] * np.sqrt(252)
        if not np.isfinite(realized_vol_n):
            continue
        df_sorted["iv_minus_realized"] = df_sorted["impliedVolatility"] - realized_vol_n
        df_sorted["days_till_expiration"] = days_till_exp
        df_sorted["realized_vol"] = float(realized_vol_n)
        df_sorted["moneyness_pct"] = (df_sorted["strike"] / spot_price) - 1.0
        df_sorted["moneyness_mid_pct"] = (
            df_sorted["moneyness_pct"] / IV_RV_MONEYNESS_CLUSTER_WIDTH
        ).round() * IV_RV_MONEYNESS_CLUSTER_WIDTH
        dte_cluster_order, dte_cluster_label = _block19_dte_cluster(days_till_exp)
        df_sorted["dte_cluster_order"] = dte_cluster_order
        df_sorted["dte_cluster"] = dte_cluster_label
        raw_cluster_frames.append(
            df_sorted[
                [
                    "strike",
                    "impliedVolatility",
                    "iv_minus_realized",
                    "realized_vol",
                    "days_till_expiration",
                    "moneyness_mid_pct",
                    "dte_cluster_order",
                    "dte_cluster",
                ]
            ]
        )

    if not raw_cluster_frames:
        return {}

    raw_cluster_frame = pd.concat(raw_cluster_frames, ignore_index=True)
    clustered = (
        raw_cluster_frame.groupby(
            ["dte_cluster_order", "dte_cluster", "moneyness_mid_pct"],
            as_index=False,
            sort=True,
        )
        .agg(
            strike=("strike", "mean"),
            iv_minus_realized=("iv_minus_realized", "mean"),
            avg_iv=("impliedVolatility", "mean"),
            avg_realized_vol=("realized_vol", "mean"),
            days_till_expiration=("days_till_expiration", "mean"),
            dte_min=("days_till_expiration", "min"),
            dte_max=("days_till_expiration", "max"),
            contract_count=("iv_minus_realized", "size"),
        )
        .sort_values(["dte_cluster_order", "strike"])
        .reset_index(drop=True)
    )
    clustered["dte_contract_count"] = clustered.groupby("dte_cluster")["contract_count"].transform("sum")
    clustered["expiration_label"] = clustered.apply(
        lambda row: f"{row['dte_cluster']} avg ({int(row['dte_contract_count']):,} contracts)",
        axis=1,
    )

    clustered_by_dte = {}
    for _, cluster_frame in clustered.groupby(["dte_cluster_order", "dte_cluster"], sort=True):
        cluster_label = cluster_frame["expiration_label"].iloc[0]
        clustered_by_dte[cluster_label] = cluster_frame.sort_values("strike").reset_index(drop=True)
    return clustered_by_dte

call_iv_minus_realized_by_expiration = build_iv_minus_realized_by_strike(call_contract_chain)
put_iv_minus_realized_by_expiration = build_iv_minus_realized_by_strike(put_contract_chain)
block19_call_cluster_count = sum(len(frame) for frame in call_iv_minus_realized_by_expiration.values())
block19_put_cluster_count = sum(len(frame) for frame in put_iv_minus_realized_by_expiration.values())
block19_call_contract_count = sum(frame['contract_count'].sum() for frame in call_iv_minus_realized_by_expiration.values())
block19_put_contract_count = sum(frame['contract_count'].sum() for frame in put_iv_minus_realized_by_expiration.values())
print(
    'Clustered Block 19 IV-RV by DTE band and '
    f'{IV_RV_MONEYNESS_CLUSTER_WIDTH:.1%} moneyness bucket | '
    f'calls: {len(call_iv_minus_realized_by_expiration)} DTE bands, '
    f'{block19_call_cluster_count:,} strike buckets from {int(block19_call_contract_count):,} contracts | '
    f'puts: {len(put_iv_minus_realized_by_expiration)} DTE bands, '
    f'{block19_put_cluster_count:,} strike buckets from {int(block19_put_contract_count):,} contracts'
)

fig = plot_iv_minus_realized_by_strike_view(
    call_iv_minus_realized_by_expiration=call_iv_minus_realized_by_expiration,
    put_iv_minus_realized_by_expiration=put_iv_minus_realized_by_expiration,
    spot_price=spot_price,
)
mark_upcoming_earnings_contracts(fig)
apply_notebook_plot_theme(fig)
fig.show()

In [ ]:
# Block 13: Put-Call IV Skew by DTE
# Plot put IV minus call IV by moneyness for each available expiration.

PUT_CALL_SKEW_MONEYNESS_LIMIT = 0.35
PUT_CALL_SKEW_MIN_SHARED_STRIKES = 3


def _block19b_resolve_dte(expiration, *frames):
    for frame in frames:
        if frame is None or frame.empty or 'Days Till Expiration' not in frame.columns:
            continue
        dte_values = pd.to_numeric(frame['Days Till Expiration'], errors='coerce').dropna()
        if not dte_values.empty:
            return int(round(float(dte_values.median())))
    return int((pd.to_datetime(expiration).normalize() - pd.Timestamp.today().normalize()).days)


def _block19b_prepare_iv_side(frame, side_name):
    required_columns = {'strike', 'impliedVolatility'}
    if frame is None or frame.empty or not required_columns.issubset(frame.columns):
        return pd.DataFrame(columns=['strike', f'{side_name}_iv'])

    prepared = frame[['strike', 'impliedVolatility']].copy()
    prepared['strike'] = pd.to_numeric(prepared['strike'], errors='coerce')
    prepared[f'{side_name}_iv'] = pd.to_numeric(prepared['impliedVolatility'], errors='coerce')
    prepared = prepared.drop(columns=['impliedVolatility']).dropna(subset=['strike', f'{side_name}_iv'])
    prepared = prepared[prepared['strike'].gt(0) & prepared[f'{side_name}_iv'].gt(0)]
    return prepared.groupby('strike', as_index=False)[f'{side_name}_iv'].mean()


def build_put_call_skew_by_dte(call_chain, put_chain, *, spot_price, moneyness_limit=None):
    spot = float(spot_price)
    if not np.isfinite(spot) or spot <= 0:
        raise ValueError('A positive spot_price is required to compute put-call skew moneyness.')

    skew_by_dte = {}
    shared_expirations = sorted(set(call_chain.keys()) & set(put_chain.keys()), key=pd.to_datetime)

    for expiration in shared_expirations:
        call_frame = call_chain.get(expiration)
        put_frame = put_chain.get(expiration)
        dte = _block19b_resolve_dte(expiration, call_frame, put_frame)
        if dte < 0:
            continue

        calls = _block19b_prepare_iv_side(call_frame, 'call')
        puts = _block19b_prepare_iv_side(put_frame, 'put')
        skew_frame = calls.merge(puts, on='strike', how='inner')
        if skew_frame.empty:
            continue

        skew_frame['moneyness_pct'] = (skew_frame['strike'] / spot) - 1.0
        if moneyness_limit is not None:
            skew_frame = skew_frame[skew_frame['moneyness_pct'].abs().le(float(moneyness_limit))]
        if len(skew_frame) < PUT_CALL_SKEW_MIN_SHARED_STRIKES:
            continue

        expiration_date = pd.to_datetime(expiration).normalize()
        skew_frame['put_call_skew'] = skew_frame['put_iv'] - skew_frame['call_iv']
        skew_frame['expiration_date'] = expiration_date
        skew_frame['days_till_expiration'] = dte
        skew_frame = skew_frame.sort_values('moneyness_pct').reset_index(drop=True)
        skew_by_dte[f'{dte} DTE - {expiration_date:%Y-%m-%d}'] = skew_frame

    return skew_by_dte


put_call_skew_by_dte = build_put_call_skew_by_dte(
    call_contract_chain,
    put_contract_chain,
    spot_price=spot_price,
    moneyness_limit=PUT_CALL_SKEW_MONEYNESS_LIMIT,
)

if not put_call_skew_by_dte:
    raise ValueError('No shared call/put strikes were available to plot put-call skew by DTE.')

put_call_skew_fig = go.Figure()
put_call_skew_labels = list(put_call_skew_by_dte.keys())
put_call_skew_overlays = {}

def _block19b_chain_for_expiration(chain, expiration_date):
    for expiration_key, frame in chain.items():
        if pd.to_datetime(expiration_key).normalize() == expiration_date:
            return expiration_key, frame
    return None, None

def _block19b_max_oi_moneyness(frame):
    if frame is None or frame.empty or not {'strike', 'openInterest'}.issubset(frame.columns):
        return None
    values = frame[['strike', 'openInterest']].apply(pd.to_numeric, errors='coerce').dropna()
    if values.empty:
        return None
    max_row = values.loc[values['openInterest'].idxmax()]
    return (float(max_row['strike']) / float(spot_price)) - 1.0

for dte_index, (label, skew_frame) in enumerate(put_call_skew_by_dte.items()):
    hover_frame = skew_frame[
        ['strike', 'call_iv', 'put_iv', 'put_call_skew', 'expiration_date', 'days_till_expiration']
    ].copy()
    hover_frame['expiration_date'] = hover_frame['expiration_date'].dt.strftime('%Y-%m-%d')
    expiration_date = pd.to_datetime(skew_frame['expiration_date'].iloc[0]).normalize()
    expiration_key, call_expiration_chain = _block19b_chain_for_expiration(call_contract_chain, expiration_date)
    _, put_expiration_chain = _block19b_chain_for_expiration(put_contract_chain, expiration_date)
    overlay_shapes = [
        dict(type='line', x0=0, x1=0, y0=0, y1=1, xref='x', yref='paper', line=dict(color='#F87171', dash='dash', width=2)),
        dict(type='line', x0=0, x1=1, y0=0, y1=0, xref='paper', yref='y', line=dict(color='#CBD5E1', dash='dash')),
    ]
    overlay_annotations = []
    if expiration_key is not None and call_expiration_chain is not None and put_expiration_chain is not None:
        implied_move_row = build_atm_implied_move_term_structure(
            {expiration_key: call_expiration_chain},
            {expiration_key: put_expiration_chain},
            [expiration_key],
            spot_price=spot_price,
        )
        if not implied_move_row.empty:
            lower_moneyness = (float(implied_move_row.iloc[0]['Straddle Lower']) / float(spot_price)) - 1.0
            upper_moneyness = (float(implied_move_row.iloc[0]['Straddle Upper']) / float(spot_price)) - 1.0
            overlay_shapes.insert(0, dict(
                type='rect', x0=lower_moneyness, x1=upper_moneyness, y0=0, y1=1,
                xref='x', yref='paper', fillcolor='#F59E0B', opacity=0.18, layer='below',
                line=dict(color='#F59E0B', width=1, dash='dot'),
            ))
            overlay_annotations.extend([
                dict(x=lower_moneyness, y=1, xref='x', yref='paper', text='ATM lower', showarrow=False, xanchor='right', yanchor='bottom', font=dict(color='#FBBF24')),
                dict(x=upper_moneyness, y=1, xref='x', yref='paper', text='ATM upper', showarrow=False, xanchor='left', yanchor='bottom', font=dict(color='#FBBF24')),
            ])
        for marker_name, marker_color, marker_x in (
            ('Max Call OI', '#60A5FA', _block19b_max_oi_moneyness(call_expiration_chain)),
            ('Max Put OI', '#4ADE80', _block19b_max_oi_moneyness(put_expiration_chain)),
        ):
            if marker_x is not None:
                overlay_shapes.append(dict(type='line', x0=marker_x, x1=marker_x, y0=0, y1=1, xref='x', yref='paper', line=dict(color=marker_color, dash='dot', width=2)))
                overlay_annotations.append(dict(x=marker_x, y=0, xref='x', yref='paper', text=marker_name, showarrow=False, yanchor='bottom', textangle=-90, font=dict(color=marker_color)))
    put_call_skew_overlays[label] = (overlay_shapes, overlay_annotations)
    trace_visible = dte_index == 0
    put_call_skew_fig.add_trace(
        go.Scatter(
            x=skew_frame['moneyness_pct'],
            y=skew_frame['call_iv'],
            customdata=hover_frame.to_numpy(),
            mode='lines+markers',
            name='Call IV',
            line=dict(color='#60A5FA'),
            visible=trace_visible,
            hovertemplate=(
                'Moneyness: %{x:+.1%}<br>Call IV: %{y:.2%}<br>'
                'Strike: %{customdata[0]:,.2f}<br>Expiration: %{customdata[4]}<br>'
                'DTE: %{customdata[5]:.0f} days<extra>Call IV</extra>'
            ),
        )
    )
    put_call_skew_fig.add_trace(
        go.Scatter(
            x=skew_frame['moneyness_pct'],
            y=skew_frame['put_iv'],
            customdata=hover_frame.to_numpy(),
            mode='lines+markers',
            name='Put IV',
            line=dict(color='#4ADE80'),
            visible=trace_visible,
            hovertemplate=(
                'Moneyness: %{x:+.1%}<br>Put IV: %{y:.2%}<br>'
                'Strike: %{customdata[0]:,.2f}<br>Expiration: %{customdata[4]}<br>'
                'DTE: %{customdata[5]:.0f} days<extra>Put IV</extra>'
            ),
        )
    )
    put_call_skew_fig.add_trace(
        go.Scatter(
            x=skew_frame['moneyness_pct'],
            y=skew_frame['put_call_skew'],
            customdata=hover_frame.to_numpy(),
            mode='lines+markers',
            name='Put - Call IV',
            line=dict(color='#C084FC', dash='dash'),
            visible=trace_visible,
            opacity=0.85,
            hovertemplate=(
                'Moneyness: %{x:+.1%}<br>'
                'Put-call skew: %{y:+.2%}<br>'
                'Strike: %{customdata[0]:,.2f}<br>'
                'Call IV: %{customdata[1]:.2%}<br>'
                'Put IV: %{customdata[2]:.2%}<br>'
                'Expiration: %{customdata[4]}<br>'
                'DTE: %{customdata[5]:.0f} days'
                '<extra>%{fullData.name}</extra>'
            ),
        )
    )

put_call_skew_buttons = []
for trace_index, label in enumerate(put_call_skew_labels):
    visibility = [False] * (len(put_call_skew_labels) * 3)
    visibility[trace_index * 3:trace_index * 3 + 3] = [True, True, True]
    overlay_shapes, overlay_annotations = put_call_skew_overlays[label]
    put_call_skew_buttons.append(
        dict(
            label=label,
            method='update',
            args=[
                {'visible': visibility},
                {'title': f'{ticker_str} Put-Call IV Skew by Moneyness - {label}', 'shapes': overlay_shapes, 'annotations': overlay_annotations},
            ],
        )
    )

put_call_skew_fig.add_hline(y=0, line_color='#CBD5E1', line_dash='dash')
put_call_skew_fig.add_vline(x=0, line_color='#F87171', line_dash='dash')
put_call_skew_fig.update_layout(
    title=f'{ticker_str} Call IV, Put IV, and Put-Call Skew - {put_call_skew_labels[0]}',
    height=850,
    xaxis_title='Moneyness vs Spot',
    yaxis_title='Implied Volatility / Put IV - Call IV',
    hovermode='closest',
    legend_title_text='Series',
    shapes=put_call_skew_overlays[put_call_skew_labels[0]][0],
    annotations=put_call_skew_overlays[put_call_skew_labels[0]][1],
    updatemenus=[
        dict(
            active=0,
            buttons=put_call_skew_buttons,
            direction='down',
            x=0,
            y=1.12,
            xanchor='left',
            yanchor='top',
        )
    ],
)
put_call_skew_fig.update_xaxes(tickformat='+.0%')
put_call_skew_fig.update_yaxes(tickformat='+.1%')
mark_upcoming_earnings_contracts(put_call_skew_fig)
apply_notebook_plot_theme(put_call_skew_fig)
put_call_skew_fig.show()

In [ ]:
# Block 14: Median IV Minus Realized

today = pd.to_datetime("today")

# --- Helper function to calculate median IV - Realized Vol ---
def calc_median_iv_minus_realized(contract_chain, filter_func=None):
    median_dict = {}
    for exp in contract_chain.keys():
        df = contract_chain[exp].copy()
        if filter_func:
            df = df[filter_func(df)]
        if df.empty:
            continue
        exp_date = pd.to_datetime(exp)
        days_till_exp = (exp_date - today).days
        if days_till_exp < 2 or days_till_exp > len(log_returns):
            continue
        realized_vol_n = log_returns.rolling(window=days_till_exp).std().iloc[-1] * np.sqrt(252)
        median_val = (df['impliedVolatility'] - realized_vol_n).median()
        median_dict[exp_date] = median_val
    return median_dict

# --- Top subplot: All strikes ---
median_calls_all = calc_median_iv_minus_realized(call_contract_chain)
median_puts_all = calc_median_iv_minus_realized(put_contract_chain)

# --- Bottom subplot: OTM strikes ---
median_calls_otm = calc_median_iv_minus_realized(
    call_contract_chain,
    filter_func=lambda df: df['strike'] > spot_price
)
median_puts_otm = calc_median_iv_minus_realized(
    put_contract_chain,
    filter_func=lambda df: df['strike'] < spot_price
)

df_calls_all = pd.DataFrame({'Expiration': list(median_calls_all.keys()), 'Median': list(median_calls_all.values()), 'Type': 'Call'})
df_puts_all = pd.DataFrame({'Expiration': list(median_puts_all.keys()), 'Median': list(median_puts_all.values()), 'Type': 'Put'})
df_all = pd.concat([df_calls_all, df_puts_all])
df_all['DTE'] = (df_all['Expiration'] - today).dt.days
df_all = df_all.sort_values(['Expiration', 'Type']).reset_index(drop=True)

df_calls_otm = pd.DataFrame({'Expiration': list(median_calls_otm.keys()), 'Median': list(median_calls_otm.values()), 'Type': 'Call_OTM'})
df_puts_otm = pd.DataFrame({'Expiration': list(median_puts_otm.keys()), 'Median': list(median_puts_otm.values()), 'Type': 'Put_OTM'})
df_otm = pd.concat([df_calls_otm, df_puts_otm])
df_otm['DTE'] = (df_otm['Expiration'] - today).dt.days
df_otm = df_otm.sort_values(['Expiration', 'Type']).reset_index(drop=True)

fig = plot_median_iv_minus_realized_view(df_all, df_otm, today=today)

# Space expirations evenly while preserving their true dates and DTE values.
median_expiration_dates = pd.DatetimeIndex(
    pd.concat([df_all['Expiration'], df_otm['Expiration']], ignore_index=True).dropna().unique()
).normalize().sort_values()
median_x_positions = list(range(len(median_expiration_dates)))
median_expiration_to_x = dict(zip(median_expiration_dates, median_x_positions))
median_dte_values = [(expiration - today).days for expiration in median_expiration_dates]
median_x_tick_labels = [
    f'{expiration:%Y-%m-%d}<br>{int(dte)} DTE{_earnings_expiration_label(expiration)}'
    for expiration, dte in zip(median_expiration_dates, median_dte_values)
]

for trace in fig.data:
    trace_expirations = pd.DatetimeIndex(pd.to_datetime(list(trace.x))).normalize()
    trace_positions = [median_expiration_to_x[expiration] for expiration in trace_expirations]
    trace_hover_data = [
        (expiration.strftime('%Y-%m-%d'), int((expiration - today).days))
        for expiration in trace_expirations
    ]
    trace.update(
        x=trace_positions,
        customdata=trace_hover_data,
        hovertemplate=(
            'Expiration: %{customdata[0]}<br>'
            'DTE: %{customdata[1]} days<br>'
            'Median IV - Realized: %{y:.2%}<extra></extra>'
        ),
    )

if median_x_positions:
    fig.update_xaxes(
        type='linear',
        tickmode='array',
        tickvals=median_x_positions,
        ticktext=median_x_tick_labels,
        range=[-0.5, len(median_x_positions) - 0.5],
        tickangle=-45,
        automargin=True,
    )
    fig.update_xaxes(title_text='Expiration Date (DTE)', row=2, col=1)

    # Add the same year shading and expiration markers used in Block 13.
    median_current_year = pd.Timestamp.today().year
    median_expiration_years = median_expiration_dates.year.tolist()
    median_year_band_colors = [
        ('rgba(59, 130, 246, 0.14)', '#60A5FA'),
        ('rgba(52, 211, 153, 0.14)', '#34D399'),
        ('rgba(251, 191, 36, 0.14)', '#FBBF24'),
        ('rgba(192, 132, 252, 0.14)', '#C084FC'),
    ]

    for band_number, year in enumerate(dict.fromkeys(median_expiration_years)):
        year_positions = [
            i for i, expiration_year in enumerate(median_expiration_years) if expiration_year == year
        ]
        first_position, last_position = min(year_positions), max(year_positions)
        fill_color, accent_color = median_year_band_colors[
            band_number % len(median_year_band_colors)
        ]

        for row in (1, 2):
            fig.add_vrect(
                x0=first_position - 0.5,
                x1=last_position + 0.5,
                fillcolor=fill_color,
                line_width=0,
                layer='below',
                row=row,
                col=1,
            )

        if first_position > 0:
            fig.add_vline(
                x=first_position - 0.5,
                line_color=accent_color,
                line_width=2,
                line_dash='dot',
                row='all',
                col=1,
            )

        if year == median_current_year:
            year_context = 'Current year'
        elif year == median_current_year + 1:
            year_context = 'Next year'
        else:
            year_context = f'{year - median_current_year:+d} years'

        fig.add_annotation(
            x=(first_position + last_position) / 2,
            y=1,
            xref='x2',
            yref='y2 domain',
            yshift=-6,
            text=f'<b>{year}</b><br><span style="font-size:10px">{year_context}</span>',
            showarrow=False,
            bordercolor=accent_color,
            borderwidth=1,
            bgcolor='rgba(17, 24, 39, 0.90)',
            font=dict(color=accent_color),
        )

    median_quarterly_expirations = []
    median_quarterly_year_months = sorted(
        {(date.year, date.month) for date in median_expiration_dates if date.month in (3, 6, 9, 12)}
    )

    for year, month in median_quarterly_year_months:
        month_start = pd.Timestamp(year=year, month=month, day=1)
        first_friday = month_start + pd.Timedelta(days=(4 - month_start.weekday()) % 7)
        third_friday = first_friday + pd.Timedelta(weeks=2)
        quarterly_date = third_friday

        if quarterly_date not in median_expiration_to_x:
            quarterly_date = third_friday - pd.Timedelta(days=1)

        if quarterly_date in median_expiration_to_x:
            median_quarterly_expirations.append(
                (median_expiration_to_x[quarterly_date], quarterly_date)
            )

    for position, quarterly_date in median_quarterly_expirations:
        fig.add_vline(
            x=position,
            line_color='#F87171',
            line_width=3,
            line_dash='dash',
            row='all',
            col=1,
        )
        fig.add_annotation(
            x=position,
            y=1,
            xref='x',
            yref='y domain',
            xshift=7,
            yshift=-5,
            text=f'<b>Q{quarterly_date.quarter} {quarterly_date.year}</b>',
            textangle=-90,
            showarrow=False,
            xanchor='left',
            yanchor='top',
            font=dict(color='#FCA5A5', size=10),
        )
median_earnings_band_lookup = {
    expiration: {
        'band_start': 0,
        'band_end': position,
        'marker': position,
    }
    for expiration, position in median_expiration_to_x.items()
}
mark_upcoming_earnings_contracts(fig, median_earnings_band_lookup)
apply_notebook_plot_theme(fig)
fig.show()

In [ ]:
# Block 15: SVI Surface
#plot SVI surface for calls and puts
from scipy.optimize import minimize
import scipy.interpolate as interp

print(f"Spot price for {ticker_str}: {spot_price}")

def svi_total_variance(k, a, b, rho, m, sigma):
    return a + b * (rho * (k - m) + np.sqrt((k - m) ** 2 + sigma ** 2))

def svi_objective(params, k, total_var):
    a, b, rho, m, sigma = params
    model_var = svi_total_variance(k, a, b, rho, m, sigma)
    return np.sum((model_var - total_var) ** 2)

def fit_svi(k, total_var):
    x0 = [0.1, 0.1, 0.0, 0.0, 0.1]
    bounds = [(-1, 1), (1e-5, 5), (-0.999, 0.999), (-5, 5), (1e-5, 5)]
    res = minimize(svi_objective, x0, args=(k, total_var), bounds=bounds, method='L-BFGS-B')
    return res.x if res.success else None

def get_realized_vol_surface(price_history, dtes):
    hist = price_history[['Close']].copy()
    hist['returns'] = np.log(hist['Close'] / hist['Close'].shift(1))
    hist.dropna(inplace=True)

    rv_data = []
    for dte in sorted(set(dtes)):
        dte = int(dte)
        sub_ret = hist['returns'].iloc[-dte:]
        if len(sub_ret) >= dte * 0.8:
            realized_vol = np.std(sub_ret) * np.sqrt(252)
            rv_data.append((dte, realized_vol))
    return pd.DataFrame(rv_data, columns=['Days Till Expiration', 'Realized Vol'])

call_df = call_contract_chain_concat.copy()
call_df = call_df[call_df['impliedVolatility'].notna() & (call_df['impliedVolatility'] > 0)].copy()
call_df['T'] = call_df['Days Till Expiration'] / 365.0

put_df = put_contract_chain_concat.copy()
put_df = put_df[put_df['impliedVolatility'].notna() & (put_df['impliedVolatility'] > 0)].copy()
put_df['T'] = put_df['Days Till Expiration'] / 365.0

def fit_svi_surface(df, spot_price):
    unique_Ts = np.sort(df['T'].unique())
    svi_params_per_T = {}

    for T in unique_Ts:
        slice_df = df[df['T'] == T]
        if len(slice_df) < 5:
            continue
        K = slice_df['strike'].values
        iv = slice_df['impliedVolatility'].values
        total_var = iv ** 2 * T
        k = np.log(K / spot_price)
        params = fit_svi(k, total_var)
        if params is not None:
            svi_params_per_T[T] = params

    Ts = np.array(sorted(svi_params_per_T.keys()))
    params_array = np.array([svi_params_per_T[T] for T in Ts])
    param_interpolators = [
        interp.interp1d(Ts, params_array[:, i], kind='linear', fill_value='extrapolate')
        for i in range(5)
    ]

    strike_grid = np.linspace(df['strike'].min(), df['strike'].max(), 50)
    T_grid = np.linspace(df['T'].min(), df['T'].max(), 30)
    iv_surface = np.full((len(T_grid), len(strike_grid)), np.nan)

    for i, T in enumerate(T_grid):
        a, b, rho, m, sigma = [f(T) for f in param_interpolators]
        k_vals = np.log(strike_grid / spot_price)
        total_var = svi_total_variance(k_vals, a, b, rho, m, sigma)
        iv_surface[i, :] = np.sqrt(total_var / T)

    return strike_grid, T_grid, iv_surface

call_strike_grid, call_T_grid, call_iv_svi_surface = fit_svi_surface(call_df, spot_price)
put_strike_grid, put_T_grid, put_iv_svi_surface = fit_svi_surface(put_df, spot_price)

all_dtes = np.unique(np.concatenate([
    call_df['Days Till Expiration'].unique(),
    put_df['Days Till Expiration'].unique()
]))

# Get RV surface from the shared top-loaded price history
rv_df = get_realized_vol_surface(ticker, all_dtes)

fig = plot_svi_surface_view(
    call_df=call_df,
    put_df=put_df,
    call_strike_grid=call_strike_grid,
    call_t_grid=call_T_grid,
    call_iv_svi_surface=call_iv_svi_surface,
    put_strike_grid=put_strike_grid,
    put_t_grid=put_T_grid,
    put_iv_svi_surface=put_iv_svi_surface,
    realized_vol_surface_df=rv_df,
    spot_price=spot_price,
    ticker_label=ticker_str,
)
mark_upcoming_earnings_contracts(fig)
apply_notebook_plot_theme(fig)
fig.show()

In [ ]:
# Block 16: Positive Deviation Screen
import pandas as pd
import numpy as np
from scipy.interpolate import griddata

# Ensure all rows and columns are displayed
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

def calc_positive_deviation_otm(df, strike_grid, T_grid, iv_svi_surface, spot_price, strike_range=100, limit=100, option_type='call'):
    """
    Calculate OTM options where market IV exceeds SVI IV, with positive deviation and distance from spot.
    """
    # Prepare SVI points and values for interpolation
    points = []
    values = []
    for i, T in enumerate(T_grid):
        for j, K in enumerate(strike_grid):
            points.append((K, T))
            values.append(iv_svi_surface[i, j])
    points = np.array(points)
    values = np.array(values)

    # Market points for interpolation
    market_points = np.vstack([df['strike'].values, df['T'].values]).T

    # Interpolate SVI IV at market points
    svi_iv_at_market = griddata(points, values, market_points, method='linear')

    df = df.copy()
    df['svi_iv'] = svi_iv_at_market

    # Filter by strike distance from spot price
    df = df[(df['strike'] >= spot_price - strike_range) & (df['strike'] <= spot_price + strike_range)]

    # Filter for OTM contracts
    if option_type.lower() == 'call':
        df = df[df['strike'] > spot_price] # calls OTM
    elif option_type.lower() == 'put':
        df = df[df['strike'] < spot_price] # puts OTM

    # Filter where market IV > SVI IV
    df_filtered = df[df['impliedVolatility'] > df['svi_iv']].copy()

    # Calculate positive deviation
    df_filtered['positive_deviation'] = df_filtered['impliedVolatility'] - df_filtered['svi_iv']

    # Add distance from spot column
    df_filtered['distance_from_spot'] = df_filtered['strike'] - spot_price

    # Sort descending by positive deviation
    df_filtered_sorted = df_filtered.sort_values(by='positive_deviation', ascending=False)

    # Select relevant columns and limit to top N
    result_df = df_filtered_sorted[['strike', 'Days Till Expiration', 'impliedVolatility', 'svi_iv', 'positive_deviation', 'distance_from_spot']].head(limit)

    return result_df.reset_index(drop=True)


# --- Example Usage ---
calls_deviations_otm = calc_positive_deviation_otm(
    df=call_df,
    strike_grid=call_strike_grid,
    T_grid=call_T_grid,
    iv_svi_surface=call_iv_svi_surface,
    spot_price=spot_price,
    strike_range=100,
    limit=100,
    option_type='call'
)

puts_deviations_otm = calc_positive_deviation_otm(
    df=put_df,
    strike_grid=put_strike_grid,
    T_grid=put_T_grid,
    iv_svi_surface=put_iv_svi_surface,
    spot_price=spot_price,
    strike_range=100,
    limit=100,
    option_type='put'
)

print("Top 100 OTM Calls with Market IV > SVI IV (within Ã‚Â±100 strikes):")
display(calls_deviations_otm)

print("\nTop 100 OTM Puts with Market IV > SVI IV (within Ã‚Â±100 strikes):")
display(puts_deviations_otm)

In [ ]:
# Block 17: First Expiration Contracts
#retreive the contracts for the first expiration date and display them as a dataframe
columns_to_drop = ['currency', 'contractSize', 'percentChange', 'change']

call_contracts_table = call_contracts.drop(columns=columns_to_drop, errors='ignore').copy()
put_contracts_table = put_contracts.drop(columns=columns_to_drop, errors='ignore').copy()

call_contracts_table

In [ ]:
# Block 18: Historical Options Chain EOD Panel
# Output is one row per (as_of_date, contract) with EOD OHLCV fields.

# Use all history available to the Massive Options Starter plan without probing older dates.
historical_price_dates = pd.DatetimeIndex(
    pd.to_datetime(ticker.index, errors='coerce', utc=True)
).tz_convert(None).dropna().normalize()
if historical_price_dates.empty:
    raise ValueError('No dated underlying history is available for historical options retrieval.')

latest_complete_day = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
HISTORICAL_END_DATE = min(historical_price_dates.max(), latest_complete_day)
MASSIVE_OPTIONS_HISTORY_YEARS = 2
massive_history_start = (
    HISTORICAL_END_DATE - pd.DateOffset(years=MASSIVE_OPTIONS_HISTORY_YEARS)
    + pd.Timedelta(days=1)
)
HISTORICAL_START_DATE = max(historical_price_dates.min(), massive_history_start)
HISTORICAL_LOOKBACK_DAYS = (HISTORICAL_END_DATE - HISTORICAL_START_DATE).days + 1
CURRENT_CHAIN_DTES = sorted(
    pd.to_numeric(options_expiration_dates['Date Till Expiration'], errors='coerce')
    .dropna()
    .round()
    .astype(int)
    .loc[lambda dtes: dtes.gt(0)]
    .unique()
    .tolist()
)
if not CURRENT_CHAIN_DTES:
    raise ValueError('The current options chain has no positive DTE values.')
HISTORICAL_TARGET_DTES = CURRENT_CHAIN_DTES
HISTORICAL_INTERPOLATION_ANCHOR_DTES = sorted(
    {max(1, min(HISTORICAL_TARGET_DTES) // 2), *HISTORICAL_TARGET_DTES, max(HISTORICAL_TARGET_DTES) + 30}
)
MAX_CONTRACTS_PER_DAY = 2 * len(HISTORICAL_INTERPOLATION_ANCHOR_DTES)
MAX_DAYS_TO_EXPIRY = max(HISTORICAL_INTERPOLATION_ANCHOR_DTES)
MONEYNESS_BAND = 0.05  # Server-side near-ATM filter keeps reference pages small.
HISTORICAL_MAX_WORKERS = 24

historical_options_panel = get_historical_options_eod_panel(
    ticker_str,
    underlying_history=ticker,
    lookback_days=HISTORICAL_LOOKBACK_DAYS,
    max_contracts_per_day=MAX_CONTRACTS_PER_DAY,
    max_days_to_expiry=MAX_DAYS_TO_EXPIRY,
    moneyness_band=MONEYNESS_BAND,
    selection_mode='nearest_atm_by_target_dte',
    target_dtes=HISTORICAL_INTERPOLATION_ANCHOR_DTES,
    reference_refresh='weekly',
    bar_retrieval_mode='per_contract_range',
    max_workers=HISTORICAL_MAX_WORKERS,
    end_date=HISTORICAL_END_DATE,
)
historical_chain = historical_options_panel.chain
historical_chain_summary = historical_options_panel.summary
historical_query_summary = historical_options_panel.query_summary
historical_retrieved_dates = pd.to_datetime(
    historical_chain['as_of_date'], errors='coerce'
).dropna() if not historical_chain.empty else pd.Series(dtype='datetime64[ns]')
print(
    f'API queries: {historical_query_summary["total_queries"]:,} total = '
    f'{historical_query_summary["reference_queries"]:,} reference + '
    f'{historical_query_summary["bar_queries"]:,} batched contract-range queries | '
    f'{historical_query_summary["reference_queries_avoided"]:,} reference and '
    f'{historical_query_summary["bar_queries_avoided"]:,} one-day bar queries avoided'
)
print(
    f'Requested full history: {HISTORICAL_START_DATE:%Y-%m-%d} to '
    f'{HISTORICAL_END_DATE:%Y-%m-%d} ({HISTORICAL_LOOKBACK_DAYS:,} calendar days)'
)
print(
    f'Historical EOD rows: {len(historical_chain):,} across '
    f'{historical_chain["as_of_date"].nunique() if not historical_chain.empty else 0:,} dates | '
    f'Retrieved range: '
    f'{historical_retrieved_dates.min():%Y-%m-%d} to {historical_retrieved_dates.max():%Y-%m-%d}'
    if not historical_retrieved_dates.empty
    else 'Historical EOD retrieval returned no rows; inspect historical_chain_summary.'
)

In [ ]:
# Block 19: Historical ATM IV Across Fixed DTE Targets
# Invert near-ATM EOD prices, then interpolate total variance to fixed tenors.

historical_iv_annual_rate = float(risk_free_rate) * 252
historical_atm_iv_by_expiration, historical_atm_iv_summary = (
    build_historical_atm_iv_history(
        historical_chain,
        annual_rate=historical_iv_annual_rate,
        min_dte=1,
        target_dtes=HISTORICAL_TARGET_DTES,
    )
)

if historical_atm_iv_summary.empty:
    raise ValueError(
        'No valid historical ATM IV observations could be calculated. '
        'Re-run Block 10 and check its historical EOD retrieval summary.'
    )

historical_atm_iv_fig = plot_historical_atm_iv_summary_view(
    historical_atm_iv_summary,
    expiration_history=historical_atm_iv_by_expiration,
    target_dtes=HISTORICAL_TARGET_DTES,
    ticker_label=ticker_str,
)
apply_notebook_plot_theme(historical_atm_iv_fig)
historical_atm_iv_fig.show()

current_atm_iv_spot_price = np.nan
if 'underlying_price' in globals() and pd.notna(underlying_price):
    current_atm_iv_spot_price = float(underlying_price)
elif 'spot_price' in globals() and pd.notna(spot_price):
    current_atm_iv_spot_price = float(spot_price)

current_atm_iv_by_expiration = build_current_atm_iv_by_expiration(
    all_contracts_concat,
    spot_price=current_atm_iv_spot_price,
    today=today if 'today' in globals() else None,
)
if current_atm_iv_by_expiration.empty:
    print('Current ATM IV DTE snapshot: no valid current options IV observations to plot.')
else:
    current_atm_iv_spread_fig = plot_current_atm_iv_spread_view(
        current_atm_iv_by_expiration,
        ticker_label=ticker_str,
    )
    apply_notebook_plot_theme(current_atm_iv_spread_fig)
    current_atm_iv_spread_fig.show()
    print(
        f'Current ATM IV DTE snapshot: {len(current_atm_iv_by_expiration)} expirations | '
        f'mean ATM IV: {current_atm_iv_by_expiration["Current Mean ATM IV"].iloc[0]:.2%} | '
        f'spread min/max: '
        f'{current_atm_iv_by_expiration["ATM IV - Current Mean ATM IV"].min():+.2%}/'
        f'{current_atm_iv_by_expiration["ATM IV - Current Mean ATM IV"].max():+.2%}'
    )

print(
    f'ATM IV history: {len(historical_atm_iv_summary)} dates | '
    f'{historical_atm_iv_by_expiration["Expiration Date"].nunique()} unique expirations | '
    f'expirations/date min/median/max: '
    f'{int(historical_atm_iv_summary["Expirations Used"].min())}/'
    f'{int(historical_atm_iv_summary["Expirations Used"].median())}/'
    f'{int(historical_atm_iv_summary["Expirations Used"].max())}'
)

In [ ]:
# Block 20: Historical IV Premium vs Realized Volatility
# Compares fixed-DTE ATM IV with trailing and forward realized volatility.

from importlib import reload
from Quantapp.visualization.views.single_asset_profile.pricing.options_pricing import historical_atm_iv as block12_historical_atm_iv_views

block12_historical_atm_iv_views = reload(block12_historical_atm_iv_views)
build_historical_iv_premium_history = block12_historical_atm_iv_views.build_historical_iv_premium_history
plot_historical_iv_premium_view = block12_historical_atm_iv_views.plot_historical_iv_premium_view

# Use every positive DTE that is actually listed in the current options chain.
BLOCK12_TARGET_DTES = list(CURRENT_CHAIN_DTES)
if not BLOCK12_TARGET_DTES:
    raise ValueError('The current options chain has no positive DTE values for Block 20.')

_, block12_historical_atm_iv_summary = build_historical_atm_iv_history(
    historical_chain,
    annual_rate=historical_iv_annual_rate,
    min_dte=1,
    target_dtes=BLOCK12_TARGET_DTES,
)
print(f'Block 20 current-chain DTEs: {BLOCK12_TARGET_DTES}')

historical_iv_premium = build_historical_iv_premium_history(
    block12_historical_atm_iv_summary,
    ticker,
    target_dtes=BLOCK12_TARGET_DTES,
)

if historical_iv_premium.empty:
    raise ValueError(
        'No historical IV premium observations could be calculated. '
        'Run Block 11 first and confirm ticker contains close prices.'
    )

historical_iv_premium_fig = plot_historical_iv_premium_view(
    historical_iv_premium,
    target_dtes=BLOCK12_TARGET_DTES,
    ticker_label=ticker_str,
)
apply_notebook_plot_theme(historical_iv_premium_fig)
historical_iv_premium_fig.show()

try:
    from arch import arch_model as block12_arch_model
except ModuleNotFoundError as exc:
    if exc.name != 'arch':
        raise
    block12_arch_model = None

from plotly.subplots import make_subplots

historical_garch_warning = None
historical_garch_errors = {}

def _block12_format_target_dte(target_dte):
    return int(target_dte) if float(target_dte).is_integer() else float(target_dte)

def _block12_trading_days_for_dte(target_dte):
    return max(2, int(round(float(target_dte) * 252.0 / 365.25)))

def _block12_prepare_log_returns(price_history):
    close = pd.to_numeric(price_history['Close'], errors='coerce').copy()
    close.index = pd.to_datetime(close.index, errors='coerce', utc=True).tz_convert(None).normalize()
    close = close.replace([np.inf, -np.inf], np.nan).dropna().sort_index()
    close = close[close.gt(0)].groupby(level=0).last()
    return np.log(close / close.shift(1)).dropna()

def _block12_zscore(series, ddof=0):
    values = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan)
    mean = values.mean()
    std = values.std(ddof=ddof)
    if pd.isna(std) or std == 0:
        return pd.Series(np.nan, index=values.index)
    return (values - mean) / std

block12_garch_model_specs = [
    {
        'label': 'GARCH(1,1)',
        'column_prefix': 'GARCH(1,1)',
        'arch_kwargs': {'vol': 'GARCH', 'p': 1, 'q': 1, 'dist': 'normal'},
        'forecast_kwargs': {},
        'color': '#38BDF8',
        'dash': 'solid',
    },
    {
        'label': 'EGARCH(1,1)',
        'column_prefix': 'EGARCH(1,1)',
        'arch_kwargs': {'vol': 'EGARCH', 'p': 1, 'o': 1, 'q': 1, 'dist': 'normal'},
        'forecast_kwargs': {'method': 'simulation', 'simulations': 500},
        'color': '#F97316',
        'dash': 'solid',
    },
    {
        'label': 'EGARCH(1,1) Student-t',
        'column_prefix': 'EGARCH(1,1) Student-t',
        'arch_kwargs': {'vol': 'EGARCH', 'p': 1, 'o': 1, 'q': 1, 'dist': 't'},
        'forecast_kwargs': {'method': 'simulation', 'simulations': 500},
        'color': '#A78BFA',
        'dash': 'dash',
    },
]


def _block12_model_column(target_label, model_spec, metric):
    return f"{target_label} DTE {model_spec['column_prefix']} {metric}"


def _block12_mean_model_column(model_spec, metric):
    return f"Mean {model_spec['column_prefix']} {metric}"


def _add_historical_garch_realized_columns(premium_frame, price_history, target_dtes):
    augmented = premium_frame.copy()
    if block12_arch_model is None:
        return augmented, "GARCH view unavailable: install the 'arch' package in this notebook kernel.", {}

    target_values = sorted({float(target_dte) for target_dte in target_dtes if float(target_dte) > 0})
    target_specs = [
        (
            target_dte,
            _block12_format_target_dte(target_dte),
            _block12_trading_days_for_dte(target_dte),
        )
        for target_dte in target_values
    ]
    if not target_specs:
        return augmented, 'GARCH view unavailable: no positive target DTEs were provided.', {}

    log_return_history = _block12_prepare_log_returns(price_history)
    if log_return_history.empty:
        return augmented, 'GARCH view unavailable: no valid close-to-close returns are available.', {}

    max_forecast_horizon = max(trading_days for _, _, trading_days in target_specs)
    minimum_fit_observations = max(252, max_forecast_horizon * 3)
    fit_lookback = min(756, len(log_return_history))
    if fit_lookback < minimum_fit_observations:
        return augmented, f'GARCH view unavailable: only {fit_lookback} return observations are available.', {}

    for _, target_label, _ in target_specs:
        for model_spec in block12_garch_model_specs:
            augmented[_block12_model_column(target_label, model_spec, 'Vol')] = np.nan
            augmented[_block12_model_column(target_label, model_spec, '- Trailing RV')] = np.nan
            augmented[_block12_model_column(target_label, model_spec, '- RV Z-Score')] = np.nan

    errors = {}
    for row_index, as_of_date in augmented['as_of_date'].items():
        date_value = pd.Timestamp(as_of_date).normalize()
        fit_returns = log_return_history.loc[:date_value].tail(fit_lookback).dropna()
        if len(fit_returns) < minimum_fit_observations:
            continue
        if not np.isfinite(fit_returns.std(ddof=0)) or fit_returns.std(ddof=0) <= 0:
            continue

        forecast_variance_by_model = {}
        for model_spec in block12_garch_model_specs:
            try:
                model_fit = block12_arch_model(
                    fit_returns.mul(100.0),
                    mean='Constant',
                    rescale=False,
                    **model_spec['arch_kwargs'],
                ).fit(disp='off')
                forecast_variance_by_model[model_spec['label']] = np.asarray(
                    model_fit.forecast(
                        horizon=max_forecast_horizon,
                        reindex=False,
                        **model_spec.get('forecast_kwargs', {}),
                    ).variance.iloc[-1],
                    dtype=float,
                )
            except Exception as exc:
                errors[f"{date_value.date()} {model_spec['label']}"] = str(exc)

        for _, target_label, trading_days in target_specs:
            trailing_rv_column = f'{target_label} DTE Trailing RV'
            trailing_rv = augmented.at[row_index, trailing_rv_column] if trailing_rv_column in augmented.columns else np.nan
            for model_spec in block12_garch_model_specs:
                forecast_variance = forecast_variance_by_model.get(model_spec['label'])
                if forecast_variance is None or len(forecast_variance) < trading_days:
                    continue
                model_daily_variance = np.clip(forecast_variance[:trading_days], a_min=0.0, a_max=None).mean()
                model_annualized_vol = np.sqrt(model_daily_variance) / 100.0 * np.sqrt(252.0)
                vol_column = _block12_model_column(target_label, model_spec, 'Vol')
                spread_column = _block12_model_column(target_label, model_spec, '- Trailing RV')
                augmented.at[row_index, vol_column] = model_annualized_vol
                if np.isfinite(model_annualized_vol) and pd.notna(trailing_rv):
                    augmented.at[row_index, spread_column] = model_annualized_vol - float(trailing_rv)

    historical_garch_spike_smoothing_reports = {}
    for _, target_label, _ in target_specs:
        trailing_rv_column = f'{target_label} DTE Trailing RV'
        trailing_rv = (
            pd.to_numeric(augmented[trailing_rv_column], errors='coerce')
            if trailing_rv_column in augmented.columns
            else pd.Series(np.nan, index=augmented.index)
        )
        for model_spec in block12_garch_model_specs:
            vol_column = _block12_model_column(target_label, model_spec, 'Vol')
            spread_column = _block12_model_column(target_label, model_spec, '- Trailing RV')
            if vol_column not in augmented.columns:
                continue
            smoothed_vol, smoothing_report = smooth_insane_series_spikes(
                augmented[vol_column],
                label=None,
                rolling_window=31,
                robust_z_threshold=8.0,
                interpolation_limit=5,
                floor=0.0,
            )
            augmented[vol_column] = smoothed_vol
            if not smoothing_report.empty:
                historical_garch_spike_smoothing_reports[vol_column] = smoothing_report
            augmented[spread_column] = pd.to_numeric(augmented[vol_column], errors='coerce') - trailing_rv
    globals()['historical_garch_spike_smoothing_reports'] = historical_garch_spike_smoothing_reports
    if historical_garch_spike_smoothing_reports:
        total_smoothed_garch_spikes = int(
            sum(report['spike_count'].sum() for report in historical_garch_spike_smoothing_reports.values())
        )
        print(
            f'Smoothed {total_smoothed_garch_spikes} insane Block 12 GARCH forecast-vol spike(s). '
            'See historical_garch_spike_smoothing_reports for details.'
        )

    for model_spec in block12_garch_model_specs:
        model_vol_columns = [_block12_model_column(target_label, model_spec, 'Vol') for _, target_label, _ in target_specs]
        model_spread_columns = [_block12_model_column(target_label, model_spec, '- Trailing RV') for _, target_label, _ in target_specs]
        model_zscore_columns = [_block12_model_column(target_label, model_spec, '- RV Z-Score') for _, target_label, _ in target_specs]
        for spread_column, zscore_column in zip(model_spread_columns, model_zscore_columns):
            augmented[zscore_column] = _block12_zscore(augmented[spread_column])
        augmented[_block12_mean_model_column(model_spec, 'Vol')] = augmented[model_vol_columns].mean(axis=1)
        augmented[_block12_mean_model_column(model_spec, '- Trailing RV')] = augmented[model_spread_columns].mean(axis=1)
        augmented[_block12_mean_model_column(model_spec, '- RV Z-Score')] = _block12_zscore(
            augmented[_block12_mean_model_column(model_spec, '- Trailing RV')]
        )

    return augmented, None, errors


def _plot_historical_garch_realized_view(premium_frame, target_dtes, ticker_label=None):
    frame = premium_frame.copy()
    frame['as_of_date'] = pd.to_datetime(frame['as_of_date'], errors='coerce')
    frame = frame.dropna(subset=['as_of_date']).sort_values('as_of_date')

    target_specs = []
    mean_model_entries = []
    for model_spec in block12_garch_model_specs:
        vol_column = _block12_mean_model_column(model_spec, 'Vol')
        spread_column = _block12_mean_model_column(model_spec, '- Trailing RV')
        zscore_column = _block12_mean_model_column(model_spec, '- RV Z-Score')
        if {vol_column, spread_column, zscore_column}.issubset(frame.columns):
            mean_model_entries.append((model_spec, vol_column, spread_column, zscore_column))
    if mean_model_entries and 'Mean Trailing RV' in frame.columns:
        target_specs.append(('Mean', 'Mean Trailing RV', mean_model_entries))

    for target_dte in sorted({float(target_dte) for target_dte in target_dtes if float(target_dte) > 0}):
        target_label = _block12_format_target_dte(target_dte)
        label = f'{target_label} DTE'
        realized_column = f'{target_label} DTE Trailing RV'
        model_entries = []
        for model_spec in block12_garch_model_specs:
            vol_column = _block12_model_column(target_label, model_spec, 'Vol')
            spread_column = _block12_model_column(target_label, model_spec, '- Trailing RV')
            zscore_column = _block12_model_column(target_label, model_spec, '- RV Z-Score')
            if {vol_column, spread_column, zscore_column}.issubset(frame.columns):
                model_entries.append((model_spec, vol_column, spread_column, zscore_column))
        if model_entries and realized_column in frame.columns:
            target_specs.append((label, realized_column, model_entries))

    if not target_specs:
        raise ValueError('No volatility-model vs realized volatility columns are available to plot.')

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.065,
        row_heights=[0.42, 0.29, 0.29],
        subplot_titles=(
            'Volatility Models vs Trailing Realized Volatility',
            'Model Forecast - Trailing Realized Volatility',
            'Model Forecast Spread Z-Score',
        ),
    )
    trace_groups = {}
    for label, realized_column, model_entries in target_specs:
        clean_columns = ['as_of_date', realized_column]
        for _, vol_column, spread_column, zscore_column in model_entries:
            clean_columns.extend([vol_column, spread_column, zscore_column])
        clean = frame[list(dict.fromkeys(clean_columns))].copy()
        for column in clean.columns:
            if column != 'as_of_date':
                clean[column] = pd.to_numeric(clean[column], errors='coerce')
        clean = clean.replace([np.inf, -np.inf], np.nan)
        start_index = len(fig.data)
        fig.add_trace(
            go.Scatter(
                x=clean['as_of_date'],
                y=clean[realized_column],
                mode='lines',
                name=f'{label} trailing RV',
                visible=False,
                line=dict(color='#F59E0B', width=2.6, dash='dash'),
                hovertemplate='Date: %{x|%Y-%m-%d}<br>Trailing RV: %{y:.2%}<extra></extra>',
            ),
            row=1,
            col=1,
        )
        for model_spec, vol_column, spread_column, zscore_column in model_entries:
            fig.add_trace(
                go.Scatter(
                    x=clean['as_of_date'],
                    y=clean[vol_column],
                    mode='lines',
                    name=f"{label} {model_spec['label']}",
                    visible=False,
                    line=dict(color=model_spec['color'], width=2.4, dash=model_spec['dash']),
                    hovertemplate=f"Date: %{{x|%Y-%m-%d}}<br>{model_spec['label']}: %{{y:.2%}}<extra></extra>",
                ),
                row=1,
                col=1,
            )
            fig.add_trace(
                go.Scatter(
                    x=clean['as_of_date'],
                    y=clean[spread_column],
                    mode='lines',
                    name=f"{label} {model_spec['label']} - RV",
                    visible=False,
                    line=dict(color=model_spec['color'], width=2.25, dash=model_spec['dash']),
                    hovertemplate=f"Date: %{{x|%Y-%m-%d}}<br>{model_spec['label']} - RV: %{{y:+.2%}}<extra></extra>",
                ),
                row=2,
                col=1,
            )
            fig.add_trace(
                go.Scatter(
                    x=clean['as_of_date'],
                    y=clean[zscore_column],
                    mode='lines',
                    name=f"{label} {model_spec['label']} z-score",
                    visible=False,
                    line=dict(color=model_spec['color'], width=2.25, dash=model_spec['dash']),
                    hovertemplate=f"Date: %{{x|%Y-%m-%d}}<br>{model_spec['label']} z-score: %{{y:+.2f}}<extra></extra>",
                ),
                row=3,
                col=1,
            )
            latest_zscore = clean.dropna(subset=[zscore_column]).tail(1)
            if not latest_zscore.empty:
                fig.add_trace(
                    go.Scatter(
                        x=latest_zscore['as_of_date'],
                        y=latest_zscore[zscore_column],
                        mode='markers',
                        name=f"{label} {model_spec['label']} latest z",
                        visible=False,
                        marker=dict(color=model_spec['color'], size=8, line=dict(color='#111827', width=1)),
                        hovertemplate=f"Date: %{{x|%Y-%m-%d}}<br>{model_spec['label']} latest z: %{{y:+.2f}}<extra></extra>",
                    ),
                    row=3,
                    col=1,
                )
        trace_groups[label] = (start_index, len(fig.data))

    default_label = '30 DTE' if '30 DTE' in trace_groups else ('Mean' if 'Mean' in trace_groups else next(iter(trace_groups)))
    start, stop = trace_groups[default_label]
    for trace_index in range(start, stop):
        fig.data[trace_index].visible = True

    dropdown_buttons = []
    for label, (start, stop) in trace_groups.items():
        visibility = [False] * len(fig.data)
        for trace_index in range(start, stop):
            visibility[trace_index] = True
        dropdown_buttons.append(
            dict(
                label=label,
                method='update',
                args=[{'visible': visibility}, {'title': f'{ticker_label + " " if ticker_label else ""}Volatility Models vs Realized Volatility - {label}'}],
            )
        )

    for yref in ('y2', 'y3'):
        fig.add_shape(
            type='line',
            xref='paper',
            yref=yref,
            x0=0,
            x1=1,
            y0=0,
            y1=0,
            line=dict(color='#9CA3AF', dash='dot', width=1),
        )
    for sigma_level in (0.5, 1.0, 1.5):
        for sign in (-1, 1):
            level = sign * sigma_level
            fig.add_shape(
                type='line',
                xref='paper',
                yref='y3',
                x0=0,
                x1=1,
                y0=level,
                y1=level,
                line=dict(color='rgba(160, 160, 160, 0.50)', dash='dash', width=1),
            )
    for y0, y1, fillcolor in [(-1.5, -1.0, 'rgba(0, 170, 0, 0.13)'), (1.0, 1.5, 'rgba(220, 0, 0, 0.13)')]:
        fig.add_shape(
            type='rect',
            xref='paper',
            yref='y3',
            x0=0,
            x1=1,
            y0=y0,
            y1=y1,
            fillcolor=fillcolor,
            line=dict(width=0),
            layer='below',
        )
    fig.update_layout(
        title=f'{ticker_label + " " if ticker_label else ""}Volatility Models vs Realized Volatility - {default_label}',
        height=960,
        hovermode='x unified',
        updatemenus=[
            dict(
                active=list(trace_groups).index(default_label),
                buttons=dropdown_buttons,
                direction='down',
                x=0,
                y=1.15,
                xanchor='left',
                yanchor='top',
                bgcolor='#111827',
                bordercolor='#374151',
                font=dict(color='white'),
            )
        ],
        margin=dict(t=120, b=50),
    )
    fig.update_yaxes(title_text='Annualized Vol', tickformat='.0%', row=1, col=1)
    fig.update_yaxes(title_text='Spread', tickformat='+.0%', row=2, col=1)
    fig.update_yaxes(title_text='Z-Score', row=3, col=1)
    fig.update_xaxes(title_text='Date', row=3, col=1)
    return fig


historical_iv_premium, historical_garch_warning, historical_garch_errors = _add_historical_garch_realized_columns(
    historical_iv_premium,
    ticker,
    BLOCK12_TARGET_DTES,
)
if historical_garch_warning:
    print(historical_garch_warning)
else:
    historical_garch_realized_fig = _plot_historical_garch_realized_view(
        historical_iv_premium,
        BLOCK12_TARGET_DTES,
        ticker_label=ticker_str,
    )
    apply_notebook_plot_theme(historical_garch_realized_fig)
    historical_garch_realized_fig.show()
    if historical_garch_errors:
        print(f'Volatility model fit failed for {len(historical_garch_errors)} date/model pairs. See historical_garch_errors for details.')

garch_preview_columns = []
for model_spec in block12_garch_model_specs:
    for prefix in ('Mean', '30 DTE'):
        for metric in ('Vol', '- Trailing RV', '- RV Z-Score'):
            garch_preview_columns.append(f"{prefix} {model_spec['column_prefix']} {metric}")

premium_preview_columns = [
    column
    for column in [
        'as_of_date',
        'Underlying Close',
        'Mean ATM IV',
        'Mean Trailing RV',
        *[column for column in garch_preview_columns if column.startswith('Mean ')],
        'Mean Forward RV',
        'Mean Ex-Ante Premium',
        'Mean Ex-Post Premium',
        'Mean IV Rank',
        'Mean IV Percentile',
        'Mean Implied Move %',
        'Mean Implied Move',
        'Mean Implied Lower Price',
        'Mean Implied Upper Price',
        '30 DTE ATM IV',
        '30 DTE Trailing RV',
        *[column for column in garch_preview_columns if column.startswith('30 DTE ')],
        '30 DTE Forward RV',
        '30 DTE Ex-Ante Premium',
        '30 DTE Ex-Post Premium',
        '30 DTE IV Rank',
        '30 DTE IV Percentile',
        '30 DTE Implied Move %',
        '30 DTE Implied Move',
        '30 DTE Implied Lower Price',
        '30 DTE Implied Upper Price',
    ]
    if column in historical_iv_premium.columns
]
display(historical_iv_premium[premium_preview_columns].tail())

mean_ex_ante_premium = (
    historical_iv_premium['Mean Ex-Ante Premium'].mean()
    if 'Mean Ex-Ante Premium' in historical_iv_premium.columns
    else np.nan
)
mean_ex_post_premium = (
    historical_iv_premium['Mean Ex-Post Premium'].mean()
    if 'Mean Ex-Post Premium' in historical_iv_premium.columns
    else np.nan
)
latest_premium_row = historical_iv_premium.dropna(subset=['as_of_date']).sort_values('as_of_date').tail(1)
latest_30d_rank = (
    latest_premium_row['30 DTE IV Rank'].iloc[0]
    if not latest_premium_row.empty and '30 DTE IV Rank' in latest_premium_row.columns
    else np.nan
)
latest_30d_percentile = (
    latest_premium_row['30 DTE IV Percentile'].iloc[0]
    if not latest_premium_row.empty and '30 DTE IV Percentile' in latest_premium_row.columns
    else np.nan
)
print(
    f'IV premium history: {len(historical_iv_premium)} dates | '
    f'ex-ante mean premium: {mean_ex_ante_premium:.2%} | '
    f'ex-post mean premium: {mean_ex_post_premium:.2%} | '
    f'latest 30D IV rank/percentile: {latest_30d_rank:.1%}/{latest_30d_percentile:.1%}'
)

In [ ]:
# Block 21: Historical Candlesticks With Implied and Forward RV Move Ranges
# Overlays selected fixed-DTE implied, trailing RV, and forward RV move bands on historical price candles.

from importlib import reload
from Quantapp.visualization.views.single_asset_profile.pricing.options_pricing import historical_atm_iv as block13_historical_atm_iv_views

block13_historical_atm_iv_views = reload(block13_historical_atm_iv_views)
plot_historical_implied_move_candlestick_view = block13_historical_atm_iv_views.plot_historical_implied_move_candlestick_view

block13_historical_iv_premium = historical_iv_premium
block13_forward_realized_columns = [
    f"{int(target_dte) if float(target_dte).is_integer() else float(target_dte)} DTE Forward Realized Move %"
    for target_dte in HISTORICAL_TARGET_DTES
]
if not set(block13_forward_realized_columns).issubset(block13_historical_iv_premium.columns):
    block13_historical_iv_premium = block13_historical_atm_iv_views.build_historical_iv_premium_history(
        historical_atm_iv_summary,
        ticker,
        target_dtes=HISTORICAL_TARGET_DTES,
    )

historical_implied_move_candlestick_fig = plot_historical_implied_move_candlestick_view(
    block13_historical_iv_premium,
    ticker,
    target_dtes=HISTORICAL_TARGET_DTES,
    ticker_label=ticker_str,
    show_implied_move_shading=False,
)
apply_notebook_plot_theme(historical_implied_move_candlestick_fig)
historical_implied_move_candlestick_fig.show()

In [ ]:
# Block 22: Distribution Analysis Objective

#This notebook is being used to evaluate the distribution shape, tail risk, and volatility behavior of a single asset.
#Original Risk Analysis blocks included here: 6-10.

In [ ]:
# Block 23: Distribution Analytics Helpers

# Notebook-local risk distribution analytics. Kept here because this workflow is Distribution-specific.
class RiskDistributionAnalytics:
    """Prepare rolling drawdown/skew/kurtosis/gini metric sets for visualization."""

    @staticmethod
    def _coerce_ohlc_frame(data) -> pd.DataFrame:
        if not isinstance(data, pd.DataFrame):
            raise TypeError("price_frame must be a pandas DataFrame with 'Open' and 'Close' columns.")

        required_columns = {"Open", "Close"}
        missing_columns = sorted(required_columns.difference(data.columns))
        if missing_columns:
            raise ValueError(
                f"price_frame is missing required columns: {missing_columns}"
            )

        frame = data.loc[:, ["Open", "Close"]].dropna().sort_index()
        if frame.empty:
            raise ValueError("price_frame is empty after dropping NaN values.")
        return frame

    @staticmethod
    def _normalize_windows(windows):
        if windows is None:
            raise ValueError("windows must be provided.")
        try:
            windows_iter = list(windows)
        except TypeError:
            windows_iter = [windows]

        normalized = []
        for window in windows_iter:
            try:
                w = int(window)
            except (TypeError, ValueError):
                continue
            if w > 0:
                normalized.append(w)

        normalized = list(dict.fromkeys(normalized))
        if not normalized:
            raise ValueError("No valid windows supplied.")
        return normalized

    @staticmethod
    def _select_default_window(window_options, default_window=None):
        if default_window in window_options:
            return default_window
        if 200 in window_options:
            return 200
        return max(window_options)

    @staticmethod
    def _normalize_confidence_levels(confidence_levels):
        if confidence_levels is None:
            confidence_levels = (0.95, 0.99)

        try:
            level_iter = list(confidence_levels)
        except TypeError:
            level_iter = [confidence_levels]

        normalized = []
        for level in level_iter:
            try:
                confidence = float(level)
            except (TypeError, ValueError):
                continue

            if confidence > 1:
                confidence = confidence / 100.0
            if 0 < confidence < 1:
                normalized.append(confidence)

        normalized = list(dict.fromkeys(normalized))
        if not normalized:
            raise ValueError("No valid confidence levels supplied.")
        return sorted(normalized)

    @staticmethod
    def _select_default_confidence(confidence_levels, default_confidence=None):
        if default_confidence is not None:
            try:
                confidence = float(default_confidence)
            except (TypeError, ValueError):
                confidence = None
            else:
                if confidence > 1:
                    confidence = confidence / 100.0
                if confidence in confidence_levels:
                    return confidence

        if 0.95 in confidence_levels:
            return 0.95
        return confidence_levels[0]

    @staticmethod
    def _normalize_horizon_sessions(horizon_sessions):
        try:
            horizon = int(horizon_sessions)
        except (TypeError, ValueError) as exc:
            raise ValueError("horizon_sessions must be a non-negative integer.") from exc
        if horizon < 0:
            raise ValueError("horizon_sessions must be a non-negative integer.")
        return horizon

    @staticmethod
    def _resolve_current_reference_price(frame):
        reference_price = frame.attrs.get("current_session_reference_price")
        if reference_price is None:
            current_session_date = frame.attrs.get("current_session_date")
            last_frame_date = pd.Timestamp(frame.index[-1]).normalize()
            if current_session_date is not None:
                current_session_date = pd.Timestamp(current_session_date).normalize()
                if current_session_date > last_frame_date:
                    reference_price = frame["Close"].iloc[-1]
                else:
                    reference_price = frame["Close"].shift(1).iloc[-1]
            else:
                reference_price = frame["Close"].shift(1).iloc[-1]
        if pd.isna(reference_price):
            reference_price = frame["Close"].iloc[-1]
        return float(reference_price)

    @staticmethod
    def _build_session_holding_period_frame(frame, horizon_sessions):
        horizon = RiskDistributionAnalytics._normalize_horizon_sessions(horizon_sessions)
        close_series = frame["Close"].astype(float)
        if horizon == 0:
            reference_series = frame["Open"].astype(float)
            exit_close = close_series
        else:
            reference_series = close_series.shift(1)
            exit_close = close_series.shift(-(horizon - 1))
        holding_frame = pd.DataFrame(
            {
                "session_open": reference_series,
                "session_close": exit_close,
                "session_return": exit_close.div(reference_series).sub(1.0),
            }
        ).dropna()
        if holding_frame.empty:
            raise ValueError(
                f"price_frame does not contain enough completed sessions for a {horizon}-session horizon."
            )
        return holding_frame

    def build_risk_distribution_context(self, close_series, windows, default_window=None):
        """
        Build drawdown/skew/kurtosis/gini rolling metrics for each window.

        Returns
        -------
        dict
            {
                "close_series": pd.Series,
                "daily_returns": pd.Series,
                "windows": list[int],
                "default_window": int,
                "metrics_by_window": dict[int, dict[str, pd.Series]],
            }
        """
        close = coerce_close_series(close_series)
        window_options = self._normalize_windows(windows)
        default_window = self._select_default_window(window_options, default_window=default_window)

        daily_returns = close.pct_change().dropna()

        def return_quantile(level):
            def quantile_metric(series):
                return series.quantile(level)

            return quantile_metric

        metrics_by_window = {}
        for window in window_options:
            max_drawdown_series = calculate_textbook_rolling_max_drawdown(close, window=window).dropna()
            rolling_return_q10 = compute.rolling(daily_returns, metric=return_quantile(0.10), window=window).dropna()
            rolling_return_q25 = compute.rolling(daily_returns, metric=return_quantile(0.25), window=window).dropna()
            rolling_return_median = compute.rolling(daily_returns, metric=pd.Series.median, window=window).dropna()
            rolling_return_q75 = compute.rolling(daily_returns, metric=return_quantile(0.75), window=window).dropna()
            rolling_return_q90 = compute.rolling(daily_returns, metric=return_quantile(0.90), window=window).dropna()
            rolling_skew = compute.rolling(
                daily_returns,
                metric=lambda series: skew(series, bias=False),
                window=window,
            ).dropna()
            rolling_kurtosis = compute.rolling(
                daily_returns,
                metric=lambda series: kurtosis(series, fisher=True, bias=False),
                window=window,
            ).dropna()
            rolling_gini = compute.rolling(daily_returns, metric=gini_coefficient, window=window).dropna()

            metrics_by_window[window] = {
                "daily_returns": daily_returns.copy(),
                "return_q10": rolling_return_q10,
                "return_q25": rolling_return_q25,
                "return_median": rolling_return_median,
                "return_q75": rolling_return_q75,
                "return_q90": rolling_return_q90,
                "max_drawdown": max_drawdown_series,
                "skew_z": calculate_zscore(rolling_skew),
                "kurtosis_z": calculate_zscore(rolling_kurtosis),
                "gini_z": calculate_zscore(rolling_gini),
            }

        return {
            "close_series": close,
            "daily_returns": daily_returns,
            "windows": window_options,
            "default_window": default_window,
            "metrics_by_window": metrics_by_window,
        }

    def build_value_at_risk_context(
        self,
        close_series,
        windows,
        confidence_levels=(0.95, 0.99),
        default_window=None,
        default_confidence=None,
        position_value=None,
    ):
        """
        Build rolling historical VaR / CVaR (Expected Shortfall) metrics for each window and confidence level.

        Returns
        -------
        dict
            {
                "close_series": pd.Series,
                "daily_returns": pd.Series,
                "windows": list[int],
                "default_window": int,
                "confidence_levels": list[float],
                "default_confidence": float,
                "metrics_by_window": dict[int, dict[float, dict[str, pd.Series]]],
                "summary_table": pd.DataFrame,
                "position_value": float | None,
            }
        """
        close = coerce_close_series(close_series)
        window_options = self._normalize_windows(windows)
        default_window = self._select_default_window(window_options, default_window=default_window)
        confidence_levels = self._normalize_confidence_levels(confidence_levels)
        default_confidence = self._select_default_confidence(
            confidence_levels,
            default_confidence=default_confidence,
        )

        daily_returns = close.pct_change(fill_method=None).dropna()
        metrics_by_window = {}
        summary_rows = []

        for window in window_options:
            metrics_by_confidence = {}
            for confidence in confidence_levels:
                alpha = 1 - confidence
                metric_set = calculate_historical_var_metrics(daily_returns, window=window, alpha=alpha)

                if position_value is not None:
                    metric_set["var_dollar"] = metric_set["var"] * float(position_value)
                    metric_set["expected_shortfall_dollar"] = (
                        metric_set["expected_shortfall"] * float(position_value)
                    )

                metrics_by_confidence[confidence] = metric_set

                var_series = metric_set["var"].dropna()
                es_series = metric_set["expected_shortfall"].dropna()
                breach_series = metric_set["breaches"].dropna()
                rolling_breach_rate = metric_set["rolling_breach_rate"].dropna()

                summary_row = {
                    "VaR Lookback Window": window,
                    "Confidence": f"{confidence:.0%}",
                    "Latest VaR": var_series.iloc[-1] if not var_series.empty else np.nan,
                    "Latest CVaR": es_series.iloc[-1] if not es_series.empty else np.nan,
                    "Observed Breach Rate": breach_series.mean() if not breach_series.empty else np.nan,
                    "Expected Breach Rate": alpha,
                    "Latest Rolling Breach Rate": (
                        rolling_breach_rate.iloc[-1] if not rolling_breach_rate.empty else np.nan
                    ),
                }

                if position_value is not None:
                    var_dollar = metric_set["var_dollar"].dropna()
                    es_dollar = metric_set["expected_shortfall_dollar"].dropna()
                    summary_row["Latest VaR Dollar"] = var_dollar.iloc[-1] if not var_dollar.empty else np.nan
                    summary_row["Latest CVaR Dollar"] = es_dollar.iloc[-1] if not es_dollar.empty else np.nan

                summary_rows.append(summary_row)

            metrics_by_window[window] = metrics_by_confidence

        summary_table = pd.DataFrame(summary_rows)
        if not summary_table.empty:
            summary_table = summary_table.sort_values(["VaR Lookback Window", "Confidence"]).reset_index(drop=True)

        return {
            "close_series": close,
            "daily_returns": daily_returns,
            "windows": window_options,
            "default_window": default_window,
            "confidence_levels": confidence_levels,
            "default_confidence": default_confidence,
            "metrics_by_window": metrics_by_window,
            "summary_table": summary_table,
            "position_value": position_value,
        }

    def build_session_probability_cone_context(
        self,
        price_frame,
        window=200,
        interval_confidence_levels=(0.50, 0.80, 0.90, 0.95),
        var_confidence_levels=(0.95, 0.99),
        anchor_price=None,
        latest_price=None,
        session_date=None,
    ):
        """
        Build an open-anchored probability cone for the latest session using trailing
        open-to-close returns from completed sessions.

        Returns
        -------
        dict
            {
                "session_date": pd.Timestamp,
                "window": int,
                "effective_window": int,
                "anchor_price": float,
                "latest_price": float,
                "sample_returns": pd.Series,
                "interval_confidence_levels": list[float],
                "var_confidence_levels": list[float],
                "intervals": dict[float, dict[str, float]],
                "var_levels": dict[float, dict[str, float]],
                "median_return": float,
                "median_price": float,
                "summary_table": pd.DataFrame,
            }
        """
        frame = self._coerce_ohlc_frame(price_frame)
        try:
            window = int(window)
        except (TypeError, ValueError) as exc:
            raise ValueError("window must be a positive integer.") from exc
        if window <= 0:
            raise ValueError("window must be a positive integer.")

        interval_confidence_levels = self._normalize_confidence_levels(interval_confidence_levels)
        var_confidence_levels = self._normalize_confidence_levels(var_confidence_levels)

        session_returns = frame["Close"].div(frame["Open"]).sub(1.0).dropna()
        if len(session_returns) < 2:
            raise ValueError(
                "At least two sessions with valid open and close prices are required "
                "to build a current-session probability cone."
            )

        historical_returns = session_returns.iloc[:-1].dropna()
        if historical_returns.empty:
            raise ValueError("No completed session returns are available for the cone sample.")

        effective_window = min(window, len(historical_returns))
        sample_returns = historical_returns.tail(effective_window)
        session_date = pd.Timestamp(
            frame.attrs.get("current_session_date", frame.index[-1])
            if session_date is None
            else session_date
        )
        latest_price = float(
            frame.attrs.get("current_session_latest_price", frame["Close"].iloc[-1])
            if latest_price is None
            else latest_price
        )

        if anchor_price is None:
            anchor_price = float(
                frame.attrs.get("current_session_anchor_price", frame["Open"].iloc[-1])
            )
        else:
            anchor_price = float(anchor_price)

        if not np.isfinite(anchor_price) or anchor_price <= 0:
            raise ValueError("anchor_price must be a positive finite value.")

        median_return = float(sample_returns.median())
        median_price = anchor_price * (1.0 + median_return)

        interval_map = {}
        summary_rows = []
        for confidence in interval_confidence_levels:
            tail_probability = (1.0 - confidence) / 2.0
            lower_return = float(sample_returns.quantile(tail_probability))
            upper_return = float(sample_returns.quantile(1.0 - tail_probability))
            lower_price = anchor_price * (1.0 + lower_return)
            upper_price = anchor_price * (1.0 + upper_return)
            interval_map[confidence] = {
                "lower_return": lower_return,
                "upper_return": upper_return,
                "lower_price": lower_price,
                "upper_price": upper_price,
            }
            summary_rows.append(
                {
                    "Metric": f"{confidence:.0%} Central Range",
                    "Lower Return": lower_return,
                    "Upper Return": upper_return,
                    "Lower Price": lower_price,
                    "Upper Price": upper_price,
                }
            )

        var_level_map = {}
        for confidence in var_confidence_levels:
            alpha = 1.0 - confidence
            metric_set = calculate_historical_var_metrics(
                sample_returns,
                window=effective_window,
                alpha=alpha,
            )
            var_series = metric_set["var"].dropna()
            expected_shortfall_series = metric_set["expected_shortfall"].dropna()

            var_loss = float(var_series.iloc[-1]) if not var_series.empty else np.nan
            expected_shortfall_loss = (
                float(expected_shortfall_series.iloc[-1])
                if not expected_shortfall_series.empty
                else np.nan
            )
            if pd.isna(var_loss):
                var_loss = float(max(-float(sample_returns.quantile(alpha)), 0.0))
            if pd.isna(expected_shortfall_loss):
                tail_values = sample_returns[sample_returns <= sample_returns.quantile(alpha)]
                tail_mean = float(tail_values.mean()) if not tail_values.empty else -var_loss
                expected_shortfall_loss = float(max(-tail_mean, var_loss))

            var_return = -var_loss
            expected_shortfall_return = -expected_shortfall_loss
            var_price = anchor_price * (1.0 + var_return)
            expected_shortfall_price = anchor_price * (1.0 + expected_shortfall_return)

            var_level_map[confidence] = {
                "var_loss": var_loss,
                "var_return": var_return,
                "var_price": var_price,
                "expected_shortfall_loss": expected_shortfall_loss,
                "expected_shortfall_return": expected_shortfall_return,
                "expected_shortfall_price": expected_shortfall_price,
            }
            summary_rows.extend(
                [
                    {
                        "Metric": f"{confidence:.0%} VaR Floor",
                        "Lower Return": var_return,
                        "Upper Return": np.nan,
                        "Lower Price": var_price,
                        "Upper Price": np.nan,
                    },
                    {
                        "Metric": f"{confidence:.0%} CVaR Floor",
                        "Lower Return": expected_shortfall_return,
                        "Upper Return": np.nan,
                        "Lower Price": expected_shortfall_price,
                        "Upper Price": np.nan,
                    },
                ]
            )

        summary_rows.insert(
            0,
            {
                "Metric": "Session Open",
                "Lower Return": 0.0,
                "Upper Return": 0.0,
                "Lower Price": anchor_price,
                "Upper Price": anchor_price,
            },
        )
        summary_rows.insert(
            1,
            {
                "Metric": "Latest Session Price",
                "Lower Return": (latest_price / anchor_price) - 1.0 if anchor_price else np.nan,
                "Upper Return": np.nan,
                "Lower Price": latest_price,
                "Upper Price": np.nan,
            },
        )

        summary_table = pd.DataFrame(summary_rows)

        return {
            "session_date": session_date,
            "window": window,
            "effective_window": effective_window,
            "anchor_price": anchor_price,
            "latest_price": latest_price,
            "sample_returns": sample_returns,
            "interval_confidence_levels": interval_confidence_levels,
            "var_confidence_levels": var_confidence_levels,
            "intervals": interval_map,
            "var_levels": var_level_map,
            "median_return": median_return,
            "median_price": median_price,
            "summary_table": summary_table,
        }

    def build_trade_range_probability_context(
        self,
        price_frame,
        window=200,
        horizon_sessions=1,
        interval_confidence_levels=(0.95, 0.99),
        tail_confidence_levels=(0.95, 0.99),
        anchor_price=None,
        latest_price=None,
        session_date=None,
    ):
        """
        Build a two-sided close-based session probability context for long and short
        decision support using trailing completed holding-period returns.
        """
        frame = self._coerce_ohlc_frame(price_frame)
        try:
            window = int(window)
        except (TypeError, ValueError) as exc:
            raise ValueError("window must be a positive integer.") from exc
        if window <= 0:
            raise ValueError("window must be a positive integer.")
        horizon_sessions = self._normalize_horizon_sessions(horizon_sessions)

        interval_confidence_levels = self._normalize_confidence_levels(interval_confidence_levels)
        tail_confidence_levels = self._normalize_confidence_levels(tail_confidence_levels)

        holding_frame = self._build_session_holding_period_frame(frame, horizon_sessions)
        session_returns = holding_frame["session_return"]
        if len(session_returns) < 2:
            raise ValueError(
                "At least two completed holding-period returns are required "
                "to build a current-session probability range."
            )

        historical_returns = session_returns.iloc[:-1].dropna()
        if historical_returns.empty:
            raise ValueError("No completed holding-period returns are available for the trade range sample.")

        effective_window = min(window, len(historical_returns))
        sample_returns = historical_returns.tail(effective_window)
        session_date = pd.Timestamp(
            frame.attrs.get("current_session_date", frame.index[-1])
            if session_date is None
            else session_date
        )
        latest_price = float(
            frame.attrs.get("current_session_latest_price", frame["Close"].iloc[-1])
            if latest_price is None
            else latest_price
        )

        if anchor_price is None:
            if horizon_sessions == 0:
                anchor_price = frame.attrs.get("current_session_anchor_price", frame["Open"].iloc[-1])
            else:
                anchor_price = self._resolve_current_reference_price(frame)
        else:
            anchor_price = float(anchor_price)

        if not np.isfinite(anchor_price) or anchor_price <= 0:
            raise ValueError("anchor_price must be a positive finite value.")

        median_return = float(sample_returns.median())
        median_price = anchor_price * (1.0 + median_return)

        interval_map = {}
        range_rows = []
        for confidence in interval_confidence_levels:
            tail_probability = (1.0 - confidence) / 2.0
            lower_return = float(sample_returns.quantile(tail_probability))
            upper_return = float(sample_returns.quantile(1.0 - tail_probability))
            lower_price = anchor_price * (1.0 + lower_return)
            upper_price = anchor_price * (1.0 + upper_return)
            interval_map[confidence] = {
                "lower_return": lower_return,
                "upper_return": upper_return,
                "lower_price": lower_price,
                "upper_price": upper_price,
            }
            range_rows.append(
                {
                    "Confidence": f"{confidence:.0%}",
                    "Lower Return": lower_return,
                    "Upper Return": upper_return,
                    "Lower Close": lower_price,
                    "Upper Close": upper_price,
                }
            )

        long_tail_map = {}
        short_tail_map = {}
        tail_rows = []
        for confidence in tail_confidence_levels:
            alpha = 1.0 - confidence

            lower_cutoff = float(sample_returns.quantile(alpha))
            upper_cutoff = float(sample_returns.quantile(1.0 - alpha))

            lower_tail_values = sample_returns[sample_returns <= lower_cutoff]
            upper_tail_values = sample_returns[sample_returns >= upper_cutoff]

            long_cvar_return = (
                float(lower_tail_values.mean())
                if not lower_tail_values.empty
                else lower_cutoff
            )
            short_cvar_return = (
                float(upper_tail_values.mean())
                if not upper_tail_values.empty
                else upper_cutoff
            )

            long_tail_map[confidence] = {
                "var_return": lower_cutoff,
                "var_price": anchor_price * (1.0 + lower_cutoff),
                "expected_shortfall_return": long_cvar_return,
                "expected_shortfall_price": anchor_price * (1.0 + long_cvar_return),
            }
            short_tail_map[confidence] = {
                "var_return": upper_cutoff,
                "var_price": anchor_price * (1.0 + upper_cutoff),
                "expected_shortfall_return": short_cvar_return,
                "expected_shortfall_price": anchor_price * (1.0 + short_cvar_return),
            }

            tail_rows.append(
                {
                    "Confidence": f"{confidence:.0%}",
                    "Long VaR Floor": long_tail_map[confidence]["var_price"],
                    "Long CVaR Floor": long_tail_map[confidence]["expected_shortfall_price"],
                    "Short VaR Ceiling": short_tail_map[confidence]["var_price"],
                    "Short CVaR Ceiling": short_tail_map[confidence]["expected_shortfall_price"],
                }
            )

        return {
            "session_date": session_date,
            "window": window,
            "horizon_sessions": horizon_sessions,
            "effective_window": effective_window,
            "anchor_price": anchor_price,
            "latest_price": latest_price,
            "sample_returns": sample_returns,
            "return_basis": "open_to_close" if horizon_sessions == 0 else "close_to_close",
            "interval_confidence_levels": interval_confidence_levels,
            "tail_confidence_levels": tail_confidence_levels,
            "intervals": interval_map,
            "long_tail_levels": long_tail_map,
            "short_tail_levels": short_tail_map,
            "median_return": median_return,
            "median_price": median_price,
            "range_summary_table": pd.DataFrame(range_rows),
            "tail_summary_table": pd.DataFrame(tail_rows),
        }

    def build_trade_range_history_context(
        self,
        price_frame,
        window=200,
        windows=None,
        horizon_sessions=1,
        interval_confidence_levels=(0.95, 0.99),
        tail_confidence_levels=(0.95, 0.99),
        default_window=None,
    ):
        """
        Build ex-ante historical session trade-range metrics for one or more rolling
        windows using only information available before each entry session.
        """
        frame = self._coerce_ohlc_frame(price_frame)
        window_seed = windows if windows is not None else [window]
        window_options = self._normalize_windows(window_seed)
        default_window = self._select_default_window(
            window_options,
            default_window=default_window,
        )
        horizon_sessions = self._normalize_horizon_sessions(horizon_sessions)
        interval_confidence_levels = self._normalize_confidence_levels(interval_confidence_levels)
        tail_confidence_levels = self._normalize_confidence_levels(tail_confidence_levels)

        holding_frame = self._build_session_holding_period_frame(frame, horizon_sessions)
        open_series = holding_frame["session_open"]
        close_series = holding_frame["session_close"]
        session_returns = holding_frame["session_return"]
        max_window = max(window_options)
        minimum_observations = max_window + horizon_sessions - 1
        if len(session_returns) <= minimum_observations:
            raise ValueError(
                f"Not enough completed sessions to build a {max_window}-session lookback with a "
                f"{horizon_sessions}-session horizon."
            )

        def lower_tail_mean(values, quantile_level):
            arr = np.asarray(values, dtype=float)
            arr = arr[~np.isnan(arr)]
            if arr.size == 0:
                return np.nan
            cutoff = np.quantile(arr, quantile_level)
            tail_values = arr[arr <= cutoff]
            if tail_values.size == 0:
                return cutoff
            return tail_values.mean()

        def upper_tail_mean(values, quantile_level):
            arr = np.asarray(values, dtype=float)
            arr = arr[~np.isnan(arr)]
            if arr.size == 0:
                return np.nan
            cutoff = np.quantile(arr, quantile_level)
            tail_values = arr[arr >= cutoff]
            if tail_values.size == 0:
                return cutoff
            return tail_values.mean()

        all_confidences = sorted(set(interval_confidence_levels).union(tail_confidence_levels))
        metrics_by_window = {}
        for rolling_window in window_options:
            metrics_by_confidence = {}
            for confidence in all_confidences:
                alpha = 1.0 - confidence
                interval_alpha = (1.0 - confidence) / 2.0
                shift_periods = horizon_sessions

                lower_interval_return = (
                    session_returns.rolling(rolling_window).quantile(interval_alpha).shift(shift_periods).dropna()
                )
                upper_interval_return = (
                    session_returns.rolling(rolling_window).quantile(1.0 - interval_alpha).shift(shift_periods).dropna()
                )
                lower_var_return = session_returns.rolling(rolling_window).quantile(alpha).shift(shift_periods).dropna()
                upper_var_return = session_returns.rolling(rolling_window).quantile(1.0 - alpha).shift(shift_periods).dropna()
                lower_expected_shortfall_return = (
                    session_returns
                    .rolling(rolling_window)
                    .apply(lambda values, q=alpha: lower_tail_mean(values, q), raw=True)
                    .shift(shift_periods)
                    .dropna()
                )
                upper_expected_shortfall_return = (
                    session_returns
                    .rolling(rolling_window)
                    .apply(lambda values, q=1.0 - alpha: upper_tail_mean(values, q), raw=True)
                    .shift(shift_periods)
                    .dropna()
                )

                aligned_returns = session_returns.reindex(lower_var_return.index).dropna()
                lower_var_return = lower_var_return.reindex(aligned_returns.index)
                upper_var_return = upper_var_return.reindex(aligned_returns.index)
                lower_breaches = aligned_returns.lt(lower_var_return).astype(float)
                upper_breaches = aligned_returns.gt(upper_var_return).astype(float)
                either_side_breaches = lower_breaches.add(upper_breaches, fill_value=0.0).clip(upper=1.0)
                lower_rolling_breach_rate = lower_breaches.rolling(rolling_window).mean().dropna()
                upper_rolling_breach_rate = upper_breaches.rolling(rolling_window).mean().dropna()
                either_side_rolling_breach_rate = (
                    either_side_breaches.rolling(rolling_window).mean().dropna()
                )

                lower_expected_breach_rate = pd.Series(
                    data=np.full(len(lower_rolling_breach_rate.index), alpha, dtype=float),
                    index=lower_rolling_breach_rate.index,
                )
                upper_expected_breach_rate = pd.Series(
                    data=np.full(len(upper_rolling_breach_rate.index), alpha, dtype=float),
                    index=upper_rolling_breach_rate.index,
                )
                either_side_expected_breach_rate = pd.Series(
                    data=np.full(
                        len(either_side_rolling_breach_rate.index),
                        min(1.0, 2.0 * alpha),
                        dtype=float,
                    ),
                    index=either_side_rolling_breach_rate.index,
                )

                open_for_interval = open_series.reindex(lower_interval_return.index)
                open_for_var = open_series.reindex(lower_var_return.index)
                open_for_es = open_series.reindex(lower_expected_shortfall_return.index)

                metrics_by_confidence[confidence] = {
                    "session_returns": session_returns,
                    "session_open": open_series,
                    "session_close": close_series,
                    "lower_interval_return": lower_interval_return,
                    "upper_interval_return": upper_interval_return,
                    "lower_interval_price": open_for_interval.mul(1.0 + lower_interval_return),
                    "upper_interval_price": open_for_interval.mul(1.0 + upper_interval_return),
                    "lower_var_return": lower_var_return,
                    "upper_var_return": upper_var_return,
                    "lower_var_price": open_for_var.mul(1.0 + lower_var_return),
                    "upper_var_price": open_for_var.mul(1.0 + upper_var_return),
                    "lower_expected_shortfall_return": lower_expected_shortfall_return,
                    "upper_expected_shortfall_return": upper_expected_shortfall_return,
                    "lower_expected_shortfall_price": open_for_es.mul(1.0 + lower_expected_shortfall_return),
                    "upper_expected_shortfall_price": open_for_es.mul(1.0 + upper_expected_shortfall_return),
                    "lower_breaches": lower_breaches,
                    "upper_breaches": upper_breaches,
                    "either_side_breaches": either_side_breaches,
                    "lower_rolling_breach_rate": lower_rolling_breach_rate,
                    "upper_rolling_breach_rate": upper_rolling_breach_rate,
                    "either_side_rolling_breach_rate": either_side_rolling_breach_rate,
                    "lower_expected_breach_rate": lower_expected_breach_rate,
                    "upper_expected_breach_rate": upper_expected_breach_rate,
                    "either_side_expected_breach_rate": either_side_expected_breach_rate,
                }

            metrics_by_window[rolling_window] = metrics_by_confidence

        return {
            "window": default_window,
            "windows": window_options,
            "default_window": default_window,
            "horizon_sessions": horizon_sessions,
            "interval_confidence_levels": interval_confidence_levels,
            "tail_confidence_levels": tail_confidence_levels,
            "metrics_by_confidence": metrics_by_window[default_window],
            "metrics_by_window": metrics_by_window,
            "return_basis": "open_to_close" if horizon_sessions == 0 else "close_to_close",
            "session_returns": session_returns,
            "session_open": open_series,
            "session_close": close_series,
        }
risk_distribution_analytics = RiskDistributionAnalytics()


In [ ]:
# Block 24: Plotly Display Helpers

PLOTLY_NOTEBOOK_CONFIG = {"responsive": True, "scrollZoom": True}
for renderer_name in ("plotly_mimetype", "notebook", "notebook_connected", "jupyterlab"):
    try:
        pio.renderers[renderer_name].config = PLOTLY_NOTEBOOK_CONFIG.copy()
    except Exception:
        pass

def show_plotly_figure(fig, *, config=None, **layout_kwargs):
    merged_config = PLOTLY_NOTEBOOK_CONFIG.copy()
    if config:
        merged_config.update(config)
    fig.update_layout(autosize=True, **layout_kwargs)
    fig.show(config=merged_config)


In [ ]:
# Block 25: Distribution Parameters

distribution_params = {
    "ticker_str": ticker_str,
    "interval": interval,
    "period": period,
    "risk_free_ticker": risk_free_ticker,
    "benchmark_tickers": list(options_params.get("benchmark_tickers", ["SPY"])),
    "trading_strategy": options_params.get("trading_strategy", "position"),
    "length_of_plots": options_params.get("length_of_plots", 20),
    "var_position_value": options_params.get("var_position_value"),
}

benchmark_tickers = list(distribution_params["benchmark_tickers"])
trading_strategy = distribution_params["trading_strategy"]
length_of_plots = distribution_params["length_of_plots"]
var_position_value = distribution_params["var_position_value"]

distribution_params


In [ ]:
# Block 26: Distribution Timeframe Setup

strategy = str(trading_strategy).strip().lower()
time_frame_map = resolve_time_frame_map(strategy)
time_frame_short = time_frame_map["short"]
time_frame_mid = time_frame_map["mid"]
time_frame_long = time_frame_map["long"]

return_frequencies = ('monthly', 'weekly', 'daily')

In [ ]:
# Block 27: Distribution Data Load and Alignment

# Reuse the top-loaded asset history when it has the OHLC fields this section needs.
shared_ticker_history = globals().get('ticker')
required_distribution_columns = {'Open', 'High', 'Low', 'Close'}
if (
    isinstance(shared_ticker_history, pd.DataFrame)
    and not shared_ticker_history.empty
    and required_distribution_columns.issubset(shared_ticker_history.columns)
):
    ticker = shared_ticker_history.copy()
else:
    ticker = qa_yf.Ticker(ticker_str).history(period=period, interval=interval)
    ticker, distribution_ticker_spike_smoothing_report = smooth_insane_price_spikes(
        ticker,
        label=ticker_str,
    )
vix = qa_yf.Ticker('^VIX').history(period=period, interval=interval)
risk_free_proxy = qa_yf.Ticker(risk_free_ticker).history(period=period, interval=interval)
ticker = helper.simplify_datetime_index(ticker)
ticker_intraday = None
if helper.is_futures_ticker(ticker_str):
    try:
        ticker_intraday = qa_yf.Ticker(ticker_str).history(period='60d', interval='30m')
    except Exception:
        ticker_intraday = None
ticker_trade_range_source = helper.build_equity_like_trade_range_source(
    ticker_str,
    ticker,
    intraday_frame=ticker_intraday,
)
vix = helper.simplify_datetime_index(vix)
risk_free_proxy = helper.simplify_datetime_index(risk_free_proxy)
if risk_free_proxy.empty or 'Close' not in risk_free_proxy:
    raise ValueError(f"No risk-free history available for {risk_free_ticker}.")
risk_free_annual_yield = risk_free_proxy['Close'].dropna().sort_index().div(100)
risk_free_daily_rate = ((1 + risk_free_annual_yield) ** (1 / 252) - 1).shift(1)

# Download benchmark data once and keep it in collections for downstream cells
benchmark_tickers = normalize_benchmark_tickers(benchmark_tickers, ticker_str, include_asset=True)
benchmark_data, skipped_benchmarks = load_benchmark_data(benchmark_tickers, period, interval, helper)
if skipped_benchmarks:
    print(f'Skipped benchmarks with no data: {skipped_benchmarks}')

analysis_index, ticker, vix, benchmark_data = align_series_to_common_index(ticker, vix, benchmark_data)
risk_free_daily_rate = risk_free_daily_rate.reindex(ticker.index).ffill()

# Calculate asset and benchmark returns for the frequencies used elsewhere in the notebook
ticker_returns = {frequency: series_transforms.returns(ticker, frequency=frequency) for frequency in return_frequencies}
ticker_monthly_returns = ticker_returns['monthly']
ticker_weekly_returns = ticker_returns['weekly']
ticker_daily_returns = ticker_returns['daily']

benchmark_returns = {
    symbol: {frequency: series_transforms.returns(frame, frequency=frequency) for frequency in return_frequencies}
    for symbol, frame in benchmark_data.items()
}

vix_returns = {frequency: series_transforms.returns(vix, frequency=frequency) for frequency in return_frequencies}
vix_monthly_returns = vix_returns['monthly']
vix_weekly_returns = vix_returns['weekly']
vix_daily_returns = vix_returns['daily']

In [ ]:
# Block 28: Return Distribution Shape

distribution_window_options = [21, 50, 200]

distribution_context = risk_distribution_analytics.build_risk_distribution_context(
    close_series=ticker['Close'],
    windows=distribution_window_options,
    default_window=200 if 200 in distribution_window_options else max(distribution_window_options),
)

fig = plot_distribution_shape_zscores_view(
    metrics_by_window=distribution_context['metrics_by_window'],
    window_options=distribution_context['windows'],
    default_window=distribution_context['default_window'],
    ticker_label=ticker_str,
    include_return_panel=False,
)
show_plotly_figure(fig)

In [ ]:
# Block 29: Trade Range Methods and Selectors



trade_range_price_frame = globals().get('ticker_trade_range_source', ticker)[['Open', 'Close']].dropna().copy()
trade_range_window_candidates = [21, 50, 200]
trade_range_max_supported_window = max(1, len(trade_range_price_frame) - 1)
trade_range_window_options = [window for window in trade_range_window_candidates if window <= trade_range_max_supported_window]
if not trade_range_window_options:
    raise ValueError(
        f'Trade-range analysis needs at least 21 completed sessions. '
        f'Only {trade_range_max_supported_window} are available for {ticker_str} '
        f'using {trade_range_price_frame.attrs.get("session_mode", "the current session mode")}.'
    )
trade_range_default_window = 50 if 50 in trade_range_window_options else max(trade_range_window_options)
trade_range_default_horizon = 0
trade_range_interval_levels = [0.85, 0.95, 0.99]
trade_range_tail_levels = [0.85, 0.95, 0.99]
trade_range_confidence_options = sorted(set(trade_range_interval_levels).union(trade_range_tail_levels))
trade_range_default_confidence = 0.95 if 0.95 in trade_range_confidence_options else trade_range_confidence_options[0]

trade_range_history_context = risk_distribution_analytics.build_trade_range_history_context(
    price_frame=trade_range_price_frame,
    windows=trade_range_window_options,
    horizon_sessions=trade_range_default_horizon,
    interval_confidence_levels=trade_range_interval_levels,
    tail_confidence_levels=trade_range_tail_levels,
    default_window=trade_range_default_window,
)
trade_range_history_contexts_by_horizon = {
    int(trade_range_default_horizon): trade_range_history_context,
}

trade_range_history_fig = plot_trade_range_history_profile(
    history_context=trade_range_history_context,
    ticker_label=ticker_str,
)
import copy as _copy

trade_range_cone_contexts_by_key = {
    (int(trade_range_default_window), int(trade_range_default_horizon)): (
        risk_distribution_analytics.build_trade_range_probability_context(
            price_frame=trade_range_price_frame,
            window=int(trade_range_default_window),
            horizon_sessions=int(trade_range_default_horizon),
            interval_confidence_levels=trade_range_interval_levels,
            tail_confidence_levels=trade_range_tail_levels,
        )
    )
}

trade_range_window = trade_range_history_context['default_window']
trade_range_horizon = int(trade_range_default_horizon)
trade_range_context = trade_range_cone_contexts_by_key[(trade_range_window, trade_range_horizon)]

range_summary = trade_range_context['range_summary_table'].copy()
tail_summary = trade_range_context['tail_summary_table'].copy()

range_summary = range_summary.rename(columns={
    'Lower Close': 'Lower Exit Price',
    'Upper Close': 'Upper Exit Price',
})
tail_summary = tail_summary.rename(columns={
    'Confidence': 'Tail Confidence',
})

if not range_summary.empty:
    for column in ('Lower Return', 'Upper Return'):
        if column in range_summary.columns:
            range_summary[column] = range_summary[column].map(
                lambda value: f'{value:.2%}' if pd.notna(value) else value
            )
    for column in ('Lower Exit Price', 'Upper Exit Price'):
        if column in range_summary.columns:
            range_summary[column] = range_summary[column].map(
                lambda value: f'${value:,.2f}' if pd.notna(value) else value
            )

if not tail_summary.empty:
    for column in ('Long VaR Floor', 'Long CVaR Floor', 'Short VaR Ceiling', 'Short CVaR Ceiling'):
        if column in tail_summary.columns:
            tail_summary[column] = tail_summary[column].map(
                lambda value: f'${value:,.2f}' if pd.notna(value) else value
            )

# Build the optional GARCH(1,1)-normal variant so it can share the same selector.
for _garch_output_name in (
    'garch_trade_history_context',
    'formatted_garch_model_summary',
    'garch_range_summary',
    'garch_tail_summary',
    'garch_trade_combined_fig',
    'garch_trade_range_fig',
    'garch_cone_components_by_key',
    'garch_trade_history_contexts_by_key',
    'build_garch_trade_range_history_context',
    'garch_cone_components_by_window',
    'garch_default_components',
):
    globals().pop(_garch_output_name, None)

garch_option_error = None
try:


    from scipy.stats import norm

    try:
        from arch import arch_model
    except ModuleNotFoundError as exc:
        if exc.name != 'arch':
            raise
        raise ImportError(
            "The GARCH option requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
        ) from exc

    trade_range_garch_window = trade_range_default_window if 'trade_range_default_window' in globals() else 50
    trade_range_garch_window_options = trade_range_window_options if 'trade_range_window_options' in globals() else [trade_range_garch_window]
    trade_range_garch_default_horizon = 1
    trade_range_garch_interval_levels = trade_range_interval_levels if 'trade_range_interval_levels' in globals() else [0.95, 0.99]
    trade_range_garch_tail_levels = trade_range_tail_levels if 'trade_range_tail_levels' in globals() else [0.95, 0.99]

    # Fit the model to completed close-to-close returns so the forecast stays aligned with the simplified trade-range view.
    garch_trade_frame = trade_range_price_frame.dropna().copy()
    garch_session_returns = garch_trade_frame['Close'].pct_change().dropna()
    if len(garch_session_returns) < 3:
        raise ValueError(
            'The GARCH option needs at least three sessions with valid close prices to fit a GARCH trade-range variant.'
        )

    garch_completed_frame = garch_trade_frame.iloc[:-1].copy()
    garch_completed_returns = garch_trade_frame['Close'].pct_change().iloc[:-1].dropna()
    if len(garch_completed_returns) < 50:
        raise ValueError(
            f'The GARCH option needs at least 50 completed sessions to fit a stable GARCH trade-range variant. Only {len(garch_completed_returns)} are available.'
        )

    # Historical diagnostics use a filtered full-sample GARCH fit across completed sessions.
    garch_history_model = arch_model(
        garch_completed_returns.mul(100.0),
        mean='Constant',
        vol='GARCH',
        p=1,
        q=1,
        dist='normal',
        rescale=False,
    )
    garch_history_fit = garch_history_model.fit(disp='off')
    garch_history_mean_return = float(garch_history_fit.params.get('mu', 0.0) / 100.0)
    garch_history_sigma_series = pd.Series(
        garch_history_fit.conditional_volatility / 100.0,
        index=garch_completed_returns.index,
    )
    garch_history_open = garch_trade_frame['Close'].shift(1).astype(float).reindex(garch_completed_returns.index)
    garch_history_close = garch_completed_frame['Close'].astype(float).reindex(garch_completed_returns.index)
    garch_current_history_forecast = garch_history_fit.forecast(horizon=1, reindex=False)
    garch_current_history_mean_return = float(garch_current_history_forecast.mean.iloc[-1, 0] / 100.0)
    garch_current_history_sigma_return = float(np.sqrt(garch_current_history_forecast.variance.iloc[-1, 0]) / 100.0)
    if not np.isfinite(garch_current_history_sigma_return) or garch_current_history_sigma_return <= 0:
        raise ValueError(
            'The GARCH option could not produce a positive current-session GARCH volatility forecast for the historical diagnostics.'
        )
    garch_current_history_date = garch_trade_frame.index[-1]
    garch_current_history_return = float((garch_trade_frame.attrs.get('current_session_latest_price', garch_trade_frame['Close'].iloc[-1]) / risk_distribution_analytics._resolve_current_reference_price(garch_trade_frame)) - 1.0)
    garch_history_metrics_by_confidence = {}
    for confidence in sorted(set(trade_range_garch_interval_levels).union(trade_range_garch_tail_levels)):
        alpha = 1.0 - confidence
        interval_alpha = (1.0 - confidence) / 2.0
        interval_z = float(norm.ppf(interval_alpha))
        z_alpha = float(norm.ppf(alpha))
        pdf_alpha = float(norm.pdf(z_alpha))

        lower_interval_return = garch_history_mean_return + (garch_history_sigma_series * interval_z)
        upper_interval_return = garch_history_mean_return - (garch_history_sigma_series * interval_z)
        lower_var_return = garch_history_mean_return + (garch_history_sigma_series * z_alpha)
        upper_var_return = garch_history_mean_return - (garch_history_sigma_series * z_alpha)
        lower_expected_shortfall_return = garch_history_mean_return - (garch_history_sigma_series * (pdf_alpha / alpha))
        upper_expected_shortfall_return = garch_history_mean_return + (garch_history_sigma_series * (pdf_alpha / alpha))

        current_lower_var_return = garch_current_history_mean_return + (garch_current_history_sigma_return * z_alpha)
        current_upper_var_return = garch_current_history_mean_return - (garch_current_history_sigma_return * z_alpha)
        current_lower_expected_shortfall_return = (
            garch_current_history_mean_return - (garch_current_history_sigma_return * (pdf_alpha / alpha))
        )
        current_upper_expected_shortfall_return = (
            garch_current_history_mean_return + (garch_current_history_sigma_return * (pdf_alpha / alpha))
        )

        lower_var_return_with_current = pd.concat([
            lower_var_return,
            pd.Series([current_lower_var_return], index=[garch_current_history_date]),
        ]).sort_index()
        upper_var_return_with_current = pd.concat([
            upper_var_return,
            pd.Series([current_upper_var_return], index=[garch_current_history_date]),
        ]).sort_index()
        lower_expected_shortfall_return_with_current = pd.concat([
            lower_expected_shortfall_return,
            pd.Series([current_lower_expected_shortfall_return], index=[garch_current_history_date]),
        ]).sort_index()
        upper_expected_shortfall_return_with_current = pd.concat([
            upper_expected_shortfall_return,
            pd.Series([current_upper_expected_shortfall_return], index=[garch_current_history_date]),
        ]).sort_index()

        lower_breaches = garch_completed_returns.lt(lower_var_return).astype(float)
        upper_breaches = garch_completed_returns.gt(upper_var_return).astype(float)
        lower_breaches_with_current = pd.concat([
            lower_breaches,
            pd.Series([float(garch_current_history_return < current_lower_var_return)], index=[garch_current_history_date]),
        ]).sort_index()
        upper_breaches_with_current = pd.concat([
            upper_breaches,
            pd.Series([float(garch_current_history_return > current_upper_var_return)], index=[garch_current_history_date]),
        ]).sort_index()
        either_side_breaches_with_current = lower_breaches_with_current.add(upper_breaches_with_current, fill_value=0.0).clip(upper=1.0)
        lower_rolling_breach_rate = lower_breaches_with_current.rolling(trade_range_garch_window).mean().dropna()
        upper_rolling_breach_rate = upper_breaches_with_current.rolling(trade_range_garch_window).mean().dropna()
        either_side_rolling_breach_rate = either_side_breaches_with_current.rolling(trade_range_garch_window).mean().dropna()

        lower_expected_breach_rate = pd.Series(
            data=np.full(len(lower_rolling_breach_rate.index), alpha, dtype=float),
            index=lower_rolling_breach_rate.index,
        )
        upper_expected_breach_rate = pd.Series(
            data=np.full(len(upper_rolling_breach_rate.index), alpha, dtype=float),
            index=upper_rolling_breach_rate.index,
        )
        either_side_expected_breach_rate = pd.Series(
            data=np.full(len(either_side_rolling_breach_rate.index), min(1.0, 2.0 * alpha), dtype=float),
            index=either_side_rolling_breach_rate.index,
        )

        open_for_interval = garch_history_open.reindex(lower_interval_return.index)
        open_for_var = garch_history_open.reindex(lower_var_return.index)
        open_for_es = garch_history_open.reindex(lower_expected_shortfall_return.index)

        garch_history_metrics_by_confidence[confidence] = {
            'session_returns': garch_completed_returns,
            'session_open': garch_history_open,
            'session_close': garch_history_close,
            'lower_interval_return': lower_interval_return,
            'upper_interval_return': upper_interval_return,
            'lower_interval_price': open_for_interval.mul(1.0 + lower_interval_return),
            'upper_interval_price': open_for_interval.mul(1.0 + upper_interval_return),
            'lower_var_return': lower_var_return_with_current,
            'upper_var_return': upper_var_return_with_current,
            'lower_var_price': open_for_var.mul(1.0 + lower_var_return),
            'upper_var_price': open_for_var.mul(1.0 + upper_var_return),
            'lower_expected_shortfall_return': lower_expected_shortfall_return_with_current,
            'upper_expected_shortfall_return': upper_expected_shortfall_return_with_current,
            'lower_expected_shortfall_price': open_for_es.mul(1.0 + lower_expected_shortfall_return),
            'upper_expected_shortfall_price': open_for_es.mul(1.0 + upper_expected_shortfall_return),
            'lower_breaches': lower_breaches_with_current,
            'upper_breaches': upper_breaches_with_current,
            'either_side_breaches': either_side_breaches_with_current,
            'lower_rolling_breach_rate': lower_rolling_breach_rate,
            'upper_rolling_breach_rate': upper_rolling_breach_rate,
            'either_side_rolling_breach_rate': either_side_rolling_breach_rate,
            'lower_expected_breach_rate': lower_expected_breach_rate,
            'upper_expected_breach_rate': upper_expected_breach_rate,
            'either_side_expected_breach_rate': either_side_expected_breach_rate,
        }

    garch_trade_history_context = {
        'window': trade_range_garch_window,
        'horizon_sessions': trade_range_garch_default_horizon,
        'interval_confidence_levels': trade_range_garch_interval_levels,
        'tail_confidence_levels': trade_range_garch_tail_levels,
        'metrics_by_confidence': garch_history_metrics_by_confidence,
        'session_returns': garch_session_returns,
        'session_open': garch_trade_frame['Close'].shift(1).astype(float),
        'session_close': garch_trade_frame['Close'].astype(float),
    }
    garch_trade_history_fig = plot_trade_range_history_profile(
        history_context=garch_trade_history_context,
        ticker_label=f'{ticker_str} GARCH(1,1) Normal',
    )
    garch_trade_history_contexts_by_key = {
        (int(trade_range_garch_window), int(trade_range_garch_default_horizon)): garch_trade_history_context,
    }

    def build_garch_trade_range_history_context(window, horizon_sessions=1):
        window = int(window)
        horizon_sessions = int(horizon_sessions)
        history_cache_key = (window, horizon_sessions)
        cached_history_context = garch_trade_history_contexts_by_key.get(history_cache_key)
        if cached_history_context is not None:
            return cached_history_context

        if horizon_sessions == int(trade_range_garch_default_horizon):
            history_session_returns = garch_session_returns.copy()
            history_session_open = garch_trade_frame['Close'].shift(1).astype(float).reindex(history_session_returns.index)
            history_session_close = garch_trade_frame['Close'].astype(float).reindex(history_session_returns.index)
            history_fit_returns = garch_completed_returns.copy()
            include_current_session = True
        else:
            history_holding_frame = risk_distribution_analytics._build_session_holding_period_frame(
                garch_trade_frame,
                horizon_sessions,
            )
            history_session_returns = history_holding_frame['session_return'].dropna()
            history_session_open = history_holding_frame['session_open'].astype(float).reindex(history_session_returns.index)
            history_session_close = history_holding_frame['session_close'].astype(float).reindex(history_session_returns.index)
            history_fit_returns = history_session_returns.copy()
            include_current_session = False

        if len(history_fit_returns) < 50:
            raise ValueError(
                f'The GARCH option needs at least 50 completed {horizon_sessions}-session returns to build the selected history view. Only {len(history_fit_returns)} are available.'
            )

        history_model = arch_model(
            history_fit_returns.mul(100.0),
            mean='Constant',
            vol='GARCH',
            p=1,
            q=1,
            dist='normal',
            rescale=False,
        )
        history_fit = history_model.fit(disp='off')
        history_mean_return = float(history_fit.params.get('mu', 0.0) / 100.0)
        history_sigma_series = pd.Series(
            history_fit.conditional_volatility / 100.0,
            index=history_fit_returns.index,
        )

        current_history_date = None
        current_history_return = None
        current_history_mean_return = None
        current_history_sigma_return = None
        if include_current_session:
            current_history_forecast = history_fit.forecast(horizon=1, reindex=False)
            current_history_mean_return = float(current_history_forecast.mean.iloc[-1, 0] / 100.0)
            current_history_sigma_return = float(np.sqrt(current_history_forecast.variance.iloc[-1, 0]) / 100.0)
            if not np.isfinite(current_history_sigma_return) or current_history_sigma_return <= 0:
                raise ValueError(
                    'The GARCH option could not produce a positive current-session GARCH volatility forecast for the selected history view.'
                )
            current_history_date = garch_trade_frame.index[-1]
            current_history_return = float(garch_session_returns.iloc[-1])

        history_metrics_by_confidence = {}
        for confidence in sorted(set(trade_range_garch_interval_levels).union(trade_range_garch_tail_levels)):
            alpha = 1.0 - confidence
            interval_alpha = (1.0 - confidence) / 2.0
            interval_z = float(norm.ppf(interval_alpha))
            z_alpha = float(norm.ppf(alpha))
            pdf_alpha = float(norm.pdf(z_alpha))

            lower_interval_return = history_mean_return + (history_sigma_series * interval_z)
            upper_interval_return = history_mean_return - (history_sigma_series * interval_z)
            lower_var_return = history_mean_return + (history_sigma_series * z_alpha)
            upper_var_return = history_mean_return - (history_sigma_series * z_alpha)
            lower_expected_shortfall_return = history_mean_return - (history_sigma_series * (pdf_alpha / alpha))
            upper_expected_shortfall_return = history_mean_return + (history_sigma_series * (pdf_alpha / alpha))

            if include_current_session:
                current_lower_var_return = current_history_mean_return + (current_history_sigma_return * z_alpha)
                current_upper_var_return = current_history_mean_return - (current_history_sigma_return * z_alpha)
                current_lower_expected_shortfall_return = (
                    current_history_mean_return - (current_history_sigma_return * (pdf_alpha / alpha))
                )
                current_upper_expected_shortfall_return = (
                    current_history_mean_return + (current_history_sigma_return * (pdf_alpha / alpha))
                )

                lower_var_return_for_plot = pd.concat([
                    lower_var_return,
                    pd.Series([current_lower_var_return], index=[current_history_date]),
                ]).sort_index()
                upper_var_return_for_plot = pd.concat([
                    upper_var_return,
                    pd.Series([current_upper_var_return], index=[current_history_date]),
                ]).sort_index()
                lower_expected_shortfall_return_for_plot = pd.concat([
                    lower_expected_shortfall_return,
                    pd.Series([current_lower_expected_shortfall_return], index=[current_history_date]),
                ]).sort_index()
                upper_expected_shortfall_return_for_plot = pd.concat([
                    upper_expected_shortfall_return,
                    pd.Series([current_upper_expected_shortfall_return], index=[current_history_date]),
                ]).sort_index()

                lower_breaches = history_fit_returns.lt(lower_var_return).astype(float)
                upper_breaches = history_fit_returns.gt(upper_var_return).astype(float)
                lower_breaches_for_plot = pd.concat([
                    lower_breaches,
                    pd.Series([float(current_history_return < current_lower_var_return)], index=[current_history_date]),
                ]).sort_index()
                upper_breaches_for_plot = pd.concat([
                    upper_breaches,
                    pd.Series([float(current_history_return > current_upper_var_return)], index=[current_history_date]),
                ]).sort_index()
            else:
                lower_var_return_for_plot = lower_var_return.copy()
                upper_var_return_for_plot = upper_var_return.copy()
                lower_expected_shortfall_return_for_plot = lower_expected_shortfall_return.copy()
                upper_expected_shortfall_return_for_plot = upper_expected_shortfall_return.copy()
                lower_breaches_for_plot = history_session_returns.lt(lower_var_return_for_plot).astype(float)
                upper_breaches_for_plot = history_session_returns.gt(upper_var_return_for_plot).astype(float)

            either_side_breaches_for_plot = lower_breaches_for_plot.add(upper_breaches_for_plot, fill_value=0.0).clip(upper=1.0)
            lower_rolling_breach_rate = lower_breaches_for_plot.rolling(window).mean().dropna()
            upper_rolling_breach_rate = upper_breaches_for_plot.rolling(window).mean().dropna()
            either_side_rolling_breach_rate = either_side_breaches_for_plot.rolling(window).mean().dropna()

            lower_expected_breach_rate = pd.Series(
                data=np.full(len(lower_rolling_breach_rate.index), alpha, dtype=float),
                index=lower_rolling_breach_rate.index,
            )
            upper_expected_breach_rate = pd.Series(
                data=np.full(len(upper_rolling_breach_rate.index), alpha, dtype=float),
                index=upper_rolling_breach_rate.index,
            )
            either_side_expected_breach_rate = pd.Series(
                data=np.full(len(either_side_rolling_breach_rate.index), min(1.0, 2.0 * alpha), dtype=float),
                index=either_side_rolling_breach_rate.index,
            )

            open_for_interval = history_session_open.reindex(lower_interval_return.index)
            open_for_var = history_session_open.reindex(lower_var_return_for_plot.index)
            open_for_es = history_session_open.reindex(lower_expected_shortfall_return_for_plot.index)

            history_metrics_by_confidence[confidence] = {
                'session_returns': history_session_returns,
                'session_open': history_session_open,
                'session_close': history_session_close,
                'lower_interval_return': lower_interval_return,
                'upper_interval_return': upper_interval_return,
                'lower_interval_price': open_for_interval.mul(1.0 + lower_interval_return),
                'upper_interval_price': open_for_interval.mul(1.0 + upper_interval_return),
                'lower_var_return': lower_var_return_for_plot,
                'upper_var_return': upper_var_return_for_plot,
                'lower_var_price': open_for_var.mul(1.0 + lower_var_return_for_plot),
                'upper_var_price': open_for_var.mul(1.0 + upper_var_return_for_plot),
                'lower_expected_shortfall_return': lower_expected_shortfall_return_for_plot,
                'upper_expected_shortfall_return': upper_expected_shortfall_return_for_plot,
                'lower_expected_shortfall_price': open_for_es.mul(1.0 + lower_expected_shortfall_return_for_plot),
                'upper_expected_shortfall_price': open_for_es.mul(1.0 + upper_expected_shortfall_return_for_plot),
                'lower_breaches': lower_breaches_for_plot,
                'upper_breaches': upper_breaches_for_plot,
                'either_side_breaches': either_side_breaches_for_plot,
                'lower_rolling_breach_rate': lower_rolling_breach_rate,
                'upper_rolling_breach_rate': upper_rolling_breach_rate,
                'either_side_rolling_breach_rate': either_side_rolling_breach_rate,
                'lower_expected_breach_rate': lower_expected_breach_rate,
                'upper_expected_breach_rate': upper_expected_breach_rate,
                'either_side_expected_breach_rate': either_side_expected_breach_rate,
            }

        history_context = {
            'window': window,
            'horizon_sessions': horizon_sessions,
            'interval_confidence_levels': trade_range_garch_interval_levels,
            'tail_confidence_levels': trade_range_garch_tail_levels,
            'metrics_by_confidence': history_metrics_by_confidence,
            'session_returns': history_session_returns,
            'session_open': history_session_open,
            'session_close': history_session_close,
        }
        garch_trade_history_contexts_by_key[history_cache_key] = history_context
        return history_context
    # Current-session forecast uses the selected trailing fit window so the cone can switch windows like Block 7.
    def build_garch_trade_range_cone_components(window, horizon_sessions=1):
        window = int(window)
        horizon_sessions = max(1, int(horizon_sessions))

        garch_fit_returns = garch_completed_returns.tail(window).dropna()
        garch_fit_window = len(garch_fit_returns)
        if garch_fit_window < 21:
            raise ValueError(
                f'The GARCH option needs at least 21 completed sessions inside the trailing fit window. Only {garch_fit_window} are available for the {window}-session view.'
            )

        garch_trade_model = arch_model(
            garch_fit_returns.mul(100.0),
            mean='Constant',
            vol='GARCH',
            p=1,
            q=1,
            dist='normal',
            rescale=False,
        )
        garch_trade_fit = garch_trade_model.fit(disp='off')
        garch_trade_forecast = garch_trade_fit.forecast(horizon=horizon_sessions, reindex=False)

        forecast_mean_values = np.asarray(garch_trade_forecast.mean.iloc[-1], dtype=float)
        forecast_variance_values = np.asarray(garch_trade_forecast.variance.iloc[-1], dtype=float)
        if len(forecast_mean_values) < horizon_sessions or len(forecast_variance_values) < horizon_sessions:
            raise ValueError(
                f'The GARCH option could not produce a complete {horizon_sessions}-session forecast for the {window}-session fit window.'
            )

        garch_mean_return = float(np.nansum(forecast_mean_values[:horizon_sessions]) / 100.0)
        garch_sigma_return = float(
            np.sqrt(np.clip(forecast_variance_values[:horizon_sessions], a_min=0.0, a_max=None).sum()) / 100.0
        )
        if not np.isfinite(garch_sigma_return) or garch_sigma_return <= 0:
            raise ValueError(
                f'The GARCH option could not produce a positive {horizon_sessions}-session volatility forecast for the {window}-session fit window.'
            )

        garch_holding_frame = risk_distribution_analytics._build_session_holding_period_frame(
            garch_trade_frame,
            horizon_sessions,
        )
        garch_sample_returns = garch_holding_frame['session_return'].iloc[:-1].tail(window).dropna()
        garch_effective_window = len(garch_sample_returns)
        if garch_effective_window == 0:
            raise ValueError(
                f'The GARCH option could not build a completed {horizon_sessions}-session return sample for the {window}-session fit window.'
            )

        garch_anchor_price = float(risk_distribution_analytics._resolve_current_reference_price(garch_trade_frame))
        garch_latest_price = float(garch_trade_frame.attrs.get('current_session_latest_price', garch_trade_frame['Close'].iloc[-1]))
        garch_session_date = pd.Timestamp(garch_trade_frame.attrs.get('current_session_date', garch_trade_frame.index[-1]))
        garch_median_return = garch_mean_return
        garch_median_price = garch_anchor_price * (1.0 + garch_median_return)

        garch_model_summary = pd.DataFrame([
            {
                'Model': 'GARCH(1,1) Normal',
                'Fit Window': garch_fit_window,
                'Forecast Horizon': horizon_sessions,
                'Forecast Mean Return': garch_mean_return,
                'Forecast Sigma Return': garch_sigma_return,
                'Omega': float(garch_trade_fit.params.get('omega', np.nan)),
                'Alpha(1)': float(garch_trade_fit.params.get('alpha[1]', np.nan)),
                'Beta(1)': float(garch_trade_fit.params.get('beta[1]', np.nan)),
                'Alpha + Beta': float(garch_trade_fit.params.get('alpha[1]', np.nan) + garch_trade_fit.params.get('beta[1]', np.nan)),
            }
        ])

        garch_interval_map = {}
        garch_range_rows = []
        for confidence in trade_range_garch_interval_levels:
            tail_probability = (1.0 - confidence) / 2.0
            lower_return = float(norm.ppf(tail_probability, loc=garch_mean_return, scale=garch_sigma_return))
            upper_return = float(norm.ppf(1.0 - tail_probability, loc=garch_mean_return, scale=garch_sigma_return))
            lower_price = garch_anchor_price * (1.0 + lower_return)
            upper_price = garch_anchor_price * (1.0 + upper_return)
            garch_interval_map[confidence] = {
                'lower_return': lower_return,
                'upper_return': upper_return,
                'lower_price': lower_price,
                'upper_price': upper_price,
            }
            garch_range_rows.append(
                {
                    'Confidence': f'{confidence:.0%}',
                    'Lower Return': lower_return,
                    'Upper Return': upper_return,
                    'Lower Exit Price': lower_price,
                    'Upper Exit Price': upper_price,
                }
            )

        garch_long_tail_map = {}
        garch_short_tail_map = {}
        garch_tail_rows = []
        for confidence in trade_range_garch_tail_levels:
            alpha = 1.0 - confidence
            z_alpha = float(norm.ppf(alpha))
            pdf_alpha = float(norm.pdf(z_alpha))

            long_var_return = garch_mean_return + (garch_sigma_return * z_alpha)
            long_cvar_return = garch_mean_return - (garch_sigma_return * (pdf_alpha / alpha))
            short_var_return = garch_mean_return - (garch_sigma_return * z_alpha)
            short_cvar_return = garch_mean_return + (garch_sigma_return * (pdf_alpha / alpha))

            garch_long_tail_map[confidence] = {
                'var_return': long_var_return,
                'var_price': garch_anchor_price * (1.0 + long_var_return),
                'expected_shortfall_return': long_cvar_return,
                'expected_shortfall_price': garch_anchor_price * (1.0 + long_cvar_return),
            }
            garch_short_tail_map[confidence] = {
                'var_return': short_var_return,
                'var_price': garch_anchor_price * (1.0 + short_var_return),
                'expected_shortfall_return': short_cvar_return,
                'expected_shortfall_price': garch_anchor_price * (1.0 + short_cvar_return),
            }
            garch_tail_rows.append(
                {
                    'Confidence': f'{confidence:.0%}',
                    'Long VaR Floor': garch_long_tail_map[confidence]['var_price'],
                    'Long CVaR Floor': garch_long_tail_map[confidence]['expected_shortfall_price'],
                    'Short VaR Ceiling': garch_short_tail_map[confidence]['var_price'],
                    'Short CVaR Ceiling': garch_short_tail_map[confidence]['expected_shortfall_price'],
                }
            )

        garch_range_summary = pd.DataFrame(garch_range_rows)
        garch_tail_summary = pd.DataFrame(garch_tail_rows).rename(columns={
            'Confidence': 'Tail Confidence',
        })

        garch_trade_range_context = {
            'session_date': garch_session_date,
            'window': window,
            'horizon_sessions': horizon_sessions,
            'effective_window': garch_effective_window,
            'anchor_price': garch_anchor_price,
            'latest_price': garch_latest_price,
            'sample_returns': garch_sample_returns,
            'interval_confidence_levels': trade_range_garch_interval_levels,
            'tail_confidence_levels': trade_range_garch_tail_levels,
            'intervals': garch_interval_map,
            'long_tail_levels': garch_long_tail_map,
            'short_tail_levels': garch_short_tail_map,
            'median_return': garch_median_return,
            'median_price': garch_median_price,
        }

        return {
            'context': garch_trade_range_context,
            'model_summary': garch_model_summary,
            'range_summary': garch_range_summary,
            'tail_summary': garch_tail_summary,
        }

    garch_cone_components_by_key = {
        (int(trade_range_garch_window), int(trade_range_garch_default_horizon)): build_garch_trade_range_cone_components(
            int(trade_range_garch_window),
            int(trade_range_garch_default_horizon),
        )
    }
    garch_default_components = garch_cone_components_by_key[
        (int(trade_range_garch_window), int(trade_range_garch_default_horizon))
    ]

    formatted_garch_model_summary = garch_default_components['model_summary'].copy()
    for column in ('Forecast Mean Return', 'Forecast Sigma Return'):
        if column in formatted_garch_model_summary.columns:
            formatted_garch_model_summary[column] = formatted_garch_model_summary[column].map(
                lambda value: f'{value:.2%}' if pd.notna(value) else value
            )
    for column in ('Omega', 'Alpha(1)', 'Beta(1)', 'Alpha + Beta'):
        if column in formatted_garch_model_summary.columns:
            formatted_garch_model_summary[column] = formatted_garch_model_summary[column].map(
                lambda value: f'{value:.4f}' if pd.notna(value) else value
            )

    garch_range_summary = garch_default_components['range_summary'].copy()
    if not garch_range_summary.empty:
        for column in ('Lower Return', 'Upper Return'):
            if column in garch_range_summary.columns:
                garch_range_summary[column] = garch_range_summary[column].map(
                    lambda value: f'{value:.2%}' if pd.notna(value) else value
                )
        for column in ('Lower Exit Price', 'Upper Exit Price'):
            if column in garch_range_summary.columns:
                garch_range_summary[column] = garch_range_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    garch_tail_summary = garch_default_components['tail_summary'].copy()
    if not garch_tail_summary.empty:
        for column in (
            'Long VaR Floor',
            'Long CVaR Floor',
            'Short VaR Ceiling',
            'Short CVaR Ceiling',
        ):
            if column in garch_tail_summary.columns:
                garch_tail_summary[column] = garch_tail_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

except Exception as exc:
    garch_option_error = exc
    print(f'GARCH option unavailable: {exc}')



def _trade_range_horizon_label(horizon):
    horizon = int(horizon)
    if horizon == 0:
        return '0DTE same-session Open -> Close'
    return '1-session' if horizon == 1 else f'{horizon}-session'


def _trade_range_history_basis(horizon):
    horizon = int(horizon)
    if horizon == 0:
        return 'completed same-session open-to-close return history'
    if horizon == 1:
        return 'completed close-to-close return history'
    return f'completed {horizon}-session close-based holding-period returns'


def _trade_range_confidence_label(confidence):
    return f'{float(confidence):.0%}'


def _trade_range_confidence_key(levels, confidence):
    target = float(confidence)
    for level in levels:
        if np.isclose(float(level), target):
            return level
    raise KeyError(f'Trade-range confidence {target:.0%} is not available.')


def _coerce_trade_range_confidence(confidence):
    if confidence in (None, ''):
        return float(trade_range_default_confidence)
    return float(_trade_range_confidence_key(trade_range_confidence_options, confidence))


def _filter_trade_range_table_by_confidence(frame, confidence, column_name=None):
    if frame is None:
        return frame
    filtered_frame = frame.copy()
    if filtered_frame.empty:
        return filtered_frame
    selected_label = _trade_range_confidence_label(confidence)
    candidate_columns = [column_name] if column_name is not None else ['Confidence', 'Tail Confidence']
    for candidate_column in candidate_columns:
        if candidate_column in filtered_frame.columns:
            return filtered_frame.loc[
                filtered_frame[candidate_column].eq(selected_label)
            ].reset_index(drop=True)
    return filtered_frame


def _filter_trade_range_history_context(history_context, confidence):
    selected_confidence = _trade_range_confidence_key(
        history_context['metrics_by_confidence'].keys(),
        confidence,
    )
    filtered_history_context = dict(history_context)
    filtered_history_context['interval_confidence_levels'] = [float(selected_confidence)]
    filtered_history_context['tail_confidence_levels'] = [float(selected_confidence)]
    filtered_history_context['metrics_by_confidence'] = {
        selected_confidence: history_context['metrics_by_confidence'][selected_confidence]
    }
    return filtered_history_context


def _filter_trade_range_cone_context(cone_context, confidence):
    selected_confidence = _trade_range_confidence_key(cone_context['intervals'].keys(), confidence)
    filtered_cone_context = dict(cone_context)
    filtered_cone_context['interval_confidence_levels'] = [float(selected_confidence)]
    filtered_cone_context['tail_confidence_levels'] = [float(selected_confidence)]
    filtered_cone_context['intervals'] = {
        selected_confidence: cone_context['intervals'][selected_confidence]
    }
    filtered_cone_context['long_tail_levels'] = {
        selected_confidence: cone_context['long_tail_levels'][selected_confidence]
    }
    filtered_cone_context['short_tail_levels'] = {
        selected_confidence: cone_context['short_tail_levels'][selected_confidence]
    }
    if 'range_summary_table' in cone_context:
        filtered_cone_context['range_summary_table'] = _filter_trade_range_table_by_confidence(
            cone_context['range_summary_table'],
            selected_confidence,
            'Confidence',
        )
    if 'tail_summary_table' in cone_context:
        filtered_cone_context['tail_summary_table'] = _filter_trade_range_table_by_confidence(
            cone_context['tail_summary_table'],
            selected_confidence,
            'Confidence',
        )
    return filtered_cone_context


def _filter_garch_trade_range_components(components, confidence):
    filtered_context = _filter_trade_range_cone_context(components['context'], confidence)
    selected_confidence = filtered_context['interval_confidence_levels'][0]
    return {
        'context': filtered_context,
        'model_summary': components['model_summary'].copy(),
        'range_summary': _filter_trade_range_table_by_confidence(
            components['range_summary'],
            selected_confidence,
            'Confidence',
        ),
        'tail_summary': _filter_trade_range_table_by_confidence(
            components['tail_summary'],
            selected_confidence,
            'Tail Confidence',
        ),
    }


def _format_empirical_trade_range_tables(cone_context):
    formatted_range_summary = cone_context['range_summary_table'].copy().rename(columns={
        'Lower Close': 'Lower Exit Price',
        'Upper Close': 'Upper Exit Price',
    })
    formatted_tail_summary = cone_context['tail_summary_table'].copy().rename(columns={
        'Confidence': 'Tail Confidence',
    })

    if not formatted_range_summary.empty:
        for column in ('Lower Return', 'Upper Return'):
            if column in formatted_range_summary.columns:
                formatted_range_summary[column] = formatted_range_summary[column].map(
                    lambda value: f'{value:.2%}' if pd.notna(value) else value
                )
        for column in ('Lower Exit Price', 'Upper Exit Price'):
            if column in formatted_range_summary.columns:
                formatted_range_summary[column] = formatted_range_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    if not formatted_tail_summary.empty:
        for column in (
            'Long VaR Floor',
            'Long CVaR Floor',
            'Short VaR Ceiling',
            'Short CVaR Ceiling',
        ):
            if column in formatted_tail_summary.columns:
                formatted_tail_summary[column] = formatted_tail_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    return formatted_range_summary, formatted_tail_summary


def _format_garch_trade_range_tables(components):
    formatted_model_summary = components['model_summary'].copy()
    for column in ('Forecast Mean Return', 'Forecast Sigma Return'):
        if column in formatted_model_summary.columns:
            formatted_model_summary[column] = formatted_model_summary[column].map(
                lambda value: f'{value:.2%}' if pd.notna(value) else value
            )
    for column in ('Omega', 'Alpha(1)', 'Beta(1)', 'Alpha + Beta'):
        if column in formatted_model_summary.columns:
            formatted_model_summary[column] = formatted_model_summary[column].map(
                lambda value: f'{value:.4f}' if pd.notna(value) else value
            )

    formatted_range_summary = components['range_summary'].copy()
    if not formatted_range_summary.empty:
        for column in ('Lower Return', 'Upper Return'):
            if column in formatted_range_summary.columns:
                formatted_range_summary[column] = formatted_range_summary[column].map(
                    lambda value: f'{value:.2%}' if pd.notna(value) else value
                )
        for column in ('Lower Exit Price', 'Upper Exit Price'):
            if column in formatted_range_summary.columns:
                formatted_range_summary[column] = formatted_range_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    formatted_tail_summary = components['tail_summary'].copy()
    if not formatted_tail_summary.empty:
        for column in (
            'Long VaR Floor',
            'Long CVaR Floor',
            'Short VaR Ceiling',
            'Short CVaR Ceiling',
        ):
            if column in formatted_tail_summary.columns:
                formatted_tail_summary[column] = formatted_tail_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    return formatted_model_summary, formatted_range_summary, formatted_tail_summary


def _strip_trade_range_dropdowns(fig):
    stripped_fig = _copy.deepcopy(fig)
    stripped_fig.update_layout(updatemenus=[])
    return stripped_fig


def _build_empirical_trade_range_view(window, horizon, confidence=None):
    window = int(window)
    horizon = int(horizon)
    confidence = _coerce_trade_range_confidence(confidence)
    cached_history_context = trade_range_history_contexts_by_horizon.get(horizon)
    if cached_history_context is not None and window in cached_history_context['metrics_by_window']:
        history_context = {
            'window': window,
            'horizon_sessions': horizon,
            'interval_confidence_levels': cached_history_context['interval_confidence_levels'],
            'tail_confidence_levels': cached_history_context['tail_confidence_levels'],
            'metrics_by_confidence': cached_history_context['metrics_by_window'][window],
            'return_basis': cached_history_context.get('return_basis'),
            'session_returns': cached_history_context['session_returns'],
            'session_open': cached_history_context['session_open'],
            'session_close': cached_history_context['session_close'],
        }
    else:
        history_context = risk_distribution_analytics.build_trade_range_history_context(
            price_frame=trade_range_price_frame,
            windows=[window],
            horizon_sessions=horizon,
            interval_confidence_levels=trade_range_interval_levels,
            tail_confidence_levels=trade_range_tail_levels,
            default_window=window,
        )
        history_context = {
            'window': window,
            'horizon_sessions': horizon,
            'interval_confidence_levels': history_context['interval_confidence_levels'],
            'tail_confidence_levels': history_context['tail_confidence_levels'],
            'metrics_by_confidence': history_context['metrics_by_confidence'],
            'return_basis': history_context.get('return_basis'),
            'session_returns': history_context['session_returns'],
            'session_open': history_context['session_open'],
            'session_close': history_context['session_close'],
        }
    history_context = _filter_trade_range_history_context(history_context, confidence)
    history_fig = _strip_trade_range_dropdowns(
        plot_trade_range_history_profile(
            history_context=history_context,
            ticker_label=ticker_str,
        )
    )
    cone_cache_key = (window, horizon)
    cone_context = trade_range_cone_contexts_by_key.get(cone_cache_key)
    if cone_context is None:
        cone_context = risk_distribution_analytics.build_trade_range_probability_context(
            price_frame=trade_range_price_frame,
            window=window,
            horizon_sessions=horizon,
            interval_confidence_levels=trade_range_interval_levels,
            tail_confidence_levels=trade_range_tail_levels,
        )
        trade_range_cone_contexts_by_key[cone_cache_key] = cone_context
    filtered_cone_context = _filter_trade_range_cone_context(cone_context, confidence)
    range_summary_window, tail_summary_window = _format_empirical_trade_range_tables(filtered_cone_context)
    cone_fig = _strip_trade_range_dropdowns(
        plot_trade_range_probability_cone(
            cone_context=filtered_cone_context,
            ticker_label=ticker_str,
        )
    )
    combined_fig = plot_trade_range_stack_view(
        history_fig,
        cone_fig,
        title_text=(
            f'{ticker_str} Historical Two-Sided Trade Range Profile and Current Cone '
            f'({int(window)}-Session Lookback, {_trade_range_horizon_label(horizon)} Horizon)'
        ),
    )
    combined_fig.update_layout(updatemenus=[])
    return {
        'caption': (
            f'Empirical trade range using {_trade_range_history_basis(horizon)} '
            f'at {_trade_range_confidence_label(confidence)} confidence.'
        ),
        'model_summary': None,
        'range_summary': range_summary_window,
        'tail_summary': tail_summary_window,
        'cone_figure': cone_fig,
        'figure': combined_fig,
    }


def _build_garch_trade_range_view(window, horizon, confidence=None):
    window = int(window)
    horizon = int(horizon)
    confidence = _coerce_trade_range_confidence(confidence)
    garch_cache_key = (window, horizon)
    if garch_cache_key not in garch_cone_components_by_key:
        garch_cone_components_by_key[garch_cache_key] = build_garch_trade_range_cone_components(window, horizon)
    components = _filter_garch_trade_range_components(
        garch_cone_components_by_key[garch_cache_key],
        confidence,
    )
    formatted_model_summary_window, range_summary_window, tail_summary_window = _format_garch_trade_range_tables(components)
    history_context = _filter_trade_range_history_context(
        build_garch_trade_range_history_context(window, horizon),
        confidence,
    )
    history_fig = _strip_trade_range_dropdowns(
        plot_trade_range_history_profile(
            history_context=history_context,
            ticker_label=f'{ticker_str} GARCH(1,1) Normal',
        )
    )
    cone_fig = _strip_trade_range_dropdowns(
        plot_trade_range_probability_cone(
            cone_context=components['context'],
            ticker_label=f'{ticker_str} GARCH(1,1)',
        )
    )
    combined_fig = plot_trade_range_stack_view(
        history_fig,
        cone_fig,
        title_text=(
            f'{ticker_str} GARCH(1,1) Historical Diagnostics and Current Cone '
            f'({int(window)}-Session Fit Window, {int(horizon)}-Session Horizon)'
        ),
    )
    combined_fig.update_layout(updatemenus=[])
    if horizon == 1:
        caption = (
            f'GARCH(1,1)-normal trade range fit on completed one-session returns at '
            f'{_trade_range_confidence_label(confidence)} confidence. '
            'Session lookback changes the current-session cone, and the historical diagnostics stay on the same one-session basis.'
        )
    else:
        caption = (
            f'GARCH(1,1)-normal view filtered to {_trade_range_confidence_label(confidence)} confidence. '
            f'GARCH(1,1)-normal current cones still aggregate one-session forecasts across the selected {horizon}-session horizon. '
            f'The historical diagnostics now plot completed {horizon}-session holding returns through a horizon-matched GARCH filter so the rolling return panel stays aligned with the selected horizon.'
        )
    return {
        'caption': caption,
        'model_summary': formatted_model_summary_window,
        'range_summary': range_summary_window,
        'tail_summary': tail_summary_window,
        'cone_figure': cone_fig,
        'figure': combined_fig,
    }


trade_range_view_builders = {
    'Empirical': _build_empirical_trade_range_view,
}
trade_range_window_options_by_method = {
    'Empirical': list(trade_range_window_options),
}

if all(name in globals() for name in ('garch_trade_history_fig', 'garch_cone_components_by_key')):
    trade_range_view_builders['GARCH(1,1) Normal'] = _build_garch_trade_range_view
    trade_range_window_options_by_method['GARCH(1,1) Normal'] = list(trade_range_garch_window_options)

trade_range_view_cache = {}


def get_trade_range_view(method, window, horizon, confidence=None):
    confidence = _coerce_trade_range_confidence(confidence)
    cache_key = (str(method), int(window), int(horizon), float(confidence))
    if cache_key not in trade_range_view_cache:
        trade_range_view_cache[cache_key] = trade_range_view_builders[str(method)](
            int(window),
            int(horizon),
            float(confidence),
        )
    return trade_range_view_cache[cache_key]


def _set_default_trade_range_outputs():
    default_empirical_view = get_trade_range_view(
        'Empirical',
        int(trade_range_default_window),
        int(trade_range_default_horizon),
        float(trade_range_default_confidence),
    )
    globals()['range_summary'] = default_empirical_view['range_summary'].copy()
    globals()['tail_summary'] = default_empirical_view['tail_summary'].copy()
    globals()['trade_range_fig'] = default_empirical_view['cone_figure']
    globals()['trade_range_combined_fig'] = default_empirical_view['figure']

    if 'GARCH(1,1) Normal' in trade_range_view_builders:
        default_garch_window = trade_range_garch_window if 'trade_range_garch_window' in globals() else trade_range_window_options_by_method['GARCH(1,1) Normal'][0]
        default_garch_view = get_trade_range_view(
            'GARCH(1,1) Normal',
            int(default_garch_window),
            int(trade_range_garch_default_horizon if 'trade_range_garch_default_horizon' in globals() else 1),
            float(trade_range_default_confidence),
        )
        globals()['formatted_garch_model_summary'] = default_garch_view['model_summary'].copy()
        globals()['garch_range_summary'] = default_garch_view['range_summary'].copy()
        globals()['garch_tail_summary'] = default_garch_view['tail_summary'].copy()
        globals()['garch_trade_range_fig'] = default_garch_view['cone_figure']
        globals()['garch_trade_combined_fig'] = default_garch_view['figure']


from dash import Dash, Input, Output, State, dash_table, dcc, html
import socket
import uuid

required_trade_range_globals = (
    'get_trade_range_view',
    'trade_range_window_options_by_method',
    'trade_range_default_window',
    'trade_range_default_horizon',
)
missing_trade_range_globals = [name for name in required_trade_range_globals if name not in globals()]
if missing_trade_range_globals:
    raise NameError(
        'Run Block 7 before Block 7B. Missing globals: ' + ', '.join(missing_trade_range_globals)
    )



_set_default_trade_range_outputs()

def _dash_trade_range_window_bounds(method):
    method = str(method)
    min_window = 21
    if method == 'GARCH(1,1) Normal' and 'garch_completed_returns' in globals():
        max_window = int(len(garch_completed_returns))
    else:
        max_window = int(len(trade_range_price_frame.dropna()) - 1)
    max_window = max(min_window, max_window)
    return min_window, max_window


def _dash_trade_range_preferred_window(method):
    method = str(method)
    preferred_window = int(trade_range_default_window)
    if method == 'GARCH(1,1) Normal' and 'trade_range_garch_window' in globals():
        preferred_window = int(trade_range_garch_window)
    min_window, max_window = _dash_trade_range_window_bounds(method)
    return max(min_window, min(max_window, preferred_window))


def _coerce_dash_trade_range_window(method, value):
    min_window, max_window = _dash_trade_range_window_bounds(method)
    preferred_window = _dash_trade_range_preferred_window(method)
    helper_text = f'Available range: {min_window} to {max_window} completed sessions.'

    if value in (None, ''):
        return preferred_window, min_window, max_window, helper_text

    try:
        normalized_value = int(value)
    except (TypeError, ValueError):
        return (
            preferred_window,
            min_window,
            max_window,
            f'{helper_text} Using {preferred_window}-session lookback because the input was not a whole number.',
        )

    if normalized_value < min_window:
        return (
            min_window,
            min_window,
            max_window,
            f'{helper_text} Using {min_window}-session lookback because the requested value was below the supported minimum.',
        )

    if normalized_value > max_window:
        return (
            max_window,
            min_window,
            max_window,
            f'{helper_text} Using {max_window}-session lookback because the request exceeded the available completed-session history.',
        )

    return normalized_value, min_window, max_window, helper_text


def _dash_trade_range_default_window(method, current_window=None):
    if current_window is None:
        return _dash_trade_range_preferred_window(method)
    return _coerce_dash_trade_range_window(method, current_window)[0]


def _dash_trade_range_horizon_bounds(method, window):
    method = str(method)
    window, _, _, _ = _coerce_dash_trade_range_window(method, window)
    available_sessions = int(len(trade_range_price_frame.dropna()))
    max_horizon = max(1, (available_sessions - int(window) + 1) // 2)
    min_horizon = 1 if method == 'GARCH(1,1) Normal' else 0
    return min_horizon, max_horizon


def _dash_trade_range_preferred_horizon(method, window):
    min_horizon, max_horizon = _dash_trade_range_horizon_bounds(method, window)
    preferred_horizon = int(trade_range_default_horizon)
    if method == 'GARCH(1,1) Normal':
        preferred_horizon = int(globals().get('trade_range_garch_default_horizon', 1))
    return max(min_horizon, min(max_horizon, preferred_horizon))


def _coerce_dash_trade_range_horizon(method, window, value):
    min_horizon, max_horizon = _dash_trade_range_horizon_bounds(method, window)
    preferred_horizon = _dash_trade_range_preferred_horizon(method, window)
    if min_horizon == 0:
        helper_text = (
            f'Available range: 0 to {max_horizon} sessions. '
            '0 means 0DTE same-session Open -> Close; 1+ means close-to-close holding periods.'
        )
    else:
        helper_text = (
            f'Available range: {min_horizon} to {max_horizon} sessions. '
            '1 session means close-to-close.'
        )

    if value in (None, ''):
        return preferred_horizon, min_horizon, max_horizon, helper_text

    try:
        normalized_value = int(value)
    except (TypeError, ValueError):
        return (
            preferred_horizon,
            min_horizon,
            max_horizon,
            f'{helper_text} Using {preferred_horizon}-session horizon because the input was not a whole number.',
        )

    if normalized_value < min_horizon:
        return (
            min_horizon,
            min_horizon,
            max_horizon,
            f'{helper_text} Using {min_horizon}-session horizon because the requested value was below the supported minimum.',
        )

    if normalized_value > max_horizon:
        return (
            max_horizon,
            min_horizon,
            max_horizon,
            f'{helper_text} Using {max_horizon}-session horizon because the request exceeded the available completed-session history.',
        )

    return normalized_value, min_horizon, max_horizon, helper_text


def _dash_trade_range_default_horizon(method, window, current_horizon=None):
    if current_horizon is None:
        return _dash_trade_range_preferred_horizon(method, window)
    return _coerce_dash_trade_range_horizon(method, window, current_horizon)[0]


def _dash_trade_range_snapshot_table(method, window, horizon, confidence, view):
    rows = [
        {'Metric': 'Method', 'Value': method},
        {'Metric': 'Session Lookback Window', 'Value': f'{int(window)}-session'},
        {'Metric': 'Holding Horizon', 'Value': _trade_range_horizon_label(horizon)},
        {'Metric': 'Confidence Level', 'Value': _trade_range_confidence_label(confidence)},
        {'Metric': 'Description', 'Value': view['caption']},
    ]

    if method == 'GARCH(1,1) Normal' and view['model_summary'] is not None and not view['model_summary'].empty:
        summary_row = view['model_summary'].iloc[0]
        preferred_fields = [
            ('Forecast Mean', 'Forecast Mean Return'),
            ('Forecast Sigma', 'Forecast Sigma Return'),
            ('Omega', 'Omega'),
            ('Alpha(1)', 'Alpha(1)'),
            ('Beta(1)', 'Beta(1)'),
            ('Alpha + Beta', 'Alpha + Beta'),
        ]
        for label, column in preferred_fields:
            if column in summary_row.index:
                rows.append({'Metric': label, 'Value': str(summary_row[column])})
    else:
        rows.append({'Metric': 'Data Basis', 'Value': _trade_range_history_basis(horizon)})

    return pd.DataFrame(rows)


def _dash_table_payload(frame):
    if frame is None or frame.empty:
        frame = pd.DataFrame({'Info': ['No data available']})
    frame = frame.fillna('')
    columns = [{'name': column, 'id': column} for column in frame.columns]
    data = frame.astype(str).to_dict('records')
    return columns, data


def _find_open_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(('127.0.0.1', 0))
        sock.listen(1)
        return sock.getsockname()[1]


dash_trade_range_id = f"trade-range-dash-{uuid.uuid4().hex[:8]}"
dash_trade_range_methods = [str(method) for method in trade_range_window_options_by_method.keys()]
dash_trade_range_default_method = 'Empirical' if 'Empirical' in dash_trade_range_methods else dash_trade_range_methods[0]
dash_trade_range_default_window = _dash_trade_range_default_window(dash_trade_range_default_method)
dash_trade_range_default_horizon = _dash_trade_range_default_horizon(
    dash_trade_range_default_method,
    dash_trade_range_default_window,
)
dash_trade_range_default_confidence = float(trade_range_default_confidence)

dash_trade_range_app = Dash(dash_trade_range_id)
dash_trade_range_app.layout = html.Div(
    [
        html.Div(
            [
                html.Div(
                    [
                        html.Label('Method', style={'display': 'block', 'marginBottom': '6px', 'fontWeight': '600'}),
                        dcc.Dropdown(
                            id=f'{dash_trade_range_id}-method',
                            options=[{'label': method, 'value': method} for method in dash_trade_range_methods],
                            value=dash_trade_range_default_method,
                            clearable=False,
                        ),
                    ],
                    style={'flex': '1 1 320px', 'minWidth': '260px'},
                ),
                html.Div(
                    [
                        html.Label('Session Lookback Window', style={'display': 'block', 'marginBottom': '6px', 'fontWeight': '600'}),
                        dcc.Input(
                            id=f'{dash_trade_range_id}-window',
                            type='number',
                            value=dash_trade_range_default_window,
                            min=_dash_trade_range_window_bounds(dash_trade_range_default_method)[0],
                            max=_dash_trade_range_window_bounds(dash_trade_range_default_method)[1],
                            step=1,
                            debounce=True,
                            inputMode='numeric',
                            style={'width': '100%'},
                        ),
                        html.Div(
                            id=f'{dash_trade_range_id}-window-status',
                            style={'marginTop': '6px', 'fontSize': '12px', 'color': '#94a3b8'},
                        ),
                    ],
                    style={'flex': '0 0 220px', 'minWidth': '180px'},
                ),
                html.Div(
                    [
                        html.Label('Holding Horizon (0 = 0DTE Open -> Close)', style={'display': 'block', 'marginBottom': '6px', 'fontWeight': '600'}),
                        dcc.Input(
                            id=f'{dash_trade_range_id}-horizon',
                            type='number',
                            value=dash_trade_range_default_horizon,
                            min=_dash_trade_range_horizon_bounds(
                                dash_trade_range_default_method,
                                dash_trade_range_default_window,
                            )[0],
                            max=_dash_trade_range_horizon_bounds(
                                dash_trade_range_default_method,
                                dash_trade_range_default_window,
                            )[1],
                            step=1,
                            debounce=True,
                            inputMode='numeric',
                            style={'width': '100%'},
                        ),
                        html.Div(
                            id=f'{dash_trade_range_id}-horizon-status',
                            style={'marginTop': '6px', 'fontSize': '12px', 'color': '#94a3b8'},
                        ),
                    ],
                    style={'flex': '0 0 220px', 'minWidth': '180px'},
                ),
                html.Div(
                    [
                        html.Label('Confidence', style={'display': 'block', 'marginBottom': '6px', 'fontWeight': '600'}),
                        dcc.Dropdown(
                            id=f'{dash_trade_range_id}-confidence',
                            options=[
                                {
                                    'label': _trade_range_confidence_label(confidence),
                                    'value': float(confidence),
                                }
                                for confidence in trade_range_confidence_options
                            ],
                            value=dash_trade_range_default_confidence,
                            clearable=False,
                        ),
                    ],
                    style={'flex': '0 0 200px', 'minWidth': '180px'},
                ),
            ],
            style={'display': 'flex', 'gap': '12px', 'flexWrap': 'wrap', 'marginBottom': '14px'},
        ),
        html.Div(id=f'{dash_trade_range_id}-caption', style={'marginBottom': '14px', 'fontWeight': '600'}),
        html.Div(
            [
                html.Div('Model Snapshot', style={'marginBottom': '8px', 'fontWeight': '600'}),
                dash_table.DataTable(
                    id=f'{dash_trade_range_id}-snapshot-table',
                    style_table={'overflowX': 'auto', 'width': '100%'},
                    style_cell={
                        'textAlign': 'left',
                        'padding': '6px 10px',
                        'backgroundColor': '#0f172a',
                        'color': '#e2e8f0',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                        'whiteSpace': 'normal',
                        'height': 'auto',
                    },
                    style_header={
                        'backgroundColor': '#1e293b',
                        'color': '#f8fafc',
                        'fontWeight': '700',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                    },
                ),
            ],
            style={'marginBottom': '18px'},
        ),
        html.Div(
            [
                html.Div('Projected Return / Price Range Summary', style={'marginBottom': '8px', 'fontWeight': '600'}),
                dash_table.DataTable(
                    id=f'{dash_trade_range_id}-range-table',
                    style_table={'overflowX': 'auto', 'width': '100%'},
                    style_cell={
                        'textAlign': 'left',
                        'padding': '6px 10px',
                        'backgroundColor': '#0f172a',
                        'color': '#e2e8f0',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                        'whiteSpace': 'normal',
                        'height': 'auto',
                    },
                    style_header={
                        'backgroundColor': '#1e293b',
                        'color': '#f8fafc',
                        'fontWeight': '700',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                    },
                ),
            ],
            style={'marginBottom': '18px'},
        ),
        html.Div(
            [
                html.Div('Projected Tail Summary', style={'marginBottom': '8px', 'fontWeight': '600'}),
                dash_table.DataTable(
                    id=f'{dash_trade_range_id}-tail-table',
                    style_table={'overflowX': 'auto', 'width': '100%'},
                    style_cell={
                        'textAlign': 'left',
                        'padding': '6px 10px',
                        'backgroundColor': '#0f172a',
                        'color': '#e2e8f0',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                        'whiteSpace': 'normal',
                        'height': 'auto',
                    },
                    style_header={
                        'backgroundColor': '#1e293b',
                        'color': '#f8fafc',
                        'fontWeight': '700',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                    },
                ),
            ],
            style={'marginBottom': '18px'},
        ),
        dcc.Graph(
            id=f'{dash_trade_range_id}-figure',
            style={'width': '100%', 'height': '3000px'},
            config={'responsive': True, 'displaylogo': False},
        ),
    ],
    style={
        'width': '100%',
        'maxWidth': '100%',
        'padding': '4px 0 16px 0',
        'fontFamily': 'Inter, Segoe UI, sans-serif',
    },
)


@dash_trade_range_app.callback(
    Output(f'{dash_trade_range_id}-window', 'value'),
    Output(f'{dash_trade_range_id}-window', 'min'),
    Output(f'{dash_trade_range_id}-window', 'max'),
    Input(f'{dash_trade_range_id}-method', 'value'),
    State(f'{dash_trade_range_id}-window', 'value'),
)
def _sync_dash_trade_range_window(method, current_window):
    selected_window, min_window, max_window, _ = _coerce_dash_trade_range_window(method, current_window)
    return selected_window, min_window, max_window


@dash_trade_range_app.callback(
    Output(f'{dash_trade_range_id}-horizon', 'value'),
    Output(f'{dash_trade_range_id}-horizon', 'min'),
    Output(f'{dash_trade_range_id}-horizon', 'max'),
    Input(f'{dash_trade_range_id}-method', 'value'),
    Input(f'{dash_trade_range_id}-window', 'value'),
    State(f'{dash_trade_range_id}-horizon', 'value'),
)
def _sync_dash_trade_range_horizon(method, window, current_horizon):
    selected_window, _, _, _ = _coerce_dash_trade_range_window(method, window)
    selected_horizon, min_horizon, max_horizon, _ = _coerce_dash_trade_range_horizon(
        method,
        selected_window,
        current_horizon,
    )
    return selected_horizon, min_horizon, max_horizon


@dash_trade_range_app.callback(
    Output(f'{dash_trade_range_id}-caption', 'children'),
    Output(f'{dash_trade_range_id}-window-status', 'children'),
    Output(f'{dash_trade_range_id}-horizon-status', 'children'),
    Output(f'{dash_trade_range_id}-snapshot-table', 'columns'),
    Output(f'{dash_trade_range_id}-snapshot-table', 'data'),
    Output(f'{dash_trade_range_id}-range-table', 'columns'),
    Output(f'{dash_trade_range_id}-range-table', 'data'),
    Output(f'{dash_trade_range_id}-tail-table', 'columns'),
    Output(f'{dash_trade_range_id}-tail-table', 'data'),
    Output(f'{dash_trade_range_id}-figure', 'figure'),
    Input(f'{dash_trade_range_id}-method', 'value'),
    Input(f'{dash_trade_range_id}-window', 'value'),
    Input(f'{dash_trade_range_id}-horizon', 'value'),
    Input(f'{dash_trade_range_id}-confidence', 'value'),
)
def _render_dash_trade_range_view(method, window, horizon, confidence):
    method = str(method)
    window, _, _, window_status = _coerce_dash_trade_range_window(method, window)
    horizon, _, _, horizon_status = _coerce_dash_trade_range_horizon(method, window, horizon)
    confidence = _coerce_trade_range_confidence(confidence)
    view = get_trade_range_view(method, window, horizon, confidence)

    snapshot_columns, snapshot_data = _dash_table_payload(
        _dash_trade_range_snapshot_table(method, window, horizon, confidence, view)
    )
    range_columns, range_data = _dash_table_payload(view['range_summary'])
    tail_columns, tail_data = _dash_table_payload(view['tail_summary'])

    figure = _copy.deepcopy(view['figure'])
    figure.update_layout(autosize=True)

    caption = (
        f'{method} | {window}-session lookback | {_trade_range_horizon_label(horizon)} horizon | '
        f'{_trade_range_confidence_label(confidence)} confidence | {view["caption"]}'
    )
    return (
        caption,
        window_status,
        horizon_status,
        snapshot_columns,
        snapshot_data,
        range_columns,
        range_data,
        tail_columns,
        tail_data,
        figure,
    )


dash_trade_range_port = _find_open_port()
print(f'Dash Block 7B preview starting on port {dash_trade_range_port}.')
dash_trade_range_app.run(
    port=dash_trade_range_port,
    debug=False,
    jupyter_mode='inline',
    jupyter_width='100%',
    jupyter_height=3900,
    dev_tools_ui=False,
    dev_tools_props_check=False,
)

In [ ]:
# Block 30: Trade Range Breach Calibration

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


trade_range_breach_windows = [21, 50, 200]
trade_range_breach_confidence_levels = list(globals().get('trade_range_tail_levels', [0.95, 0.99]))
trade_range_breach_price_frame = globals().get(
    'trade_range_price_frame',
    globals().get('ticker_trade_range_source', ticker)[['Open', 'Close']].copy(),
)
trade_range_breach_complete_frame = (
    trade_range_breach_price_frame[['Open', 'Close']].dropna().sort_index().copy()
)
trade_range_breach_complete_rows = int(len(trade_range_breach_complete_frame))
if trade_range_breach_complete_rows < 2:
    raise ValueError('Block 34 requires at least two completed sessions with Open and Close prices.')

trade_range_breach_target_end = min(21, trade_range_breach_complete_rows)
trade_range_breach_horizon_step = 1
trade_range_breach_confidence_labels = {
    float(confidence): f'{confidence:.0%}' for confidence in trade_range_breach_confidence_levels
}
trade_range_breach_window_labels = [f'{int(window)}-session' for window in trade_range_breach_windows]
trade_range_breach_horizon_grid = sorted(
    {
        1,
        *range(
            min(trade_range_breach_horizon_step, trade_range_breach_target_end),
            trade_range_breach_target_end + 1,
            trade_range_breach_horizon_step,
        ),
        trade_range_breach_target_end,
    }
)
trade_range_breach_tick_step = 20 if trade_range_breach_target_end > 120 else 10
trade_range_breach_tick_values = [
    horizon
    for horizon in trade_range_breach_horizon_grid
    if horizon in (1, trade_range_breach_target_end) or horizon % trade_range_breach_tick_step == 0
]
trade_range_breach_cache_signature = (
    tuple(int(window) for window in trade_range_breach_windows),
    tuple(float(confidence) for confidence in trade_range_breach_confidence_levels),
    int(trade_range_breach_complete_rows),
    str(trade_range_breach_complete_frame.index[0]),
    str(trade_range_breach_complete_frame.index[-1]),
    float(trade_range_breach_complete_frame['Open'].iloc[0]),
    float(trade_range_breach_complete_frame['Open'].iloc[-1]),
    float(trade_range_breach_complete_frame['Close'].iloc[0]),
    float(trade_range_breach_complete_frame['Close'].iloc[-1]),
)
if globals().get('trade_range_breach_cache_signature') != trade_range_breach_cache_signature:
    trade_range_breach_snapshot_cache = {}
else:
    trade_range_breach_snapshot_cache = globals().get('trade_range_breach_snapshot_cache', {})
globals()['trade_range_breach_cache_signature'] = trade_range_breach_cache_signature


def _build_trade_range_breach_horizon_snapshot(horizon):
    horizon = int(horizon)
    if horizon <= 0:
        raise ValueError('horizon must be a positive integer.')
    if horizon > trade_range_breach_complete_rows:
        raise ValueError('horizon exceeds the available completed session count.')

    cached_snapshot = trade_range_breach_snapshot_cache.get(horizon)
    if isinstance(cached_snapshot, pd.DataFrame) and not cached_snapshot.empty:
        return cached_snapshot.copy()

    holding_frame = risk_distribution_analytics._build_session_holding_period_frame(
        trade_range_breach_complete_frame,
        horizon,
    )
    session_returns = holding_frame['session_return'].to_numpy(dtype=float)
    latest_calibration_date = pd.Timestamp(holding_frame.index[-1])

    records = []
    for window in trade_range_breach_windows:
        rolling_window = int(window)
        if len(session_returns) < rolling_window:
            rolling_samples = None
            valid_count = 0
            realized_returns = np.array([], dtype=float)
        else:
            rolling_samples = sliding_window_view(session_returns, rolling_window)
            valid_count = len(session_returns) - horizon - rolling_window + 1
            realized_returns = session_returns[horizon + rolling_window - 1:]

        for confidence in trade_range_breach_confidence_levels:
            alpha = 1.0 - float(confidence)
            actual_breach_rate = np.nan
            expected_breach_rate = np.nan
            calibration_date = pd.NaT

            if rolling_samples is not None and valid_count > 0:
                lower_var = np.quantile(rolling_samples, alpha, axis=1)[:valid_count]
                upper_var = np.quantile(rolling_samples, 1.0 - alpha, axis=1)[:valid_count]
                breaches = np.logical_or(
                    realized_returns < lower_var,
                    realized_returns > upper_var,
                ).astype(float)
                if len(breaches) >= rolling_window:
                    actual_breach_rate = float(breaches[-rolling_window:].mean())
                    expected_breach_rate = float(min(1.0, 2.0 * alpha))
                    calibration_date = latest_calibration_date

            records.append(
                {
                    'horizon': horizon,
                    'window': rolling_window,
                    'confidence': float(confidence),
                    'latest_calibration_date': calibration_date,
                    'actual_breach_rate': actual_breach_rate,
                    'expected_breach_rate': expected_breach_rate,
                    'excess_breach_rate': (
                        actual_breach_rate - expected_breach_rate
                        if pd.notna(actual_breach_rate) and pd.notna(expected_breach_rate)
                        else np.nan
                    ),
                }
            )

    snapshot_frame = pd.DataFrame(records)
    trade_range_breach_snapshot_cache[horizon] = snapshot_frame
    globals()['trade_range_breach_snapshot_cache'] = trade_range_breach_snapshot_cache
    return snapshot_frame.copy()


def _build_trade_range_breach_summary():
    summary_frames = [
        _build_trade_range_breach_horizon_snapshot(horizon)
        for horizon in trade_range_breach_horizon_grid
    ]
    summary = pd.concat(summary_frames, ignore_index=True).sort_values(
        ['confidence', 'window', 'horizon']
    ).reset_index(drop=True)

    support_text = (
        f'Displaying holding horizons 1 through {trade_range_breach_target_end} '
        f'in single-session steps across lookbacks {", ".join(str(window) for window in trade_range_breach_windows)}. '
        'Blank cells mean there was not enough completed history to measure the latest rolling breach rate '
        'for that horizon and lookback combination.'
    )
    return summary, list(trade_range_breach_horizon_grid), support_text


def _build_trade_range_breach_excess_figure(summary_frame, supported_horizons):
    return plot_trade_range_breach_excess_view(
        summary_frame,
        supported_horizons=supported_horizons,
        windows=trade_range_breach_windows,
        confidence_levels=trade_range_breach_confidence_levels,
        confidence_labels=trade_range_breach_confidence_labels,
        ticker_label=ticker_str,
        target_end=trade_range_breach_target_end,
    )


def _format_trade_range_breach_summary(summary_frame):
    formatted_summary = summary_frame.copy()
    formatted_summary['Holding Horizon'] = formatted_summary['horizon'].map(
        lambda value: f'{int(value)}-session'
    )
    formatted_summary['Session Lookback Window'] = formatted_summary['window'].map(
        lambda value: f'{int(value)}-session'
    )
    formatted_summary['Confidence'] = formatted_summary['confidence'].map(
        lambda value: f'{value:.0%}'
    )
    formatted_summary['Latest Calibration Date'] = formatted_summary['latest_calibration_date'].map(
        lambda value: pd.Timestamp(value).strftime('%Y-%m-%d') if pd.notna(value) else ''
    )
    for column in ('actual_breach_rate', 'expected_breach_rate', 'excess_breach_rate'):
        formatted_summary[column] = formatted_summary[column].map(
            lambda value: f'{value:.2%}' if pd.notna(value) else ''
        )
    return formatted_summary.rename(
        columns={
            'actual_breach_rate': 'Actual Either-Side Breach Rate',
            'expected_breach_rate': 'Expected Either-Side Breach Rate',
            'excess_breach_rate': 'Excess Either-Side Breach Rate',
        }
    )[
        [
            'Holding Horizon',
            'Session Lookback Window',
            'Confidence',
            'Latest Calibration Date',
            'Actual Either-Side Breach Rate',
            'Expected Either-Side Breach Rate',
            'Excess Either-Side Breach Rate',
        ]
    ]


trade_range_breach_calibration_summary, trade_range_breach_supported_horizons, trade_range_breach_support_text = (
    _build_trade_range_breach_summary()
)
trade_range_breach_calibration_summary_display = _format_trade_range_breach_summary(
    trade_range_breach_calibration_summary
)
trade_range_breach_excess_fig = _build_trade_range_breach_excess_figure(
    trade_range_breach_calibration_summary,
    trade_range_breach_supported_horizons,
)

globals()['trade_range_breach_supported_horizons'] = list(trade_range_breach_supported_horizons)
globals()['trade_range_breach_calibration_summary'] = trade_range_breach_calibration_summary.copy()
globals()['trade_range_breach_calibration_summary_display'] = (
    trade_range_breach_calibration_summary_display.copy()
)
globals()['trade_range_breach_excess_fig'] = trade_range_breach_excess_fig

print(trade_range_breach_support_text)
display(trade_range_breach_excess_fig)
display(trade_range_breach_calibration_summary_display)

In [ ]:
# Block 31: Average Breach Rates Over Time

import numpy as np
import pandas as pd

if 'trade_range_breach_calibration_summary' not in globals():
    raise ValueError('Run Block 34 before Block 35 to build the breach-rate calibration summary.')

trade_range_breach_average_price_frame = globals().get(
    'trade_range_breach_price_frame',
    globals().get('trade_range_price_frame', globals().get('ticker_trade_range_source', ticker)[['Open', 'Close']].copy()),
)
trade_range_breach_average_price_frame = (
    trade_range_breach_average_price_frame[['Open', 'Close']].dropna().sort_index().copy()
)
trade_range_breach_average_horizons = [
    int(horizon)
    for horizon in globals().get(
        'trade_range_breach_supported_horizons',
        sorted(int(value) for value in globals()['trade_range_breach_calibration_summary']['horizon'].dropna().unique()),
    )
]
trade_range_breach_average_windows = [
    int(window)
    for window in globals().get(
        'trade_range_breach_windows',
        sorted(int(value) for value in globals()['trade_range_breach_calibration_summary']['window'].dropna().unique()),
    )
]
trade_range_breach_average_confidences = [
    float(confidence)
    for confidence in globals().get(
        'trade_range_breach_confidence_levels',
        sorted(float(value) for value in globals()['trade_range_breach_calibration_summary']['confidence'].dropna().unique()),
    )
]
trade_range_breach_average_signature = (
    tuple(trade_range_breach_average_horizons),
    tuple(trade_range_breach_average_windows),
    tuple(trade_range_breach_average_confidences),
    int(len(trade_range_breach_average_price_frame)),
    str(trade_range_breach_average_price_frame.index[0]),
    str(trade_range_breach_average_price_frame.index[-1]),
    float(trade_range_breach_average_price_frame['Open'].iloc[0]),
    float(trade_range_breach_average_price_frame['Open'].iloc[-1]),
    float(trade_range_breach_average_price_frame['Close'].iloc[0]),
    float(trade_range_breach_average_price_frame['Close'].iloc[-1]),
)
if globals().get('trade_range_breach_average_signature') != trade_range_breach_average_signature:
    trade_range_breach_average_contexts_by_horizon = {}
else:
    trade_range_breach_average_contexts_by_horizon = globals().get(
        'trade_range_breach_average_contexts_by_horizon',
        {},
    )
globals()['trade_range_breach_average_signature'] = trade_range_breach_average_signature

trade_range_breach_average_palette = ['#38bdf8', '#22c55e', '#f59e0b', '#f97316', '#a855f7']
trade_range_breach_average_colors = {
    int(window): trade_range_breach_average_palette[index % len(trade_range_breach_average_palette)]
    for index, window in enumerate(trade_range_breach_average_windows)
}


def _build_trade_range_breach_average_context(horizon):
    horizon = int(horizon)
    cached_context = trade_range_breach_average_contexts_by_horizon.get(horizon)
    if isinstance(cached_context, dict) and cached_context:
        return cached_context

    history_context = risk_distribution_analytics.build_trade_range_history_context(
        price_frame=trade_range_breach_average_price_frame,
        windows=trade_range_breach_average_windows,
        horizon_sessions=horizon,
        interval_confidence_levels=trade_range_breach_average_confidences,
        tail_confidence_levels=trade_range_breach_average_confidences,
        default_window=trade_range_breach_average_windows[0],
    )
    trade_range_breach_average_contexts_by_horizon[horizon] = history_context
    globals()['trade_range_breach_average_contexts_by_horizon'] = trade_range_breach_average_contexts_by_horizon
    return history_context


def _build_trade_range_breach_average_panel():
    averaged_series = {}
    horizon_count_series = {}
    support_lines = []

    for confidence in trade_range_breach_average_confidences:
        for window in trade_range_breach_average_windows:
            horizon_series = []
            for horizon in trade_range_breach_average_horizons:
                history_context = _build_trade_range_breach_average_context(horizon)
                metric_set = history_context['metrics_by_window'][int(window)][float(confidence)]
                breach_rate_series = metric_set.get('either_side_rolling_breach_rate', pd.Series(dtype=float)).dropna()
                if breach_rate_series.empty:
                    continue
                horizon_series.append(
                    breach_rate_series.rename(f'h{int(horizon)}')
                )

            if not horizon_series:
                continue

            aligned_frame = pd.concat(horizon_series, axis=1, join='inner').dropna(how='any')
            if aligned_frame.empty:
                continue

            averaged_series[(float(confidence), int(window))] = aligned_frame.mean(axis=1)
            horizon_count_series[(float(confidence), int(window))] = pd.Series(
                data=np.full(len(aligned_frame.index), aligned_frame.shape[1], dtype=int),
                index=aligned_frame.index,
            )
            support_lines.append(
                f'{float(confidence):.0%} / {int(window)}-session starts {aligned_frame.index.min():%Y-%m-%d} '
                f'with all {aligned_frame.shape[1]} horizons aligned.'
            )

    if not averaged_series:
        raise ValueError('Unable to build any averaged breach-rate series across the configured horizons.')

    average_panel = pd.concat(averaged_series, axis=1).sort_index(axis=1)
    count_panel = pd.concat(horizon_count_series, axis=1).sort_index(axis=1)
    support_text = 'Averages are taken across all configured holding horizons on dates where every horizon-specific rolling breach-rate series is available. '
    support_text += ' '.join(sorted(set(support_lines)))
    return average_panel, count_panel, support_text


def _build_trade_range_breach_average_figure(average_panel, count_panel):
    return plot_trade_range_breach_average_view(
        average_panel,
        count_panel,
        horizons=trade_range_breach_average_horizons,
        windows=trade_range_breach_average_windows,
        confidence_levels=trade_range_breach_average_confidences,
        window_colors=trade_range_breach_average_colors,
        ticker_label=ticker_str,
    )


trade_range_breach_average_panel, trade_range_breach_average_count_panel, trade_range_breach_average_support_text = (
    _build_trade_range_breach_average_panel()
)
trade_range_breach_average_fig = _build_trade_range_breach_average_figure(
    trade_range_breach_average_panel,
    trade_range_breach_average_count_panel,
)
globals()['trade_range_breach_average_panel'] = trade_range_breach_average_panel.copy()
globals()['trade_range_breach_average_count_panel'] = trade_range_breach_average_count_panel.copy()
globals()['trade_range_breach_average_support_text'] = trade_range_breach_average_support_text
globals()['trade_range_breach_average_fig'] = trade_range_breach_average_fig

print(trade_range_breach_average_support_text)
display(trade_range_breach_average_fig)

In [ ]:
# Block 32: Fixed Payout Strategy Backtest

strategy_payout_options = [(float(risk_dollars), float(100 - risk_dollars)) for risk_dollars in range(10, 100, 10)]
strategy_default_payout = (70.0, 30.0)
strategy_confidence = 0.95
strategy_rolling_pnl_window = 21


def format_strategy_payout_label(risk_dollars, reward_dollars):
    return f'Risk ${risk_dollars:,.0f} / Reward ${reward_dollars:,.0f}'


def build_fixed_payout_trade_profile(strategy_label, metrics_by_confidence, *, confidence, risk_dollars, reward_dollars, window_label, rolling_window):
    metric_set = metrics_by_confidence.get(confidence)
    if metric_set is None:
        raise KeyError(f'{strategy_label} does not contain a {confidence:.0%} long interval-floor series.')

    long_interval_floor = pd.Series(metric_set['lower_interval_price']).dropna()
    session_open = pd.Series(metric_set['session_open']).reindex(long_interval_floor.index)
    session_close = pd.Series(metric_set['session_close']).reindex(long_interval_floor.index)
    session_returns = pd.Series(metric_set['session_returns']).reindex(long_interval_floor.index)

    trade_frame = pd.DataFrame({
        'Open': session_open,
        'Close': session_close,
        'Close-to-Close Return': session_returns,
        'Long 95% Interval Floor': long_interval_floor,
    }).dropna()
    if trade_frame.empty:
        raise ValueError(f'{strategy_label} does not contain any fully aligned historical trades to evaluate.')

    breakeven_win_rate = risk_dollars / (risk_dollars + reward_dollars)
    payout_label = format_strategy_payout_label(risk_dollars, reward_dollars)

    trade_frame['Win'] = trade_frame['Close'].ge(trade_frame['Long 95% Interval Floor'])
    trade_frame['Outcome'] = np.where(trade_frame['Win'], 'Win', 'Loss')
    trade_frame['Trade PnL'] = np.where(trade_frame['Win'], reward_dollars, -risk_dollars)
    trade_frame['Rolling PnL'] = trade_frame['Trade PnL'].rolling(rolling_window, min_periods=1).sum()
    trade_frame['Rolling Win Rate'] = trade_frame['Win'].rolling(rolling_window, min_periods=1).mean()
    trade_frame['Cumulative PnL'] = trade_frame['Trade PnL'].cumsum()
    trade_frame['Running Peak PnL'] = trade_frame['Cumulative PnL'].cummax()
    trade_frame['Drawdown'] = trade_frame['Cumulative PnL'] - trade_frame['Running Peak PnL']
    trade_frame['Payout Structure'] = payout_label

    gross_profit = float(trade_frame.loc[trade_frame['Trade PnL'] > 0, 'Trade PnL'].sum())
    gross_loss = float(-trade_frame.loc[trade_frame['Trade PnL'] < 0, 'Trade PnL'].sum())
    profit_factor = np.nan if gross_loss == 0 else gross_profit / gross_loss

    summary = {
        'Payout Structure': payout_label,
        'Strategy': strategy_label,
        'Session Lookback Window': window_label,
        'Confidence': f'{confidence:.0%}',
        'Trades': int(len(trade_frame)),
        'Wins': int(trade_frame['Win'].sum()),
        'Losses': int((~trade_frame['Win']).sum()),
        'Win Rate': float(trade_frame['Win'].mean()),
        'Breakeven Win Rate': float(breakeven_win_rate),
        'Average Trade PnL': float(trade_frame['Trade PnL'].mean()),
        'Latest Rolling PnL': float(trade_frame['Rolling PnL'].iloc[-1]),
        'Total PnL': float(trade_frame['Trade PnL'].sum()),
        'Profit Factor': profit_factor,
        'Worst Drawdown': float(trade_frame['Drawdown'].min()),
    }

    return {
        'label': strategy_label,
        'summary': summary,
        'trade_frame': trade_frame,
    }


strategy_include_garch = 'garch_trade_history_context' in globals()
if not strategy_include_garch:
    print('Run Block 33 first if you want the GARCH strategy included in this fixed-payout evaluation.')


def build_strategy_profiles_for_payout(risk_dollars, reward_dollars):
    strategy_profiles = []
    for window in trade_range_history_context['windows']:
        strategy_profiles.append(
            build_fixed_payout_trade_profile(
                strategy_label=f'Current Method ({window}-Session)',
                metrics_by_confidence=trade_range_history_context['metrics_by_window'][window],
                confidence=strategy_confidence,
                risk_dollars=risk_dollars,
                reward_dollars=reward_dollars,
                window_label=window,
                rolling_window=strategy_rolling_pnl_window,
            )
        )

    if strategy_include_garch:
        garch_window_label = garch_trade_history_context.get('window', trade_range_default_window)
        strategy_profiles.append(
            build_fixed_payout_trade_profile(
                strategy_label=f'GARCH(1,1) Normal ({garch_window_label}-Session)',
                metrics_by_confidence=garch_trade_history_context['metrics_by_confidence'],
                confidence=strategy_confidence,
                risk_dollars=risk_dollars,
                reward_dollars=reward_dollars,
                window_label=garch_window_label,
                rolling_window=strategy_rolling_pnl_window,
            )
        )

    return strategy_profiles


strategy_default_payout_label = format_strategy_payout_label(*strategy_default_payout)
strategy_profiles_by_payout = {}
strategy_summary_by_payout = {}
strategy_backtest_trade_logs_by_payout = {}
strategy_backtest_equity_curve_by_payout = {}
strategy_backtest_rolling_pnl_by_payout = {}

for risk_dollars, reward_dollars in strategy_payout_options:
    payout_label = format_strategy_payout_label(risk_dollars, reward_dollars)
    strategy_profiles = build_strategy_profiles_for_payout(risk_dollars, reward_dollars)
    strategy_profiles_by_payout[payout_label] = strategy_profiles
    strategy_summary_by_payout[payout_label] = pd.DataFrame([profile['summary'] for profile in strategy_profiles])
    strategy_backtest_trade_logs_by_payout[payout_label] = {
        profile['label']: profile['trade_frame'].copy()
        for profile in strategy_profiles
    }
    strategy_backtest_equity_curve_by_payout[payout_label] = pd.concat(
        {profile['label']: profile['trade_frame']['Cumulative PnL'] for profile in strategy_profiles},
        axis=1,
    ).sort_index()
    strategy_backtest_rolling_pnl_by_payout[payout_label] = pd.concat(
        {profile['label']: profile['trade_frame']['Rolling PnL'] for profile in strategy_profiles},
        axis=1,
    ).sort_index()

strategy_summary_lookup = pd.concat(strategy_summary_by_payout.values(), ignore_index=True)
formatted_strategy_summary_lookup = strategy_summary_lookup.copy()
for column in ('Win Rate', 'Breakeven Win Rate'):
    if column in formatted_strategy_summary_lookup.columns:
        formatted_strategy_summary_lookup[column] = formatted_strategy_summary_lookup[column].map(
            lambda value: f'{value:.2%}' if pd.notna(value) else value
        )
for column in ('Average Trade PnL', 'Latest Rolling PnL', 'Total PnL', 'Worst Drawdown'):
    if column in formatted_strategy_summary_lookup.columns:
        formatted_strategy_summary_lookup[column] = formatted_strategy_summary_lookup[column].map(
            lambda value: f'${value:,.2f}' if pd.notna(value) else value
        )
if 'Profit Factor' in formatted_strategy_summary_lookup.columns:
    formatted_strategy_summary_lookup['Profit Factor'] = formatted_strategy_summary_lookup['Profit Factor'].map(
        lambda value: 'N/A' if pd.isna(value) else f'{value:.2f}'
    )

display(formatted_strategy_summary_lookup)

strategy_backtest_trade_logs = strategy_backtest_trade_logs_by_payout[strategy_default_payout_label]
strategy_backtest_equity_curve = strategy_backtest_equity_curve_by_payout[strategy_default_payout_label]
strategy_backtest_rolling_pnl = strategy_backtest_rolling_pnl_by_payout[strategy_default_payout_label]
strategy_trade_log_preview = pd.concat(
    [
        profile['trade_frame'].assign(Strategy=profile['label'])
        for profile in strategy_profiles_by_payout[strategy_default_payout_label]
    ],
    axis=0,
).reset_index(names='Date')
strategy_trade_log_preview = strategy_trade_log_preview[['Date', 'Payout Structure', 'Strategy', 'Close', 'Long 95% Interval Floor', 'Outcome', 'Trade PnL', 'Rolling PnL', 'Rolling Win Rate', 'Cumulative PnL', 'Drawdown']]
display(strategy_trade_log_preview.tail(15))

strategy_backtest_fig = plot_fixed_payout_strategy_backtest_view(
    strategy_profiles_by_payout,
    payout_options=strategy_payout_options,
    default_payout=strategy_default_payout,
    rolling_pnl_window=strategy_rolling_pnl_window,
    ticker_label=ticker_str,
)
show_plotly_figure(strategy_backtest_fig)

In [ ]:
# Block 33: Volatility Model Comparison

import sys

def _purge_stale_modules(prefixes):
    for prefix in prefixes:
        matching_modules = [
            name
            for name in list(sys.modules)
            if name == prefix or name.startswith(f"{prefix}.")
        ]
        for module_name in matching_modules:
            sys.modules.pop(module_name, None)

try:
    from arch import arch_model
except ModuleNotFoundError as exc:
    if exc.name != "arch":
        raise
    raise ImportError(
        "Block 33 requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
    ) from exc
except Exception:
    _purge_stale_modules(("arch", "matplotlib"))
    try:
        from arch import arch_model
    except ModuleNotFoundError as exc:
        if exc.name != "arch":
            raise
        raise ImportError(
            "Block 33 requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
        ) from exc
    except Exception as exc:
        raise RuntimeError(
            "Block 33 could not import `arch` because the notebook kernel is holding a stale matplotlib state. Restart the kernel and rerun Block 2 if this persists."
        ) from exc

close_series = ticker["Close"]
ohlc_frame = ticker[["Open", "High", "Low", "Close"]].copy()
returns = close_series.pct_change()
garch_input = returns.dropna() * 100

log_hl = np.log(ohlc_frame["High"] / ohlc_frame["Low"])
log_ho = np.log(ohlc_frame["High"] / ohlc_frame["Open"])
log_lo = np.log(ohlc_frame["Low"] / ohlc_frame["Open"])
log_co = np.log(ohlc_frame["Close"] / ohlc_frame["Open"])
log_oc = np.log(ohlc_frame["Open"] / ohlc_frame["Close"].shift(1))
log_hc = np.log(ohlc_frame["High"] / ohlc_frame["Close"])
log_lc = np.log(ohlc_frame["Low"] / ohlc_frame["Close"])

garman_klass_variance = 0.5 * (log_hl ** 2) - ((2 * np.log(2)) - 1) * (log_co ** 2)
parkinson_variance = (log_hl ** 2) / (4 * np.log(2))
rs_variance = (log_hc * log_ho) + (log_lc * log_lo)

volatility_model_specs = [
    ("GARCH(1,1)", dict(vol="GARCH", p=1, q=1, o=0), "#111111", "solid"),
    ("EGARCH(1,1)", dict(vol="EGARCH", p=1, o=1, q=1), "#d62728", "dash"),
    ("GJR-GARCH(1,1)", dict(vol="GARCH", p=1, o=1, q=1), "#2ca02c", "dot"),
]

rolling_realized_vol_specs = [
    ("Close-to-Close", "close-to-close", "#1f77b4", "solid"),
    ("Parkinson", "parkinson", "#9467bd", "dash"),
    ("Yang-Zhang", "yang-zhang", "#ff7f0e", "dot"),
    ("Garman-Klass", "garman-klass", "#8c564b", "dashdot"),
    ("Rogers-Satchell", "rogers-satchell", "#17becf", "longdash"),
]

ewma_realized_vol_specs = [
    ("EWMA Close-to-Close", "close-to-close", "#1f77b4", "longdashdot"),
    ("EWMA Parkinson", "parkinson", "#9467bd", "longdashdot"),
    ("EWMA Yang-Zhang", "yang-zhang", "#ff7f0e", "longdashdot"),
    ("EWMA Garman-Klass", "garman-klass", "#8c564b", "longdashdot"),
    ("EWMA Rogers-Satchell", "rogers-satchell", "#17becf", "longdashdot"),
]

volatility_series_spike_smoothing_reports = {}

def _smooth_block37_volatility_series(series, label, window=63):
    smoothed, report = smooth_insane_series_spikes(
        series,
        label=None,
        rolling_window=max(21, min(126, int(window) * 3)),
        robust_z_threshold=8.0,
        interpolation_limit=max(3, min(10, int(window) // 2)),
        floor=0.0,
    )
    if not report.empty:
        volatility_series_spike_smoothing_reports[label] = report
    smoothed.name = getattr(series, 'name', label)
    return smoothed

annualized_model_vols_raw = {}
annualized_model_vols = {}
for model_name, model_kwargs, _, _ in volatility_model_specs:
    model_fit = arch_model(
        garch_input,
        mean="Zero",
        dist="normal",
        rescale=False,
        **model_kwargs,
    ).fit(disp="off")
    annualized_model_vol = (model_fit.conditional_volatility / 100.0) * np.sqrt(252)
    annualized_model_vol.name = f"Annualized {model_name}"
    annualized_model_vols_raw[model_name] = annualized_model_vol.copy()
    annualized_model_vol = _smooth_block37_volatility_series(
        annualized_model_vol,
        f'{model_name} conditional volatility',
        window=63,
    )
    annualized_model_vol.name = f"Annualized {model_name}"
    annualized_model_vols[model_name] = annualized_model_vol

def compute_rolling_realized_vol_series(window, method):
    if method == "close-to-close":
        raw_series = returns.rolling(window).std() * np.sqrt(252)
        return _smooth_block37_volatility_series(
            raw_series,
            f'Close-to-Close rolling realized vol ({window})',
            window=window,
        )
    if method == "parkinson":
        rolling_variance = parkinson_variance.rolling(window=window).mean().clip(lower=0)
    elif method == "yang-zhang":
        if window < 2:
            return pd.Series(np.nan, index=ohlc_frame.index)
        k = 0.34 / (1.34 + ((window + 1) / (window - 1)))
        overnight_variance = log_oc.rolling(window=window).var()
        open_to_close_variance = log_co.rolling(window=window).var()
        rs_component = rs_variance.rolling(window=window).mean()
        rolling_variance = (
            overnight_variance
            + (k * open_to_close_variance)
            + ((1 - k) * rs_component)
        ).clip(lower=0)
    elif method == "garman-klass":
        rolling_variance = garman_klass_variance.rolling(window=window).mean().clip(lower=0)
    elif method == "rogers-satchell":
        rolling_variance = rs_variance.rolling(window=window).mean().clip(lower=0)
    else:
        raise ValueError(f"Unsupported rolling volatility method: {method}")

    raw_series = np.sqrt(rolling_variance) * np.sqrt(252)
    return _smooth_block37_volatility_series(
        raw_series,
        f'{method} rolling realized vol ({window})',
        window=window,
    )

def compute_ewma_realized_vol_series(window, method):
    alpha = 2.0 / (window + 1.0)

    if method == "close-to-close":
        ewma_variance = returns.pow(2).ewm(alpha=alpha, adjust=False, min_periods=window).mean()
        raw_series = np.sqrt(ewma_variance * 252)
        return _smooth_block37_volatility_series(
            raw_series,
            f'Close-to-Close EWMA realized vol ({window})',
            window=window,
        )

    if method == "parkinson":
        ewma_variance = parkinson_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        raw_series = np.sqrt(ewma_variance * 252)
        return _smooth_block37_volatility_series(
            raw_series,
            f'Parkinson EWMA realized vol ({window})',
            window=window,
        )

    if method == "garman-klass":
        ewma_variance = garman_klass_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        raw_series = np.sqrt(ewma_variance * 252)
        return _smooth_block37_volatility_series(
            raw_series,
            f'Garman-Klass EWMA realized vol ({window})',
            window=window,
        )

    if method == "rogers-satchell":
        ewma_variance = rs_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        raw_series = np.sqrt(ewma_variance * 252)
        return _smooth_block37_volatility_series(
            raw_series,
            f'Rogers-Satchell EWMA realized vol ({window})',
            window=window,
        )

    if method == "yang-zhang":
        if window < 2:
            return pd.Series(np.nan, index=ohlc_frame.index)

        k = 0.34 / (1.34 + ((window + 1) / (window - 1)))
        overnight_variance = log_oc.ewm(alpha=alpha, adjust=False, min_periods=window).var(bias=False)
        open_to_close_variance = log_co.ewm(alpha=alpha, adjust=False, min_periods=window).var(bias=False)
        rs_component = rs_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean()
        yz_variance = (
            overnight_variance
            + (k * open_to_close_variance)
            + ((1 - k) * rs_component)
        ).clip(lower=0)
        raw_series = np.sqrt(yz_variance * 252)
        return _smooth_block37_volatility_series(
            raw_series,
            f'Yang-Zhang EWMA realized vol ({window})',
            window=window,
        )

    raise ValueError(f"Unsupported EWMA volatility method: {method}")

volatility_term_order = [term for term in time_frame_map if time_frame_map.get(term) is not None]
default_vol_term = 'long' if 'long' in volatility_term_order else max(
    volatility_term_order,
    key=lambda term: int(time_frame_map[term]),
)

volatility_term_plot_data = {}
for term in volatility_term_order:
    window = int(time_frame_map[term])
    rolling_realized_vol_map = {
        label: compute_rolling_realized_vol_series(window, method)
        for label, method, _, _ in rolling_realized_vol_specs
    }
    ewma_realized_vol_map = {
        label: compute_ewma_realized_vol_series(window, method)
        for label, method, _, _ in ewma_realized_vol_specs
    }

    non_empty_series = [
        series
        for series in (
            [series.dropna() for series in rolling_realized_vol_map.values()]
            + [series.dropna() for series in ewma_realized_vol_map.values()]
            + [model_series.dropna() for model_series in annualized_model_vols.values()]
        )
        if not series.empty
    ]
    if non_empty_series:
        max_index = max(series.index.max() for series in non_empty_series)
        min_index = min(series.index.min() for series in non_empty_series)
        term_range = [max(min_index, max_index - pd.DateOffset(years=3)), max_index]
    else:
        term_range = None

    volatility_term_plot_data[term] = {
        'window': window,
        'rolling_realized_vol_map': rolling_realized_vol_map,
        'ewma_realized_vol_map': ewma_realized_vol_map,
        'term_range': term_range,
    }

if volatility_series_spike_smoothing_reports:
    total_smoothed_vol_spikes = int(
        sum(report['spike_count'].sum() for report in volatility_series_spike_smoothing_reports.values())
    )
    print(
        f'Smoothed {total_smoothed_vol_spikes} insane volatility-model/realized-vol spike(s). '
        'See volatility_series_spike_smoothing_reports for details.'
    )

vol_model_fig = plot_volatility_model_comparison_view(
    annualized_model_vols=annualized_model_vols,
    term_plot_data=volatility_term_plot_data,
    volatility_model_specs=volatility_model_specs,
    rolling_realized_vol_specs=rolling_realized_vol_specs,
    ewma_realized_vol_specs=ewma_realized_vol_specs,
    term_order=volatility_term_order,
    default_term=default_vol_term,
    time_frame_map=time_frame_map,
    ticker_label=ticker_str,
)
show_plotly_figure(vol_model_fig)